# 🍌 ETL Inteligente con LlamaIndex — Sector Bananero Ecuador
### Databricks · Llama 4 Maverick · LlamaIndex Workflows · Six Sigma

---

**Proyecto de Investigación**

Este notebook replica el pipeline ETL del experimento LangGraph usando el framework de agentes **LlamaIndex Workflows**.  
El LLM orquestador es **Llama 4 Maverick** (Databricks Foundation Model).  
Se mantiene el mismo marco de métricas M1–M6 y la estructura de tablas Delta Lake.

| Fase | Tabla de métricas | Qué mide |
|------|-------------------|---------|
| 📥 Extracción | `metricas_extraccion_li` | Tiempo, archivos nuevos, errores de descarga |
| 🔧 Transformación | `metricas_transformacion_li` | Calidad, duplicados, casteos, tasa_error parcial |
| 💾 Carga | `metricas_carga_li` | Registros insertados, errores, tiempo de escritura |
| 📊 General ETL | `control_logs_etl` | M1–M6 completas del proceso end-to-end |

### Diferencias clave vs LangGraph
| Aspecto | LangGraph | LlamaIndex Workflows |
|---------|-----------|---------------------|
| Primitiva de orquestación | `StateGraph` con nodos y aristas | `Workflow` con `Events` y `@step` |
| Estado compartido | `TypedDict` mutable | Paso de eventos inmutables |
| Enrutamiento condicional | `add_conditional_edges` | `send_event` / retorno de `Event` tipado |
| LLM | `ChatDatabricks` (LangChain) | `DatabricksLLM` (LlamaIndex) |
| Memoria de agente | `StateGraph` state | `Context` del Workflow |

### Orden de ejecución
1. **Bloque 1** — Instalar librerías (reiniciar kernel después)
2. **Bloque 2** — Imports y configuración global
3. **Bloque 3** — Extracción con LlamaIndex Workflow + métricas
4. **Bloque 4** — Diccionario de conocimiento
5. **Bloque 5** — Funciones utilitarias compartidas
6. **Bloque 6** — Workflow ETL principal (mapeo, transformación, carga, métricas)
7. **Bloque 7** — Orquestador principal
8. **Bloque 8** — Métricas de transformación
9. **Bloque 9** — Métricas de carga
10. **Bloque 10** — Métricas generales M1–M6
11. **Bloque 11** — Utilidades (reset, consultas)


## 🗑️ Bloque 0 — Reset completo (opcional)

> ⚠️ **CUIDADO — estas celdas son destructivas.**  
> Úsalas solo si quieres empezar el experimento desde cero.

### Bloque 0A — Borrar tablas Delta del catálogo


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOQUE 0A — BORRAR TABLAS DELTA DEL CATÁLOGO (LlamaIndex)
# ⚠️ Descomenta las líneas para ejecutar.
# ═══════════════════════════════════════════════════════════════════════════

DB_NAME = "bd_banano_ec"

TABLAS_LOGS = [
    f"{DB_NAME}.control_logs_etl",
    f"{DB_NAME}.metricas_extraccion",
    f"{DB_NAME}.metricas_transformacion",
    f"{DB_NAME}.metricas_carga",
    f"{DB_NAME}.control_descargas_fuentes",
]

TABLAS_DATOS = [
    f"{DB_NAME}.espac_banano_platano_provincia",
    f"{DB_NAME}.faostat_produccion_banano_platano",
    f"{DB_NAME}.espac_uso_del_suelo",
    f"{DB_NAME}.sipa_temperatura_precipitacion",
    f"{DB_NAME}.sipa_uso_del_suelo",
    f"{DB_NAME}.aebe_exportaciones_regiones",
    f"{DB_NAME}.dim_regiones",
    f"{DB_NAME}.dim_provincias",
]

TODAS = TABLAS_LOGS + TABLAS_DATOS

print("Tablas que serán eliminadas si descomenta el DROP:")
for t in TODAS:
    existe = spark.catalog.tableExists(t)
    icono  = "✅ existe" if existe else "⬜ no existe"
    print(f"   {icono}  →  {t}")

# ── DESCOMENTA PARA EJECUTAR ──────────────────────────────────────────────
eliminadas = 0
for tabla in TODAS:
    try:
        spark.sql(f"DROP TABLE IF EXISTS {tabla}")
        print(f"  🗑️  Eliminada: {tabla}")
        eliminadas += 1
    except Exception as e:
        print(f"  ❌ Error eliminando {tabla}: {e}")
print(f"\n✅ {eliminadas}/{len(TODAS)} tablas eliminadas.")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# BLOQUE 0B — VACIAR VOLÚMENES (archivos descargados)
# ⚠️ Descomenta las líneas para ejecutar.
# ═══════════════════════════════════════════════════════════════════════════

import os

VOLUMENES = {
    "ESPAC"         : "/Volumes/workspace/default/raw_espac",
    "SIPA"          : "/Volumes/workspace/default/raw_sipa",
    "FAOSTAT"       : "/Volumes/workspace/default/raw_faostat",
    "AEBE"          : "/Volumes/workspace/default/raw_aebe",
}

EXTENSIONES = {".xlsx", ".xls", ".csv", ".zip", ".json"}

print("Archivos que serán eliminados si descomenta el bloque de abajo:")
total_archivos, total_mb = 0, 0.0

for label, path in VOLUMENES.items():
    try:
        archivos = [f for f in dbutils.fs.ls(path)
                    if os.path.splitext(f.name.lower())[1] in EXTENSIONES]
        mb = sum(f.size for f in archivos) / (1024*1024)
        total_archivos += len(archivos)
        total_mb += mb
        print(f"  📁 {label}: {len(archivos)} archivos — {mb:.2f} MB")
    except Exception as e:
        print(f"  ⚠️  Error listando {label}: {e}")

print(f"\nTOTAL: {total_archivos} archivos — {total_mb:.2f} MB")

# ── DESCOMENTA PARA EJECUTAR ──────────────────────────────────────────────
borrados, errores = 0, 0
for label, path in VOLUMENES.items():
    try:
        archivos = [f for f in dbutils.fs.ls(path)
                    if os.path.splitext(f.name.lower())[1] in EXTENSIONES]
        for f in archivos:
            try:
                dbutils.fs.rm(f.path)
                borrados += 1
            except Exception as e:
                errores += 1
        print(f"  ✅ {label}: {len(archivos)} archivos eliminados")
    except Exception as e:
        print(f"  ❌ Error en {label}: {e}")
print(f"\n✅ {borrados} archivos eliminados, {errores} errores.")


## 📦 Bloque 1 — Instalación de librerías

⚠️ **Ejecuta SOLO esta celda primero.** Espera el reinicio del kernel antes de continuar.


In [ ]:
%pip install llama-index llama-index-core langchain-databricks \
             google-api-python-client google-auth-httplib2 google-auth-oauthlib \
             openpyxl xlrd beautifulsoup4 numpy faostat scikit-learn \
             pdfplumber
dbutils.library.restartPython()

## ⚙️ Bloque 2 — Imports y Configuración Global

✏️ **Edita las credenciales** en esta celda antes de continuar.

**LlamaIndex vs LangGraph:** en lugar de `StateGraph` se usa `Workflow` con eventos tipados y decorador `@step`.


In [ ]:
import io, os, re, json, time, zipfile, hashlib, unicodedata
import numpy as np
import pandas as pd
import requests
from datetime import datetime
from typing import TypedDict, Optional, List
from urllib.parse import urljoin
from bs4 import BeautifulSoup

# LangGraph
# LlamaIndex core
from llama_index.core.workflow import (
    Workflow, StartEvent, StopEvent, step, Context, Event,
)
from llama_index.core.llms import ChatMessage, MessageRole
from langchain_databricks import ChatDatabricks
from langchain_core.messages import HumanMessage, SystemMessage

# Google Drive
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaInMemoryUpload

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.functions import lower, trim, col, when
from pyspark.sql.types import (
    StructType, StructField, StringType,
    TimestampType, IntegerType, DoubleType, FloatType
)

# ── CREDENCIALES — edita estos valores ────────────────────────────────────
# Ya no se necesita GEMINI_API_KEY — usaremos Databricks Foundation Models

SERVICE_ACCOUNT_INFO = {
  "type": "secreto"
}

FOLDER_OUTPUT_ID = "secreto"   # output_tableau

# ── Rutas de volúmenes Databricks ─────────────────────────────────────────
RAW_PATH_ESPAC   = "/Volumes/workspace/default/raw_espac"
RAW_PATH_SIPA    = "/Volumes/workspace/default/raw_sipa"
RAW_PATH_FAOSTAT = "/Volumes/workspace/default/raw_faostat"
RAW_PATH_AEBE    = "/Volumes/workspace/default/raw_aebe"

# ── CONFIGURACIÓN DE TABLAS (TODO CENTRALIZADO EN BD_BANANO_EC) ─────────────
DB_NAME        = "bd_banano_ec"
FRAMEWORK_NAME = "LlamaIndex"  # Identificador del framework

# 🆔 CORRECCIÓN #21: ID único de ejecución (para historial de métricas)
from datetime import datetime
EXECUTION_ID = datetime.now().strftime("%Y%m%d_%H%M%S")  # Ej: 20260619_041556

LOG_TABLE      = f"{DB_NAME}.control_logs_etl"  # Tabla unificada LangGraph + LlamaIndex
CONTROL_TABLE  = f"{DB_NAME}.control_descargas_fuentes"  # Compartida con LangGraph
LOG_EXTRACCION = f"{DB_NAME}.metricas_extraccion"
LOG_TRANSFORM  = f"{DB_NAME}.metricas_transformacion"
LOG_CARGA      = f"{DB_NAME}.metricas_carga"

HEADERS_HTTP = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
    "Referer": "https://www.banana-traders.com/"
}

# ── Credenciales FAOSTAT (auto-renovación de token) ───────────────────────
FAOSTAT_USERNAME = "secreto@gmail.com"
FAOSTAT_PASSWORD = "secreto" # ⚠️ CAMBIAR POR TU PASSWORD REAL
FAOSTAT_TOKEN = None  # Se obtiene automáticamente
FAOSTAT_TOKEN_EXPIRY = None  # Timestamp de expiración

def get_faostat_token():
    """
    Obtiene o renueva el token de FAOSTAT automáticamente.
    El token dura 60 minutos - esta función lo renueva cuando expira.
    """
    global FAOSTAT_TOKEN, FAOSTAT_TOKEN_EXPIRY
    
    # Verificar si el token sigue válido (con margen de 5 min)
    if FAOSTAT_TOKEN and FAOSTAT_TOKEN_EXPIRY:
        if time.time() < (FAOSTAT_TOKEN_EXPIRY - 300):  # 5 min antes
            return FAOSTAT_TOKEN
    
    # Obtener nuevo token
    print("  🔄 Renovando token FAOSTAT...")
    try:
        resp = requests.post(
            "https://faostatservices.fao.org/api/v1/auth/login",
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            data={"username": FAOSTAT_USERNAME, "password": FAOSTAT_PASSWORD},
            timeout=30
        )
        resp.raise_for_status()
        token_data = resp.json()
        
        # FAOSTAT usa AWS Cognito - extraer de AuthenticationResult
        if 'AuthenticationResult' in token_data:
            auth_result = token_data['AuthenticationResult']
            # Preferir IdToken, luego AccessToken
            FAOSTAT_TOKEN = auth_result.get('IdToken') or auth_result.get('AccessToken')
        else:
            # Fallback por si cambia el formato
            FAOSTAT_TOKEN = token_data.get("access_token") or token_data.get("token")
        
        FAOSTAT_TOKEN_EXPIRY = time.time() + 3600  # 60 min
        print("  ✅ Token renovado (válido por 60 min)")
        return FAOSTAT_TOKEN
    except Exception as e:
        print(f"  ❌ Error renovando token: {e}")
        raise

# ── Inicializar servicios ──────────────────────────────────────────────────
creds = service_account.Credentials.from_service_account_info(
    SERVICE_ACCOUNT_INFO,
    scopes=["https://www.googleapis.com/auth/drive"]
)
drive_service = build("drive", "v3", credentials=creds)

# Usar modelo de Databricks Foundation Models (sin límites de API gratuita)
llm = ChatDatabricks(
    endpoint="databricks-llama-4-maverick",  # Llama 4 Maverick - 400B MoE, 128K context
    temperature=0.0,
    max_tokens=2000,
)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
print("✅ Configuración cargada correctamente.")
print(f"   Base de datos      : {DB_NAME}")
print(f"   Log general ETL    : {LOG_TABLE}")
print(f"   Log extracción     : {LOG_EXTRACCION}")
print(f"   Log transformación : {LOG_TRANSFORM}")
print(f"   Log carga          : {LOG_CARGA}")

## 📥 Bloque 3 — Extracción de Fuentes con LlamaIndex Workflow

Replica el Agente de Extracción de LangGraph usando `Workflow` + eventos tipados.

| # | Fuente | Método |
|---|--------|--------|
| 1 | ESPAC (INEC) | URLs hardcodeadas + HEAD request |
| 2 | SIPA (MAG) | Descarga directa |
| 3 | FAOSTAT | API REST con token |
| 4 | Banana-Traders | Scraping XLSX |

### Diferencia arquitectónica LangGraph → LlamaIndex

| LangGraph | LlamaIndex Workflows |
|-----------|---------------------|
| Nodos conectados con `add_conditional_edges` | `@step` que devuelve `Event` tipado |
| Estado `TypedDict` mutable | Eventos inmutables pasados entre pasos |
| `graph.invoke(estado)` | `await workflow.run(StartEvent(...))` |


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# TABLA DE CONTROL DE DESCARGAS
# ══════════════════════════════════════════════════════════════════════════
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_TABLE} (
    fuente          STRING,
    framework       STRING,
    anio            INT,
    nombre_archivo  STRING,
    url_archivo     STRING,
    hash_md5        STRING,
    fecha_descarga  TIMESTAMP
) USING DELTA
""")

# ── Tabla de métricas de extracción ───────────────────────────────────────
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_EXTRACCION} (
    fuente              STRING,
    archivos_nuevos     INT,
    archivos_omitidos   INT,
    archivos_error      INT,
    kb_descargados      DOUBLE,
    tiempo_segundos     DOUBLE,
    timestamp_inicio    TIMESTAMP,
    timestamp_fin       TIMESTAMP,
    execution_id        STRING,
    framework           STRING,
    llamadas_llm        INT,
    razonamiento_llm    STRING,
    status              STRING
) USING DELTA
""")


def _hash_md5(contenido: bytes) -> str:
    return hashlib.md5(contenido).hexdigest()

def _historial() -> dict:
    """Devuelve dict {(nombre_archivo, framework): hash_md5} de todo lo ya descargado.
    
    CORRECCIÓN #20: Incluye framework para permitir múltiples pruebas con diferentes agentes.
    """
    try:
        rows = spark.table(CONTROL_TABLE).select("nombre_archivo","framework","hash_md5").collect()
        return {(r["nombre_archivo"], r.get("framework", "LangGraph")): r["hash_md5"] for r in rows}
    except:
        return {}

def _guardar_y_registrar(fuente, nombre, url, contenido, ruta_vol, anio=0):
    """
    Guarda el archivo si su hash es nuevo PARA ESTE FRAMEWORK.
    
    CORRECCIÓN #20: Validación por framework permite:
    - Omitir si MISMO framework Y MISMO hash (evita duplicados)
    - Permitir si DIFERENTE framework (nueva prueba experimental)
    
    Retorna: 'nuevo', 'omitido', o 'error'
    """
    historial   = _historial()
    nuevo_hash  = _hash_md5(contenido)
    
    # ⭐ CORRECCIÓN #20: Verificar por framework
    clave_actual = (nombre, FRAMEWORK_NAME)
    
    if historial.get(clave_actual) == nuevo_hash:
        print(f"  ⏭ Sin cambios ({FRAMEWORK_NAME}) → {nombre}")
        return "omitido"
    
    # Verificar si existe con otro framework (permitir, es nueva prueba)
    otros_frameworks = [fw for (n, fw) in historial.keys() if n == nombre and fw != FRAMEWORK_NAME]
    if otros_frameworks:
        print(f"  🆕 Ya existe en {otros_frameworks[0]}, pero permitido para {FRAMEWORK_NAME}")
    
    ruta_dest = f"{ruta_vol}/{nombre}"
    with open(ruta_dest, "wb") as f:
        f.write(contenido)
    
    schema = StructType([
        StructField("fuente",         StringType(),  True),
        StructField("framework",      StringType(),  True),  # ⭐ CORRECCIÓN #20
        StructField("anio",           IntegerType(), True),
        StructField("nombre_archivo", StringType(),  True),
        StructField("url_archivo",    StringType(),  True),
        StructField("hash_md5",       StringType(),  True),
        StructField("fecha_descarga", TimestampType(),True),
    ])
    spark.createDataFrame(
        [(fuente, FRAMEWORK_NAME, int(anio), nombre, url, nuevo_hash, datetime.now())], schema  # ⭐ CORRECCIÓN #20
    ).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(CONTROL_TABLE)
    
    print(f"  ✅ Nuevo ({FRAMEWORK_NAME}) → {nombre}  ({len(contenido)//1024} KB)")
    return "nuevo"

def _registrar_metrica_extraccion(fuente, nuevos, omitidos, errores, kb, t_inicio, t_fin):
    schema = StructType([
        StructField("fuente",            StringType(),  True),
        StructField("archivos_nuevos",   IntegerType(), True),
        StructField("archivos_omitidos", IntegerType(), True),
        StructField("archivos_error",    IntegerType(), True),
        StructField("kb_descargados",    DoubleType(),  True),
        StructField("tiempo_segundos",   DoubleType(),  True),
        StructField("timestamp_inicio",  TimestampType(),True),
        StructField("timestamp_fin",     TimestampType(),True),
    ])
    spark.createDataFrame(
        [(fuente, nuevos, omitidos, errores, round(kb,2),
          round((t_fin-t_inicio).total_seconds(),2), t_inicio, t_fin)], schema
    ).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_EXTRACCION)

print("✅ Infraestructura de control y métricas lista.")

In [ ]:
\
# ══════════════════════════════════════════════════════════════════════════
# EVENTOS DEL AGENTE 1: EXTRACCIÓN
# En LlamaIndex Workflows el estado viaja como eventos tipados (no TypedDict mutable)
# Equivalencia: cada Event = un "estado parcial" del nodo LangGraph correspondiente
# ==============================================================================

class InicioExtraccionEvent(Event):
    fuente_nombre:       str
    fuente_url:          str
    volumen_destino:     str
    keywords_relevantes: List[str]
    timestamp_inicio:    str

class ArchivosListadosEvent(Event):
    fuente_nombre:       str
    fuente_url:          str
    volumen_destino:     str
    keywords_relevantes: List[str]
    archivos:            List[dict]
    timestamp_inicio:    str

class ArchivosSeleccionadosEvent(Event):
    fuente_nombre:       str
    volumen_destino:     str
    archivos_sel:        List[dict]
    razonamiento_llm:    str
    llamadas_api:        int
    timestamp_inicio:    str

class DescargaCompletaEvent(Event):
    fuente_nombre:       str
    archivos_nuevos:     int
    archivos_omitidos:   int
    archivos_error:      int
    kb_descargados:      float
    llamadas_api:        int
    timestamp_inicio:    str

class ExtraccionErrorEvent(Event):
    fuente_nombre:    str
    causa:            str
    llamadas_api:     int
    timestamp_inicio: str

print("OK Eventos AGENTE 1 definidos.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# AGENTE 1: EXTRACTION WORKFLOW
# Misma logica que el grafo LangGraph: listar -> consultar_llm -> descargar -> metricas
# Diferencia SOLO en sintaxis: @step devuelve Event tipado en lugar de dict
# ==============================================================================

class ExtractionWorkflow(Workflow):

    @step
    async def iniciar(self, ctx: Context, ev: StartEvent) -> InicioExtraccionEvent:
        return InicioExtraccionEvent(
            fuente_nombre       = ev.get("fuente_nombre"),
            fuente_url          = ev.get("fuente_url"),
            volumen_destino     = ev.get("volumen_destino"),
            keywords_relevantes = ev.get("keywords_relevantes"),
            timestamp_inicio    = datetime.now().isoformat(),
        )

    # PASO 1: Listar archivos (= nodo_listar_archivos_extraccion en LangGraph)
    @step
    async def listar_archivos(self, ctx: Context, ev: InicioExtraccionEvent) -> ArchivosListadosEvent | ExtraccionErrorEvent:
        print(f"\n{'='*60}")
        print(f"[EXTRACCION PASO 1 - LISTAR] {ev.fuente_nombre}")
        print(f"{'='*60}")
        archivos = []
        try:
            if ev.fuente_nombre == 'ESPAC':
                import datetime as _dt
                anio_actual = _dt.datetime.now().year
                urls_espac_base = [
                    (2025, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/2025/Tabulados_excel_2025.xlsx"),
                    (2024, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/2024/Tabulados_ESPAC_2024.xlsx"),
                    (2023, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/2023/Tabulados_ESPAC_2023.xlsx"),
                    (2022, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/espac_2022/Tabulados ESPAC 2022.xlsx"),
                    (2021, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/espac-2021/Tabulados ESPAC 2021.xlsx"),
                    (2020, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/espac-2020/Tabulados ESPAC 2020.xlsx"),
                    (2019, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/espac-2019/Tabulados ESPAC 2019.xlsx"),
                    (2018, "https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/espac-2018/Tabulados ESPAC 2018.xlsx"),
                ]
                for anio, url in urls_espac_base:
                    archivos.append({"nombre": f"ESPAC_Tabulados_excel_{anio}.xlsx", "url": url, "anio": anio})
                    print(f"    OK {anio}: URL conocida agregada")
                # Intentar anio siguiente
                anio_futuro = anio_actual + 1
                patrones = [
                    f"https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/{anio_futuro}/Tabulados_excel_{anio_futuro}.xlsx",
                    f"https://www.ecuadorencifras.gob.ec/documentos/web-inec/Estadisticas_agropecuarias/espac/{anio_futuro}/Tabulados_ESPAC_{anio_futuro}.xlsx",
                ]
                for i, url_intento in enumerate(patrones, 1):
                    try:
                        r = requests.head(url_intento, headers=HEADERS_HTTP, timeout=10, allow_redirects=True)
                        if r.status_code == 200:
                            archivos.append({"nombre": f"ESPAC_Tabulados_excel_{anio_futuro}.xlsx", "url": url_intento, "anio": anio_futuro})
                            print(f"    {anio_futuro}: URL futura encontrada (patron {i})")
                            break
                    except: pass

            elif ev.fuente_nombre == 'SIPA':
                archivos = [
                    {"nombre": "SIPA_TEMPERATURA_PRECIPITACION.xls",
                     "url": "https://sipa.agricultura.gob.ec/descargas/base-estadistica/modulo_productivo/temperatura-precipitacion.xls"},
                    {"nombre": "SIPA_USO_SUELO.xlsx",
                     "url": "https://sipa.agricultura.gob.ec/descargas/base-estadistica/modulo_productivo/uso-suelo.xlsx"},
                ]

            elif ev.fuente_nombre == 'FAOSTAT':
                import datetime as _dt
                anio_actual = _dt.datetime.now().year
                years = ','.join(str(y) for y in range(1990, anio_actual + 1))
                urls_alt = [
                    f"https://faostatservices.fao.org/api/v1/en/data/QCL?area=58&item=486,489&year={years}",
                    f"https://fenixservices.fao.org/faostat/api/v1/en/data/QCL?area=58&item=486,489&year={years}",
                ]
                url_ok = None
                for i, u in enumerate(urls_alt, 1):
                    try:
                        h = HEADERS_HTTP.copy()
                        h['Authorization'] = f'Bearer {get_faostat_token()}'
                        r = requests.head(u, headers=h, timeout=10)
                        if r.status_code < 400:
                            url_ok = u; print(f"  OK Servidor FAOSTAT {i} accesible"); break
                    except: pass
                archivos = [{"nombre": f"FAOSTAT_BANANAS_PLANTAINS_1990_{anio_actual}.json", "url": url_ok or urls_alt[0]}]
                print(f"  FAOSTAT: consulta 1990-{anio_actual}, 2 productos")

            elif ev.fuente_nombre == 'AEBE_BANANOTAS':
                # Scraping de página Bananotas para obtener PDFs.
                #
                # CAMBIO CLAVE: antes se descargaban TODOS los PDFs históricos
                # (decenas de ediciones desde 2019), y cada uno se procesaba
                # íntegro con tabula-py → de ahí las horas de procesamiento.
                # Ahora solo se descarga la EDICIÓN MÁS RECIENTE.
                #
                # El sitio lista los PDFs agrupados por año en orden
                # cronológico DESCENDENTE (la sección "## 2026" aparece antes
                # que "## 2025", etc.), y dentro de cada año, el primer
                # enlace es la edición más reciente de esa sección. Como solo
                # nos interesa la última edición publicada, basta con tomar
                # el PRIMER PDF único que aparece en el HTML completo.
                #
                # LIMPIEZA PREVIA: si quedan PDFs de ediciones anteriores en
                # el volumen (de corridas previas), el Bloque 6 (ETL) los
                # reprocesaría junto con el nuevo, porque ese bloque recorre
                # TODO lo que encuentra en el volumen, no solo lo recién
                # descargado. Por eso se vacía raw_aebe ANTES de descargar,
                # garantizando que solo exista la edición más reciente.
                try:
                    archivos_previos = [f for f in dbutils.fs.ls(ev.volumen_destino)
                                         if f.name.lower().endswith(".pdf")]
                    for f in archivos_previos:
                        dbutils.fs.rm(f.path)
                    if archivos_previos:
                        print(f"  🗑️  AEBE: {len(archivos_previos)} PDF(s) de ediciones anteriores eliminados del volumen")
                except Exception as e:
                    print(f"  ⚠️  No se pudo limpiar {ev.volumen_destino}: {e}")

                resp = requests.get(ev.fuente_url, headers=HEADERS_HTTP, timeout=30)
                resp.raise_for_status()
                soup = BeautifulSoup(resp.text, "html.parser")

                vistos = set()
                for a in soup.find_all("a", href=True):
                    href = a["href"]
                    if ".pdf" in href.lower() and "62ee00_" in href:
                        url_completa = href if href.startswith("http") else f"https://www.aebe.com.ec{href}"
                        match = re.search(r'62ee00_([a-f0-9]+)', href)
                        hash_id = match.group(1)[:12] if match else "unknown"
                        if hash_id in vistos:
                            continue  # el mismo PDF se repite varias veces en el HTML (portada + grid)
                        vistos.add(hash_id)
                        nombre = f"AEBE_BANANOTAS_{hash_id}.pdf"
                        archivos.append({"nombre": nombre, "url": url_completa})
                        break  # ⭐ solo la edición más reciente — no seguir recolectando

                print(f"  AEBE Bananotas: {len(archivos)} PDF (solo edición más reciente)")

            print(f"  Archivos encontrados: {len(archivos)}")
            return ArchivosListadosEvent(
                fuente_nombre=ev.fuente_nombre, fuente_url=ev.fuente_url,
                volumen_destino=ev.volumen_destino, keywords_relevantes=ev.keywords_relevantes,
                archivos=archivos, timestamp_inicio=ev.timestamp_inicio,
            )
        except Exception as e:
            print(f"  ERROR: {e}")
            return ExtraccionErrorEvent(fuente_nombre=ev.fuente_nombre, causa=str(e),
                                        llamadas_api=0, timestamp_inicio=ev.timestamp_inicio)

    # PASO 2: Consultar LLM (= nodo_consultar_llm_seleccion en LangGraph)
    @step
    async def consultar_llm(self, ctx: Context, ev: ArchivosListadosEvent) -> ArchivosSeleccionadosEvent:
        print(f"\n[EXTRACCION PASO 2 - CONSULTAR LLM]")
        if not ev.archivos:
            return ArchivosSeleccionadosEvent(fuente_nombre=ev.fuente_nombre,
                volumen_destino=ev.volumen_destino, archivos_sel=[],
                razonamiento_llm="Sin archivos", llamadas_api=0, timestamp_inicio=ev.timestamp_inicio)
        try:
            muestra = ev.archivos[:100]
            archivos_json = json.dumps(
                [{"nombre": a['nombre'], "url": a['url'][:80]} for a in muestra],
                indent=2, ensure_ascii=False
            )
            prompt = f"""Eres un experto en datos del sector bananero de Ecuador.

FUENTE: {ev.fuente_nombre}
KEYWORDS: {', '.join(ev.keywords_relevantes)}
ARCHIVOS ({len(muestra)} de {len(ev.archivos)} totales):
{archivos_json}

REGLAS:
- ESPAC: seleccionar TODOS los archivos Tabulados/Series de TODOS los anios. NO filtrar por anio.
- FAOSTAT: seleccionar todos los anios disponibles.
- BANANA_TRADERS: solo archivos con 'acorbanec' o 'embarque' o 'precios'.
- SIPA: temperatura, precipitacion, uso de suelo.

Responde SOLO con JSON valido (sin markdown):
{{"archivos_seleccionados": [{{"nombre": "archivo.xlsx", "razon": "motivo"}}],
  "razonamiento": "tu explicacion"}}"""

            resp = llm.invoke([HumanMessage(content=prompt)])
            llamadas = 1
            texto = re.sub(r"^```json\s*|^```\s*|```$","",resp.content.strip(),flags=re.MULTILINE).strip()
            s = texto.find("{"); e = texto.rfind("}")
            if s != -1 and e != -1:
                resultado = json.loads(texto[s:e+1])
                nombres_sel = {item['nombre'] for item in resultado.get("archivos_seleccionados", [])}
                archivos_finales = [a for a in ev.archivos if a['nombre'] in nombres_sel]
                print(f"  LLM selecciono {len(archivos_finales)} archivos")
                return ArchivosSeleccionadosEvent(
                    fuente_nombre=ev.fuente_nombre, volumen_destino=ev.volumen_destino,
                    archivos_sel=archivos_finales,
                    razonamiento_llm=resultado.get("razonamiento","")[:500],
                    llamadas_api=llamadas, timestamp_inicio=ev.timestamp_inicio)
        except Exception as e:
            print(f"  LLM fallo: {e} -> fallback: todos los archivos")
        return ArchivosSeleccionadosEvent(
            fuente_nombre=ev.fuente_nombre, volumen_destino=ev.volumen_destino,
            archivos_sel=ev.archivos,
            razonamiento_llm=f"Fallback: LLM fallo, usando todos ({len(ev.archivos)})",
            llamadas_api=0, timestamp_inicio=ev.timestamp_inicio)

    # PASO 3: Descargar archivos (= nodo_descargar_archivos_extraccion en LangGraph)
    @step
    async def descargar_archivos(self, ctx: Context, ev: ArchivosSeleccionadosEvent) -> DescargaCompletaEvent | ExtraccionErrorEvent:
        print(f"\n[EXTRACCION PASO 3 - DESCARGAR]")
        nuevos, omitidos, errores, kb_total = 0, 0, 0, 0.0
        if not ev.archivos_sel:
            return DescargaCompletaEvent(fuente_nombre=ev.fuente_nombre, archivos_nuevos=0,
                archivos_omitidos=0, archivos_error=0, kb_descargados=0.0,
                llamadas_api=ev.llamadas_api, timestamp_inicio=ev.timestamp_inicio)
        try:
            from urllib.parse import urlparse, quote as _quote
            for archivo in ev.archivos_sel:
                nombre_orig = archivo['nombre']
                url = archivo['url']
                try:
                    prefijo = {'ESPAC':'ESPAC_','SIPA':'SIPA_','FAOSTAT':'FAOSTAT_',
                               'BANANA_TRADERS':'BANANO_PRECIOS_'}.get(ev.fuente_nombre,'')
                    nombre_limpio = re.sub(r"[^a-zA-Z0-9_\-\.]","_", nombre_orig)
                    nombre_final = f"{prefijo}{nombre_limpio}" if not nombre_orig.startswith(prefijo.rstrip('_')) else nombre_orig
                    parsed = urlparse(url)
                    url_enc = f"{parsed.scheme}://{parsed.netloc}{_quote(parsed.path, safe='/:')}"
                    if parsed.query: url_enc += f"?{parsed.query}"
                    headers = HEADERS_HTTP.copy()
                    if ev.fuente_nombre == 'FAOSTAT':
                        headers['Authorization'] = f'Bearer {get_faostat_token()}'
                    r = requests.get(url_enc, headers=headers, timeout=90)
                    r.raise_for_status()
                    contenido = r.content
                    kb_total += len(contenido) / 1024
                    res = _guardar_y_registrar(ev.fuente_nombre, nombre_final, url, contenido,
                                               ev.volumen_destino, anio=archivo.get('anio', 0))
                    if res == "nuevo":     nuevos   += 1
                    elif res == "omitido": omitidos += 1
                except Exception as e:
                    errores += 1; print(f"  ERROR {nombre_orig}: {e}")
            print(f"  Resumen: {nuevos} nuevos, {omitidos} omitidos, {errores} errores ({kb_total:.0f} KB)")
            return DescargaCompletaEvent(fuente_nombre=ev.fuente_nombre,
                archivos_nuevos=nuevos, archivos_omitidos=omitidos, archivos_error=errores,
                kb_descargados=kb_total, llamadas_api=ev.llamadas_api, timestamp_inicio=ev.timestamp_inicio)
        except Exception as e:
            return ExtraccionErrorEvent(fuente_nombre=ev.fuente_nombre, causa=str(e),
                                        llamadas_api=ev.llamadas_api, timestamp_inicio=ev.timestamp_inicio)

    # PASO 4: Metricas (= nodo_metricas_extraccion en LangGraph)
    @step
    async def registrar_metricas(self, ctx: Context, ev: DescargaCompletaEvent) -> StopEvent:
        print(f"\n[EXTRACCION PASO 4 - METRICAS] {ev.fuente_nombre}")
        ts_ini = datetime.fromisoformat(ev.timestamp_inicio)
        ts_fin = datetime.now()
        tiempo = (ts_fin - ts_ini).total_seconds()
        schema = StructType([
            StructField("execution_id",      StringType(),   True),
            StructField("fuente",            StringType(),   True),
            StructField("framework",         StringType(),   True),
            StructField("archivos_nuevos",   IntegerType(),  True),
            StructField("archivos_omitidos", IntegerType(),  True),
            StructField("archivos_error",    IntegerType(),  True),
            StructField("kb_descargados",    DoubleType(),   True),
            StructField("tiempo_segundos",   DoubleType(),   True),
            StructField("timestamp_inicio",  TimestampType(), True),
            StructField("timestamp_fin",     TimestampType(), True),
            StructField("llamadas_llm",      IntegerType(),  True),
            StructField("razonamiento_llm",  StringType(),   True),
            StructField("status",            StringType(),   True),
        ])
        spark.createDataFrame([(
            EXECUTION_ID, ev.fuente_nombre, FRAMEWORK_NAME,
            ev.archivos_nuevos, ev.archivos_omitidos, ev.archivos_error,
            round(ev.kb_descargados, 2), round(tiempo, 2), ts_ini, ts_fin,
            ev.llamadas_api, "LlamaIndex Workflow", "OK"
        )], schema).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_EXTRACCION)
        print(f"  OK Metricas guardadas: {tiempo:.1f}s, {ev.archivos_nuevos} nuevos")
        return StopEvent(result={
            "status": "OK", "fuente_nombre": ev.fuente_nombre,
            "archivos_nuevos": ev.archivos_nuevos, "archivos_omitidos": ev.archivos_omitidos,
            "archivos_error": ev.archivos_error, "kb_descargados": ev.kb_descargados,
            "tiempo_segundos": tiempo, "llamadas_api": ev.llamadas_api,
        })

    @step
    async def manejar_error(self, ctx: Context, ev: ExtraccionErrorEvent) -> StopEvent:
        print(f"\nERROR EXTRACCION {ev.fuente_nombre}: {ev.causa}")
        return StopEvent(result={"status":"ERROR","fuente_nombre":ev.fuente_nombre,"error":ev.causa})

print("OK ExtractionWorkflow (AGENTE 1) definido.")
print("   Flujo: iniciar -> listar_archivos -> consultar_llm -> descargar_archivos -> registrar_metricas -> END")


In [ ]:
# ==============================================================================
# ORQUESTADOR AGENTE 1 - ejecutar_agente1_extraccion()
# Equivalente a ejecutar_extraccion_todas_fuentes() de LangGraph
# En lugar de graph.invoke() usa await workflow.run()
# ==============================================================================

import asyncio

async def ejecutar_agente1_extraccion():
    """Ejecuta ExtractionWorkflow para las fuentes en secuencia."""
    fuentes = [
        {
            "fuente_nombre":       "ESPAC",
            "fuente_url":          "https://www.ecuadorencifras.gob.ec/encuesta-de-superficie-y-produccion-agropecuaria-continua-espac/",
            "volumen_destino":     RAW_PATH_ESPAC,
            "keywords_relevantes": ["banano","platano","musaceas","produccion","superficie","tabulados","series"],
        },
        {
            "fuente_nombre":       "SIPA",
            "fuente_url":          "https://sipa.agricultura.gob.ec/estadisticas-agropecuarias",
            "volumen_destino":     RAW_PATH_SIPA,
            "keywords_relevantes": ["temperatura","precipitacion","uso suelo","climatico"],
        },
        {
            "fuente_nombre":       "FAOSTAT",
            "fuente_url":          "https://faostatservices.fao.org/api/v1/en/data/QCL",
            "volumen_destino":     RAW_PATH_FAOSTAT,
            "keywords_relevantes": ["bananas","plantains","production","area harvested","yield","Ecuador"],
        },
        {
            "fuente_nombre":       "AEBE_BANANOTAS",
            "fuente_url":          "https://www.aebe.com.ec/bananotas",
            "volumen_destino":     RAW_PATH_AEBE,
            "keywords_relevantes": ["exportaciones","regiones","estadisticas","cajas","participacion"],
        },
    ]

    resultados = []
    ts_global = datetime.now()

    for fuente in fuentes:
        print(f"\n{'#'*70}")
        print(f"# AGENTE 1 - EXTRACCION: {fuente['fuente_nombre']}")
        print(f"{'#'*70}")
        try:
            wf = ExtractionWorkflow(timeout=600, verbose=False)
            resultado = await wf.run(**fuente)
            resultados.append(resultado)
            status = resultado.get('status','?') if resultado else 'ERROR'
            nuevos  = resultado.get('archivos_nuevos', 0) if resultado else 0
            omit    = resultado.get('archivos_omitidos', 0) if resultado else 0
            kb      = resultado.get('kb_descargados', 0) if resultado else 0
            print(f"  => Status: {status} | Nuevos: {nuevos} | Omitidos: {omit} | KB: {kb:.0f}")
        except Exception as e:
            print(f"  ERROR en {fuente['fuente_nombre']}: {e}")
            resultados.append({"status":"ERROR","fuente_nombre":fuente['fuente_nombre'],"error":str(e)})

    tiempo_total = (datetime.now() - ts_global).total_seconds()
    ok  = sum(1 for r in resultados if r and r.get('status') == 'OK')
    err = sum(1 for r in resultados if r and r.get('status') == 'ERROR')
    nuevos_tot = sum(r.get('archivos_nuevos', 0) for r in resultados if r)
    kb_tot     = sum(r.get('kb_descargados', 0)  for r in resultados if r)
    print(f"\n{'='*70}")
    print(f"EXTRACCION COMPLETADA en {tiempo_total:.1f}s")
    print(f"  Fuentes OK: {ok}/{len(fuentes)} | Errores: {err} | Archivos nuevos: {nuevos_tot} | Total: {kb_tot:.0f} KB")
    print(f"{'='*70}")
    return resultados

# Ejecutar
import nest_asyncio
nest_asyncio.apply()
resultados_extraccion = asyncio.run(ejecutar_agente1_extraccion())


In [ ]:
# ── Métricas de Extracción ─────────────────────────────────────────────────
print("\n📊 MÉTRICAS DE EXTRACCIÓN — por fuente:")
spark.table(LOG_EXTRACCION).orderBy("timestamp_fin", ascending=False).display()

print("\n📊 RESUMEN TOTAL DE EXTRACCIÓN:")
spark.sql(f"""
SELECT
    SUM(archivos_nuevos)    AS total_archivos_nuevos,
    SUM(archivos_omitidos)  AS total_omitidos,
    SUM(archivos_error)     AS total_errores,
    ROUND(SUM(kb_descargados)/1024, 2) AS mb_descargados,
    ROUND(SUM(tiempo_segundos), 1)     AS tiempo_total_s
FROM {LOG_EXTRACCION}
""").display()

## 📐 Bloque 4 — Dimensiones

Crea las tablas dimensionales compartidas: `dim_productos`, `dim_provincias`, `dim_tiempo`.
Identico al LangGraph — estas tablas son independientes del framework de orquestación.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# TABLAS DIMENSIONALES: REGIONES Y PROVINCIAS DE ECUADOR
# Esta estructura maestra normaliza los datos geográficos
# ══════════════════════════════════════════════════════════════════════════

print("📍 Creando/verificando tablas dimensionales...\n")

# ──────────────────────────────────────────────────────────────────────────
# 1️⃣ TABLA DIMENSIONAL: REGIONES
# ──────────────────────────────────────────────────────────────────────────
tabla_dim_regiones = f"{DB_NAME}.dim_regiones"

if spark.catalog.tableExists(tabla_dim_regiones):
    print(f"🌍 1. Tabla {tabla_dim_regiones} ya existe")
    count_regiones = spark.table(tabla_dim_regiones).count()
    print(f"   ℹ️  {count_regiones} regiones registradas (sin cambios)\n")
else:
    print("🌍 1. Creando dim_regiones...")
    
    regiones_data = [
        (1, "Sierra", "Región Interandina"),
        (2, "Costa", "Región Litoral"),
        (3, "Amazonía", "Región Amazónica"),
        (4, "Insular", "Región Insular - Galápagos"),
    ]
    
    schema_regiones = StructType([
        StructField("region_id", IntegerType(), False),
        StructField("region", StringType(), False),
        StructField("descripcion", StringType(), True),
    ])
    
    df_regiones = spark.createDataFrame(regiones_data, schema=schema_regiones)
    df_regiones.write.mode("overwrite").saveAsTable(tabla_dim_regiones)
    
    print(f"   ✅ Tabla creada: {tabla_dim_regiones}")
    print(f"   📊 {df_regiones.count()} regiones registradas\n")

# ──────────────────────────────────────────────────────────────────────────
# 2️⃣ TABLA DIMENSIONAL: PROVINCIAS (con FK a regiones)
# ──────────────────────────────────────────────────────────────────────────
tabla_dim_provincias = f"{DB_NAME}.dim_provincias"

if spark.catalog.tableExists(tabla_dim_provincias):
    print(f"🗺️  2. Tabla {tabla_dim_provincias} ya existe")
    count_provincias = spark.table(tabla_dim_provincias).count()
    print(f"   ℹ️  {count_provincias} provincias registradas (sin cambios)\n")
else:
    print("🗺️  2. Creando dim_provincias...")
    
    provincias_data = [
        (1, "Azuay", 1),
        (2, "Bolívar", 1),
        (3, "Cañar", 1),
        (4, "Carchi", 1),
        (5, "Cotopaxi", 1),
        (6, "Chimborazo", 1),
        (7, "El Oro", 2),
        (8, "Esmeraldas", 2),
        (9, "Guayas", 2),
        (10, "Imbabura", 1),
        (11, "Loja", 1),
        (12, "Los Ríos", 2),
        (13, "Manabí", 2),
        (14, "Morona Santiago", 3),
        (15, "Napo", 3),
        (16, "Pastaza", 3),
        (17, "Pichincha", 1),
        (18, "Tungurahua", 1),
        (19, "Zamora Chinchipe", 3),
        (20, "Galápagos", 4),
        (21, "Sucumbíos", 3),
        (22, "Orellana", 3),
        (23, "Santo Domingo De Los Tsáchilas", 1),
        (24, "Santa Elena", 2),
    ]
    
    schema_provincias = StructType([
        StructField("provincia_id", IntegerType(), False),
        StructField("provincia", StringType(), False),
        StructField("region_id", IntegerType(), False),
    ])
    
    df_provincias = spark.createDataFrame(provincias_data, schema=schema_provincias)
    
    # Agregar variantes normalizadas para el mapeo (sin tildes, minúsculas)
    df_provincias = df_provincias.withColumn(
        "provincia_normalizada",
        F.lower(F.translate(F.col("provincia"), "áéíóúÁÉÍÓÚñÑ", "aeiouAEIOUnn"))
    )
    
    df_provincias.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabla_dim_provincias)
    
    print(f"   ✅ Tabla creada: {tabla_dim_provincias}")
    print(f"   📊 {df_provincias.count()} provincias registradas\n")

# ──────────────────────────────────────────────────────────────────────────
# VISTA PREVIA
# ──────────────────────────────────────────────────────────────────────────
print("🔍 Vista previa de la estructura dimensional:\n")

print("📊 Consulta unificada (con JOIN):")
spark.sql(f"""
SELECT 
    p.provincia_id,
    p.provincia,
    r.region_id,
    r.region,
    r.descripcion AS region_descripcion
FROM {tabla_dim_provincias} p
JOIN {tabla_dim_regiones} r ON p.region_id = r.region_id
ORDER BY r.region_id, p.provincia_id
""").display()

print("\n✅ Estructura dimensional creada exitosamente!")
print("\n📝 Resumen:")
print("   • 4 regiones (Sierra, Costa, Amazonía, Insular)")
print("   • 24 provincias con FK a regiones")
print("   • provincia_normalizada para mapeo automático")

## 🔧 Bloque 5 — Funciones Utilitarias y Transformaciones Especializadas

Funciones compartidas entre LangGraph y LlamaIndex (independientes del framework):

- `normalizar_columna`, `identificar_fuente`
- `_aplanar_excel_openpyxl`, `_score_header`, `leer_archivo`
- `castear_columnas`, `mapear_provincia_a_id`
- `transformar_espac_t13_t26_mejorado_v3`
- `transformar_sipa_temperatura`, `transformar_uso_del_suelo`
- `transformar_faostat`, `transformar_precios_v2`

In [ ]:
def mapear_provincia_a_id(df_pandas: pd.DataFrame, columna_provincia: str = 'provincia') -> pd.DataFrame:
    """
    Mapea nombres de provincias a sus IDs usando la tabla dimensional.
    
    Args:
        df_pandas: DataFrame con datos que incluyen nombres de provincias
        columna_provincia: Nombre de la columna que contiene el nombre de provincia
    
    Returns:
        DataFrame con provincia_id en lugar de nombre de provincia
    """
    if columna_provincia not in df_pandas.columns:
        print(f"  ⚠️  Columna '{columna_provincia}' no encontrada, omitiendo mapeo")
        return df_pandas
    
    print(f"  🗺️  Mapeando provincias a IDs...")
    
    # Leer tabla dimensional como pandas para el mapeo
    dim_prov_pd = spark.table(f"{DB_NAME}.dim_provincias").toPandas()
    
    # Crear diccionario de mapeo (provincia_normalizada -> provincia_id)
    mapeo_dict = dict(zip(dim_prov_pd['provincia_normalizada'], dim_prov_pd['provincia_id']))
    
    # Normalizar la columna de provincia en el DataFrame de datos
    def normalizar_provincia(texto):
        if pd.isna(texto) or texto == '':
            return None
        # Convertir a string, quitar espacios, minusculas
        texto = str(texto).strip().lower()
        # Quitar tildes
        texto = texto.translate(str.maketrans('áéíóúÁÉÍÓÚñÑ', 'aeiouAEIOUnn'))
        return texto
    
    df_pandas['_provincia_norm'] = df_pandas[columna_provincia].apply(normalizar_provincia)
    
    # Mapear a provincia_id
    df_pandas['provincia_id'] = df_pandas['_provincia_norm'].map(mapeo_dict)
    
    # Verificar cuántas NO mapearon
    sin_mapear = df_pandas['provincia_id'].isna().sum()
    if sin_mapear > 0:
        print(f"    ⚠️  {sin_mapear} registros sin mapeo de provincia")
        provincias_no_mapeadas = df_pandas[df_pandas['provincia_id'].isna()][columna_provincia].unique()
        print(f"       Provincias no reconocidas: {list(provincias_no_mapeadas)[:5]}")
    
    # Eliminar columnas auxiliares y original
    df_pandas = df_pandas.drop(columns=['_provincia_norm', columna_provincia])
    
    # Convertir provincia_id a int (donde no sea null)
    df_pandas['provincia_id'] = df_pandas['provincia_id'].astype('Int64')
    
    mapeados = (~df_pandas['provincia_id'].isna()).sum()
    print(f"    ✅ {mapeados} provincias mapeadas correctamente")
    
    return df_pandas

print("✅ Función mapear_provincia_a_id() definida.")

In [ ]:
def normalizar_columna(col_name: str) -> str:
    c = str(col_name).lower().strip()
    c = c.replace(" ","_").replace("-","_")
    c = ''.join(ch for ch in unicodedata.normalize('NFD', c) if unicodedata.category(ch) != 'Mn')
    c = re.sub(r'[^a-z0-9_]', '', c)
    c = re.sub(r'_+', '_', c).strip('_')
    return c if c else "columna_sin_nombre"

def identificar_fuente(file_name: str) -> str:
    n = file_name.upper()
    if n.startswith("ESPAC_"):         return "INEC_ESPAC"
    if n.startswith("TEMPERATURA_"):   return "SIPA_MAG"
    if n.startswith("SIPA_"):          return "SIPA_MAG"
    if n.startswith("FAOSTAT_"):       return "FAOSTAT"
    if n.startswith("AEBE_"):          return "AEBE_BANANOTAS"
    return "DESCONOCIDA"

def _aplanar_excel_openpyxl(local_path: str) -> pd.DataFrame:
    """Expande celdas combinadas antes de leer — necesario para archivos ESPAC."""
    from openpyxl import load_workbook
    wb = load_workbook(local_path, data_only=True)
    ws = wb.active
    for mr in list(ws.merged_cells.ranges):
        val = ws.cell(mr.min_row, mr.min_col).value
        ws.unmerge_cells(str(mr))
        for r in range(mr.min_row, mr.max_row + 1):
            for c in range(mr.min_col, mr.max_col + 1):
                ws.cell(r, c).value = val
    data = [list(row) for row in ws.iter_rows(values_only=True)]
    wb.close()
    if not data: return pd.DataFrame()
    max_cols = max(len(r) for r in data)
    for r in data:
        while len(r) < max_cols: r.append(None)
    df = pd.DataFrame(data)
    return df.astype(str).replace({"None":"","nan":"","NaT":""})

def _score_header(raw_df: pd.DataFrame, row_idx: int) -> int:
    """Puntúa la probabilidad de que una fila sea el encabezado real de la tabla."""
    if row_idx >= len(raw_df): return -99
    vals  = [str(v).strip() for v in raw_df.iloc[row_idx].values]
    score = 0
    ratio = sum(1 for v in vals if v == "") / max(len(vals), 1)
    score += 3 if ratio == 0 else (1 if ratio <= 0.25 else -2)
    KEYWORDS = ["áño","anio","year","provincia","producto","cultivo","superficie",
                "area","produccion","tonelada","rendimiento","precio","valor","zona"]
    row_text = " ".join(vals).lower()
    for a, b in [("á","a"),("é","e"),("í","i"),("ó","o"),("ú","u"),("ñ","n")]:
        row_text = row_text.replace(a, b)
    score += min(sum(1 for kw in KEYWORDS if kw in row_text) * 2, 10)
    n_etiq = sum(1 for v in vals if v and len(v)<=50 and not
                 v.replace(".","").replace(",","").replace("-","").replace(" ","").isdigit())
    if n_etiq / max(len(vals),1) >= 0.65: score += 2
    if row_idx+1 < len(raw_df):
        nxt   = [str(v).strip() for v in raw_df.iloc[row_idx+1].values]
        n_num = sum(1 for v in nxt if v and v.replace(".","").replace(",","").replace("-","").isdigit())
        if n_num / max(len(nxt),1) >= 0.25: score += 4
    if any(re.search(p, row_text) for p in [r"fuente\s*:",r"nota\s*:",r"ministerio",r"encuesta",r"http"]):
        score -= 4
    return score

def leer_archivo(local_path: str, file_name: str) -> pd.DataFrame:
    """
    Lee el archivo y retorna un DataFrame con columnas normalizadas y header detectado.
    CORRECCIÓN #5 MEJORADO: Usa detección automática limitada para SIPA en lugar de hardcodear filas.
    """
    ext = file_name.lower()
    
    # ── PDF (AEBE BANANOTAS) ──────────────────────────────
    # Los PDFs se manejan directamente en transformar_aebe_bananotas
    if ext.endswith(".pdf"):
        print(f"  📝 PDF detectado - se procesará con transformación especializada")
        return pd.DataFrame({"_es_pdf": [True]})  # Placeholder
    
    # ── CORRECCIÓN #5 MEJORADO: SIPA TEMPERATURA/PRECIPITACIÓN ──────────────
    if "SIPA_TEMPERATURA" in file_name.upper():
        print(f"  🔧 SIPA Temperatura: Detección automática de header (limitada a primeras 10 filas)")
        df_raw = pd.read_excel(local_path, header=None, dtype=str, engine="xlrd")
        
        # Buscar header solo en primeras 10 filas
        scores = {i: _score_header(df_raw, i) for i in range(min(10, len(df_raw)))}
        mejor = max(scores, key=scores.get)
        print(f"    ✅ Header detectado en fila {mejor} (score: {scores[mejor]})")
        
        # Extraer header y datos
        header_vals = list(df_raw.iloc[mejor].values)
        col_names = [normalizar_columna(str(v)) if str(v).strip() else f"col_{i}" for i, v in enumerate(header_vals)]
        
        df = df_raw.iloc[mejor+1:].copy()
        df.columns = col_names
        df = df.reset_index(drop=True)
        
        # Limpiar SOLO filas completamente vacías (mejora clave: no usar .dropna(how='all') que es demasiado agresivo)
        df = df.replace(r'^\s*$', np.nan, regex=True)  # Convertir strings vacíos a NaN
        df = df[df.notna().any(axis=1)]  # Filtrar filas donde TODAS las columnas son NaN
        
        print(f"    📊 Registros después de limpieza: {len(df):,}")
        return df.reset_index(drop=True)
    
    # ── CORRECCIÓN #5 MEJORADO: SIPA USO DEL SUELO ──────────────────────────
    if "SIPA_USO" in file_name.upper() or "ESPAC_SIPA_USO" in file_name.upper():
        print(f"  🔧 SIPA Uso del Suelo: Detección automática de header (limitada a primeras 10 filas)")
        df_raw = pd.read_excel(local_path, header=None, dtype=str, engine="openpyxl")
        
        # Buscar header solo en primeras 10 filas
        scores = {i: _score_header(df_raw, i) for i in range(min(10, len(df_raw)))}
        mejor = max(scores, key=scores.get)
        print(f"    ✅ Header detectado en fila {mejor} (score: {scores[mejor]})")
        
        # Extraer header y datos
        header_vals = list(df_raw.iloc[mejor].values)
        col_names = [normalizar_columna(str(v)) if str(v).strip() else f"col_{i}" for i, v in enumerate(header_vals)]
        
        df = df_raw.iloc[mejor+1:].copy()
        df.columns = col_names
        df = df.reset_index(drop=True)
        
        # Limpiar SOLO filas completamente vacías
        df = df.replace(r'^\s*$', np.nan, regex=True)
        df = df[df.notna().any(axis=1)]
        
        print(f"    📊 Registros después de limpieza: {len(df):,}")
        return df.reset_index(drop=True)
    
    # ── JSON (FAOSTAT) ─────────────────────────────────────────────────
    if ext.endswith(".json"):
        with open(local_path, 'r') as f:
            data = json.load(f)
        
        if isinstance(data, dict) and 'data' in data:
            records = data['data']
        elif isinstance(data, list):
            records = data
        else:
            raise ValueError(f"Formato JSON no reconocido: {file_name}")
        
        if not records:
            return pd.DataFrame()
        
        df = pd.DataFrame(records)
        df.columns = [normalizar_columna(c) for c in df.columns]
        return df
    
    # ── EXCEL ────────────────────────────────────────────────────────
    elif ext.endswith(".xls"):
        raw = pd.read_excel(local_path, header=None, dtype=str, engine="xlrd")
    elif ext.endswith(".xlsx"):
        try:   raw = _aplanar_excel_openpyxl(local_path)
        except: raw = pd.read_excel(local_path, header=None, dtype=str, engine="openpyxl")
    
    # ── CSV ───────────────────────────────────────────────────────────
    elif ext.endswith(".csv"):
        sep = ';' if any(x in file_name.upper() for x in ['T13', 'T26']) else ','
        
        for encoding in ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']:
            try:
                raw = pd.read_csv(local_path, header=None, dtype=str, encoding=encoding, sep=sep, on_bad_lines='skip')
                break
            except (UnicodeDecodeError, LookupError):
                continue
        else:
            raw = pd.read_csv(local_path, header=None, dtype=str, encoding='latin1', errors='ignore', sep=sep, on_bad_lines='skip')
    else:
        raise ValueError(f"Formato no soportado: {file_name}")

    # ── DETECCIÓN AUTOMÁTICA DE HEADER (para Excel/CSV no-SIPA) ───────────────────
    raw = raw.astype(str).replace({"None":"","nan":"","NaT":""})
    scores = {i: _score_header(raw, i) for i in range(min(40, len(raw)))}
    mejor  = max(scores, key=scores.get)
    if scores[mejor] < 3: mejor = 0

    header_vals = list(raw.iloc[mejor].values)
    seen, col_names = {}, []
    for v in header_vals:
        v = normalizar_columna(str(v)) if str(v).strip() else "col_sin_nombre"
        seen[v] = seen.get(v,0) + 1
        col_names.append(f"{v}_{seen[v]-1}" if seen[v]>1 else v)

    df = raw.iloc[mejor+1:].copy()
    df.columns = col_names
    df = df.reset_index(drop=True)
    df = df.replace(r'^\s*[\.-]*\s*$', np.nan, regex=True)
    df = df.dropna(how="all")  # Esto es OK para archivos no-SIPA
    return df

def castear_columnas(df_spark, cols_double, cols_integer):
    """Castea columnas numéricas tolerando comas como separador decimal."""
    PROTEGIDAS = {"mes","meses","ano","áño","provincia","canton"}
    for c in cols_double:
        if c in df_spark.columns and c not in PROTEGIDAS:
            df_spark = (df_spark
                .withColumn(c, F.regexp_replace(F.col(c), r'\s+', ''))
                .withColumn(c, F.regexp_replace(F.col(c), ',', '.'))
                .withColumn(c, F.when(F.col(c)=='.', None).otherwise(F.col(c)))
                .withColumn(c, F.expr(f"try_cast(`{c}` as double)")))
    for c in cols_integer:
        if c in df_spark.columns and c not in PROTEGIDAS:
            df_spark = (df_spark
                .withColumn(c, F.regexp_replace(F.col(c), r'\..*', ''))
                .withColumn(c, F.regexp_replace(F.col(c), r'\s+', ''))
                .withColumn(c, F.when(F.col(c)=='.', None).otherwise(F.col(c)))
                .withColumn(c, F.expr(f"try_cast(`{c}` as integer)")))
    return df_spark

print("✅ Funciones utilitarias cargadas.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TRANSFORMACIONES ESPECIALIZADAS PARA ARCHIVOS COMPLEJOS
# ══════════════════════════════════════════════════════════════════════════════

# Funciones especializadas (banano_precios_semanales eliminado del flujo)

# PLACEHOLDER: transformar_precios ya no se usa
# (función eliminada - ya no procesamos archivos de precios de BANANA_TRADERS)


# CORRECCIÓN #8: Mejora de transformación ESPAC para múltiples hojas (UNION SIN AMBIGÜEDAD)
def transformar_espac_t13_t26_mejorado_v2(local_path: str, nombre_archivo: str) -> pd.DataFrame:
    """
    Transforma archivos ESPAC con múltiples hojas (Tabulados_excel, Series_historicas).
    Detecta automáticamente hojas T13 (banano) y T26 (plátano), aplanando headers fusionados.
    """
    import numpy as np
    import xlrd  # Usar xlrd para leer valores reales, no fórmulas
    
    print(f"  📊 Transformación ESPAC Tabulados/Series Históricas: {nombre_archivo}")
    
    # Extraer año del nombre del archivo
    anio_match = re.search(r'(\d{4})', nombre_archivo)
    anio = int(anio_match.group(1)) if anio_match else None
    print(f"    📅 Año extraído del archivo: {anio if anio else 'No detectado'}")
    
    try:
        # Leer con pandas/xlrd para obtener nombres de hojas
        xl_file = pd.ExcelFile(local_path)
        nombres_hojas = xl_file.sheet_names
        
        print(f"    📂 Hojas disponibles: {nombres_hojas}")
        
        # Buscar hojas que contengan T13 o T26 en el nombre
        hojas_relevantes = []
        for nombre_hoja in nombres_hojas:
            nombre_upper = nombre_hoja.upper()
            if 'T13' in nombre_upper or 'T26' in nombre_upper or 'BANANO' in nombre_upper or 'PLATANO' in nombre_upper or 'PLÁTANO' in nombre_upper:
                hojas_relevantes.append(nombre_hoja)
        
        print(f"    ✅ Hojas relevantes identificadas: {hojas_relevantes if hojas_relevantes else 'Ninguna, usando hoja activa'}")
        
        dfs_procesados = []
        
        # Si no se encontraron hojas específicas, usar la hoja activa
        if not hojas_relevantes:
            hojas_relevantes = [wb.active.title]
        
        for nombre_hoja in hojas_relevantes:
            print(f"    📄 Procesando hoja: {nombre_hoja}")
            
            try:
                # CORRECCIÓN #9: Leer con pandas directamente (sin openpyxl)
                # Esto lee los VALORES tal como están en las celdas, no códigos internos
                df_temp = pd.read_excel(local_path, sheet_name=nombre_hoja, header=None)
                
                if df_temp.empty:
                    print(f"       ⚠️  Hoja vacía, saltando...")
                    continue
                
                # Convertir todo a string para procesamiento uniforme
                df_temp = df_temp.astype(str).replace({"None":"","nan":"","NaT":"","<NA>":""})
                
                # Buscar fila de header (contiene palabras clave)
                header_idx = 0
                for i in range(min(20, len(df_temp))):
                    fila_text = ' '.join(str(v) for v in df_temp.iloc[i].values).upper()
                    if any(kw in fila_text for kw in ['PROVINCIA', 'SUPERFICIE', 'PRODUCCIÓN', 'PRODUCCION', 'REGION', 'REGIÓN']):
                        header_idx = i
                        print(f"       🎯 Header detectado en fila {header_idx}")
                        break
                
                # Extraer header y datos
                header_vals = list(df_temp.iloc[header_idx].values)
                df_datos = df_temp.iloc[header_idx+1:].copy()
                
                # Normalizar nombres de columnas Y HACER ÚNICOS
                col_names = [normalizar_columna(str(v)) if str(v).strip() else f"col_{i}" for i, v in enumerate(header_vals)]
                # Verificar duplicados y hacer únicos agregando sufijos
                seen = {}
                unique_names = []
                for name in col_names:
                    if name in seen:
                        seen[name] += 1
                        unique_names.append(f"{name}_{seen[name]}")
                    else:
                        seen[name] = 0
                        unique_names.append(name)
                
                df_datos.columns = unique_names
                df_datos = df_datos.reset_index(drop=True)
                
                # Filtrar filas de notas/totales
                palabras_excluir = ['TOTAL', 'REGIÓN', 'REGION', 'ZONA', 'NOTA', 'FUENTE', 'OBSERVACIÓN', 'OBSERVACION', 
                                   'INSTITUT', 'INEC', 'HTTP', 'WWW', 'ENCUESTA', 'La tabla']
                
                # Identificar columna de provincia/región
                col_provincia = None
                for col in df_datos.columns:
                    if 'provincia' in col.lower() or 'region' in col.lower():
                        col_provincia = col
                        break
                
                if col_provincia:
                    # Convertir a string primero y luego filtrar
                    df_datos = df_datos[df_datos[col_provincia].notna()].copy().reset_index(drop=True)
                    # Filtrar palabras excluidas de forma segura
                    patron = '|'.join(palabras_excluir)
                    mask = df_datos[col_provincia].astype(str).str.upper().str.contains(patron, na=False, case=False, regex=True)
                    df_datos = df_datos[~mask].copy().reset_index(drop=True)
                
                # Agregar información de producto y año
                producto = 'BANANO' if 'T13' in nombre_hoja.upper() or 'BANANO' in nombre_hoja.upper() else 'PLATANO' if 'T26' in nombre_hoja.upper() or 'PLAT' in nombre_hoja.upper() else 'DESCONOCIDO'
                df_datos['producto'] = producto
                
                if anio:
                    df_datos['anio'] = anio
                else:
                    # Intentar extraer año del nombre de la hoja
                    anio_hoja_match = re.search(r'(\d{4})', nombre_hoja)
                    if anio_hoja_match:
                        df_datos['anio'] = int(anio_hoja_match.group(1))
                
                print(f"       ✅ {len(df_datos)} registros procesados de hoja {nombre_hoja}")
                dfs_procesados.append(df_datos)
                
            except Exception as e_hoja:
                print(f"       ❌ Error en hoja {nombre_hoja}: {e_hoja}")
                continue
        
        xl_file.close()
        
        # Combinar todos los DataFrames procesados CON ESQUEMA UNIFICADO
        if dfs_procesados:
            # CORRECCIÓN #13: Mapear columnas POR POSICIÓN en lugar de por nombre
            # La estructura ESPAC es consistente: después del header, las columnas están siempre en el mismo orden
            df_unificados = []
            
            for df_hoja in dfs_procesados:
                n_rows = len(df_hoja)
                df_std = pd.DataFrame(index=range(n_rows))
                
                # CORRECCIÓN #14: Mapeo DIRECTO por nombres de columna conocidos
                # Después de normalizar, buscamos columnas por patrones específicos
                
                print(f"       📊 Columnas disponibles: {list(df_hoja.columns[:15])}")
                
                # Buscar columnas core
                col_provincia = next((c for c in df_hoja.columns if 'provincia' in c or 'region' in c), None)
                col_categoria = next((c for c in df_hoja.columns if 'categor' in c or 'tipo' in c or 'modalidad' in c), None)
                
                # CORRECCIÓN #15: Mapeo por sufijos (superficie_has y superficie_has_1)
                # La normalización crea columnas duplicadas con sufijos _1, _2, etc.
                # superficie_has → plantada, superficie_has_1 → cosechada
                
                # Buscar columnas de superficie (con o sin sufijos)
                superficie_cols = [c for c in df_hoja.columns if 'superficie' in c and 'has' in c]
                col_sup_plantada = superficie_cols[0] if len(superficie_cols) >= 1 else None
                col_sup_cosechada = superficie_cols[1] if len(superficie_cols) >= 2 else None
                
                # Buscar producción y ventas
                col_produccion = None
                col_ventas = None
                for col in df_hoja.columns:
                    col_lower = col.lower()
                    if not col_produccion and 'produccion' in col_lower and 'ventas' not in col_lower:
                        col_produccion = col
                    if not col_ventas and 'ventas' in col_lower:
                        col_ventas = col
                
                print(f"       🎯 Mapeo identificado:")
                print(f"          provincia: {col_provincia}")
                print(f"          categoria: {col_categoria}")
                print(f"          plantada: {col_sup_plantada}")
                print(f"          cosechada: {col_sup_cosechada}")
                print(f"          producción: {col_produccion}")
                print(f"          ventas: {col_ventas}")
                
                # Asignar columnas
                df_std['provincia'] = df_hoja[col_provincia].values if col_provincia else [''] * n_rows
                df_std['categoria'] = df_hoja[col_categoria].values if col_categoria else ['Solo'] * n_rows
                df_std['superficie_plantada_ha'] = df_hoja[col_sup_plantada].values if col_sup_plantada else [0] * n_rows
                df_std['superficie_cosechada_ha'] = df_hoja[col_sup_cosechada].values if col_sup_cosechada else [0] * n_rows
                df_std['produccion_tm'] = df_hoja[col_produccion].values if col_produccion else [0] * n_rows
                df_std['ventas_tm'] = df_hoja[col_ventas].values if col_ventas else [0] * n_rows
                
                df_std['producto'] = df_hoja['producto'].values if 'producto' in df_hoja.columns else ['DESCONOCIDO'] * n_rows
                df_std['anio'] = df_hoja['anio'].values if 'anio' in df_hoja.columns else [0] * n_rows
                
                # CORRECCIÓN #9B: Convertir numéricos con validación de rangos
                for col in ['superficie_plantada_ha', 'superficie_cosechada_ha', 'produccion_tm', 'ventas_tm']:
                    # Limpiar: quitar espacios, reemplazar comas por puntos
                    df_std[col] = df_std[col].astype(str).str.strip().str.replace(',', '.', regex=False)
                    # Quitar puntos solo si hay múltiples (separadores de miles)
                    df_std[col] = df_std[col].apply(lambda x: x.replace('.', '') if x.count('.') > 1 else x)
                    # Convertir a numérico
                    df_std[col] = pd.to_numeric(df_std[col], errors='coerce').fillna(0)
                    
                    # VALIDACIÓN: Valores razonables para Ecuador
                    # Superficie: max 500,000 ha por provincia
                    # Producción/Ventas: max 10,000,000 tm por provincia
                    if 'superficie' in col:
                        df_std[col] = df_std[col].apply(lambda x: 0 if x > 500000 else x)
                    else:
                        df_std[col] = df_std[col].apply(lambda x: 0 if x > 10000000 else x)
                
                # Calcular rendimiento
                df_std['rendimiento'] = np.where(
                    df_std['superficie_cosechada_ha'] > 0,
                    (df_std['produccion_tm'] / df_std['superficie_cosechada_ha']).round(2),
                    0
                )
                
                df_unificados.append(df_std)
            
            # Ahora sí, union seguro (todas las hojas tienen el MISMO esquema)
            df_final = pd.concat(df_unificados, ignore_index=True)
            
            # Filtrar filas vacías o inválidas
            df_final = df_final[df_final['provincia'].notna() & (df_final['provincia'].astype(str).str.strip() != '')]
            
            print(f"    ✅ Transformación completada: {len(df_final)} registros totales, {len(df_final.columns)} columnas")
            return df_final
        else:
            print(f"    ⚠️  No se procesaron hojas, retornando DataFrame vacío")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"    ❌ Error en transformación ESPAC: {e}")
        print(f"    🔄 Usando lectura estándar como fallback...")
        # Fallback: usar la función leer_archivo estándar
        return leer_archivo(local_path, nombre_archivo)

print("✅ Funciones de transformación especializadas cargadas (ESPAC Tabulados).")

In [ ]:
# Asignar la nueva versión corregida como la función predeterminada
transformar_espac_t13_t26_mejorado = transformar_espac_t13_t26_mejorado_v2

print("✅ Versión corregida activada: transformar_espac_t13_t26_mejorado ahora usa _v2")
print("   - Lee con pandas directamente (sin openpyxl)")
print("   - Valida rangos numéricos razonables")
print("   - Superficie < 500,000 ha")
print("   - Producción/Ventas < 10,000,000 tm")

In [ ]:
def transformar_espac_t13_t26_mejorado_v3(local_path: str, nombre_archivo: str) -> pd.DataFrame:
    """
    CORRECCIÓN #10 FINAL: Mapeo por ÍNDICE de columna fijo (no por nombre).
    Los archivos ESPAC tienen SIEMPRE la misma estructura de columnas:
      - Col 1: Provincia
      - Col 3: Superficie plantada
      - Col 4: Superficie cosechada
      - Col 5: Producción
      - Col 6: Ventas
    """
    import numpy as np
    
    print(f"  📊 Transformación ESPAC Tabulados/Series Históricas (v3 - mapeo por índice): {nombre_archivo}")
    
    # Extraer año del nombre del archivo
    anio_match = re.search(r'(\d{4})', nombre_archivo)
    anio = int(anio_match.group(1)) if anio_match else None
    print(f"    📅 Año extraído del archivo: {anio if anio else 'No detectado'}")
    
    try:
        # Leer con pandas directamente
        xl_file = pd.ExcelFile(local_path)
        nombres_hojas = xl_file.sheet_names
        
        print(f"    📂 Hojas disponibles: {len(nombres_hojas)} hojas")
        
        # Buscar hojas T13 o T26
        hojas_relevantes = []
        for nombre_hoja in nombres_hojas:
            nombre_upper = nombre_hoja.upper()
            if 'T13' in nombre_upper or 'T26' in nombre_upper:
                hojas_relevantes.append(nombre_hoja)
        
        print(f"    ✅ Hojas relevantes identificadas: {hojas_relevantes if hojas_relevantes else 'Ninguna'}")
        
        if not hojas_relevantes:
            hojas_relevantes = [xl_file.sheet_names[0]]
        
        dfs_procesados = []
        
        for nombre_hoja in hojas_relevantes:
            print(f"    📄 Procesando hoja: {nombre_hoja}")
            
            try:
                # Leer sin header (header=None)
                df_raw = pd.read_excel(local_path, sheet_name=nombre_hoja, header=None)
                
                if df_raw.empty:
                    print(f"       ⚠️  Hoja vacía, saltando...")
                    continue
                
                # Buscar fila de header
                header_idx = 0
                for i in range(min(20, len(df_raw))):
                    fila_text = ' '.join(str(v) for v in df_raw.iloc[i].values if pd.notna(v)).upper()
                    if any(kw in fila_text for kw in ['PROVINCIA', 'SUPERFICIE', 'PRODUCCIÓN', 'PRODUCCION']):
                        header_idx = i
                        print(f"       🎯 Header detectado en fila {header_idx}")
                        break
                
                # Datos empiezan después del header
                df_datos = df_raw.iloc[header_idx+1:].copy().reset_index(drop=True)
                
                # MAPEO POR ÍNDICE DE COLUMNA FIJO (estructura ESPAC estándar)
                # Estructura SIEMPRE es: [0]=índice, [1]=provincia, [2]=categoría?, [3]=sup_plant, [4]=sup_cosech, [5]=prod, [6]=ventas
                # Pero con merge cells, algunas pueden estar en posiciones ligeramente diferentes
                
                # Buscar columna de provincia (primera columna con texto largo)
                col_provincia_idx = None
                for i in range(min(5, df_datos.shape[1])):
                    muestra = df_datos.iloc[:5, i].astype(str)
                    if muestra.str.len().mean() > 5:  # Provincias son texto largo
                        col_provincia_idx = i
                        break
                
                if col_provincia_idx is None:
                    print(f"       ⚠️  No se encontró columna de provincia, saltando...")
                    continue
                
                print(f"       📍 Provincia en col[{col_provincia_idx}]")
                
                # Las columnas numéricas están DESPUÉS de provincia
                # Normalmente: provincia(col 1), luego 4 columnas numéricas
                num_cols_start = col_provincia_idx + 1
                
                # Filtrar filas válidas (que tengan provincia)
                df_datos = df_datos[df_datos.iloc[:, col_provincia_idx].notna()].copy()
                
                # Filtrar palabras excluidas
                palabras_excluir = ['TOTAL', 'REGIÓN', 'REGION', 'ZONA', 'NOTA', 'FUENTE', 'OBSERVACIÓN']
                patron = '|'.join(palabras_excluir)
                mask = df_datos.iloc[:, col_provincia_idx].astype(str).str.upper().str.contains(patron, na=False, case=False, regex=True)
                df_datos = df_datos[~mask].copy().reset_index(drop=True)
                
                # Crear DataFrame estandarizado
                n_rows = len(df_datos)
                df_std = pd.DataFrame(index=range(n_rows))
                
                df_std['provincia'] = df_datos.iloc[:, col_provincia_idx].values
                df_std['categoria'] = 'Solo'  # No usamos categoría por ahora
                
                # MAPEO NUMÉRICO POR POSICIÓN RELATIVA
                # Después de provincia, las siguientes 4-5 columnas son numéricas
                # Identificar cuáles columnas son numéricas
                cols_numericas = []
                for i in range(num_cols_start, min(num_cols_start + 6, df_datos.shape[1])):
                    muestra = df_datos.iloc[:10, i].astype(str).str.replace(',', '.', regex=False)
                    try:
                        pd.to_numeric(muestra, errors='coerce')
                        es_numerica = muestra.str.match(r'^\d').sum() >= 3  # Al menos 3 valores parecen números
                        if es_numerica:
                            cols_numericas.append(i)
                    except:
                        pass
                
                print(f"       🔢 Columnas numéricas identificadas: {cols_numericas}")
                
                # Asignar en orden: plantada, cosechada, producción, ventas
                if len(cols_numericas) >= 4:
                    df_std['superficie_plantada_ha'] = df_datos.iloc[:, cols_numericas[0]].values
                    df_std['superficie_cosechada_ha'] = df_datos.iloc[:, cols_numericas[1]].values
                    df_std['produccion_tm'] = df_datos.iloc[:, cols_numericas[2]].values
                    df_std['ventas_tm'] = df_datos.iloc[:, cols_numericas[3]].values
                elif len(cols_numericas) >= 3:
                    df_std['superficie_plantada_ha'] = df_datos.iloc[:, cols_numericas[0]].values
                    df_std['superficie_cosechada_ha'] = 0
                    df_std['produccion_tm'] = df_datos.iloc[:, cols_numericas[1]].values
                    df_std['ventas_tm'] = df_datos.iloc[:, cols_numericas[2]].values
                else:
                    print(f"       ⚠️  Insuficientes columnas numéricas ({len(cols_numericas)}), usando ceros")
                    df_std['superficie_plantada_ha'] = 0
                    df_std['superficie_cosechada_ha'] = 0
                    df_std['produccion_tm'] = 0
                    df_std['ventas_tm'] = 0
                
                # Convertir numéricos con validación
                for col in ['superficie_plantada_ha', 'superficie_cosechada_ha', 'produccion_tm', 'ventas_tm']:
                    df_std[col] = df_std[col].astype(str).str.strip().str.replace(',', '.', regex=False)
                    df_std[col] = df_std[col].apply(lambda x: x.replace('.', '') if isinstance(x, str) and x.count('.') > 1 else x)
                    df_std[col] = pd.to_numeric(df_std[col], errors='coerce').fillna(0)
                    
                    # Validación de rangos
                    if 'superficie' in col:
                        df_std[col] = df_std[col].apply(lambda x: 0 if x > 500000 else x)
                    else:
                        df_std[col] = df_std[col].apply(lambda x: 0 if x > 10000000 else x)
                
                # Producto y año
                producto = 'BANANO' if 'T13' in nombre_hoja.upper() else 'PLATANO' if 'T26' in nombre_hoja.upper() else 'DESCONOCIDO'
                df_std['producto'] = producto
                df_std['anio'] = anio if anio else 0
                
                # Calcular rendimiento
                df_std['rendimiento'] = np.where(
                    df_std['superficie_cosechada_ha'] > 0,
                    (df_std['produccion_tm'] / df_std['superficie_cosechada_ha']).round(2),
                    0
                )
                
                print(f"       ✅ {len(df_std)} registros procesados de hoja {nombre_hoja}")
                dfs_procesados.append(df_std)
                
            except Exception as e_hoja:
                print(f"       ❌ Error en hoja {nombre_hoja}: {e_hoja}")
                continue
        
        xl_file.close()
        
        if dfs_procesados:
            df_final = pd.concat(dfs_procesados, ignore_index=True)
            df_final = df_final[df_final['provincia'].notna() & (df_final['provincia'].astype(str).str.strip() != '')]
            
            # 🆕 MAPEO DE PROVINCIA A ID (usando tabla dimensional)
            df_final = mapear_provincia_a_id(df_final, 'provincia')
            
            # ELIMINAR registros con provincia_id NULL (agregaciones regionales/provincia 0)
            # 🚫 IMPORTANTE: NO aplicar KNN a provincia_id - los NULL son agregaciones que deben eliminarse
            antes_filtro = len(df_final)
            df_final = df_final.dropna(subset=['provincia_id'])
            despues_filtro = len(df_final)
            eliminados = antes_filtro - despues_filtro
            if eliminados > 0:
                print(f"    🗑️  {eliminados} registros eliminados (agregaciones regionales/provincia 0)")
            
            # 🚫 KNN DESHABILITADO para ESPAC - provincia_id NULL debe eliminarse, no imputarse
            print(f"    ℹ️  KNN deshabilitado para ESPAC (provincia_id NULL = agregaciones, no errores)")
            
            print(f"    ✅ Transformación completada: {len(df_final)} registros totales, {len(df_final.columns)} columnas")
            return df_final
        else:
            print(f"    ⚠️  No se procesaron hojas, retornando DataFrame vacío")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"    ❌ Error en transformación ESPAC: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()

# Activar la versión v3
transformar_espac_t13_t26_mejorado = transformar_espac_t13_t26_mejorado_v3

print("✅ CORRECCIÓN #10 APLICADA: Mapeo por índice de columna (v3)")
print("   - Identifica columna de provincia por contenido de texto")
print("   - Identifica columnas numéricas por patrón de dígitos")
print("   - Asigna en orden: plantada, cosechada, producción, ventas")

In [ ]:
def transformar_sipa_temperatura(df_pandas: pd.DataFrame, nombre_archivo: str) -> pd.DataFrame:
    """
    Transformación especializada para archivos SIPA de temperatura/precipitación.
    Aplica mapeo de provincia a ID.
    """
    print(f"  🌡️  Transformación SIPA Temperatura/Precipitación: {nombre_archivo}")
    
    try:
        df_clean = df_pandas.copy()
        
        # Normalizar nombres de columnas
        df_clean.columns = [normalizar_columna(str(c)) for c in df_clean.columns]
        
        print(f"    📊 Columnas detectadas: {list(df_clean.columns)[:10]}")
        
        # Mapear columnas comunes de SIPA
        columnas_mapeo = {
            'ano': 'anio',
            'mes': 'mes',
            'estacion': 'estacion',
            'provincia': 'provincia',
            'canton': 'canton',
            'precipitacion_mm': 'precipitacion_mm',
            'temperatura_promedio_c': 'temperatura_promedio_c'
        }
        
        # Renombrar columnas si existen
        for old_col, new_col in columnas_mapeo.items():
            if old_col in df_clean.columns:
                df_clean = df_clean.rename(columns={old_col: new_col})
        
        # Convertir columnas numéricas - IMPORTANTE: preservar tipo INT
        if 'anio' in df_clean.columns:
            # Convertir a float primero, luego a int para manejar decimales (2000.0 → 2000)
            df_clean['anio'] = pd.to_numeric(df_clean['anio'], errors='coerce')
            # Redondear antes de convertir a int para evitar errores
            df_clean['anio'] = df_clean['anio'].round(0).fillna(0).astype(int)
            # Reemplazar 0 con NaN y eliminar esas filas
            df_clean['anio'] = df_clean['anio'].replace(0, pd.NA)
            # Asegurar que se mantenga como Int64 (nullable int)
            df_clean['anio'] = df_clean['anio'].astype('Int64')
        
        if 'mes' in df_clean.columns:
            # Convertir a int, manejando valores vacíos
            df_clean['mes'] = pd.to_numeric(df_clean['mes'], errors='coerce')
            df_clean['mes'] = df_clean['mes'].fillna(0).astype(int)
            # Si todos son 0, intentar extraer mes de otra columna si existe
            if (df_clean['mes'] == 0).all() and 'anio' in df_clean.columns:
                # Si no hay mes, agregar columna con 0 para indicar dato anual
                print(f"    ℹ️  Columna 'mes' vacía, usando 0 (dato anual)")
        elif 'anio' in df_clean.columns:
            # Si no existe columna mes, crearla con 0
            df_clean['mes'] = 0
            print(f"    ℹ️  Columna 'mes' no existe, creada con 0 (dato anual)")
        
        if 'precipitacion_mm' in df_clean.columns:
            df_clean['precipitacion_mm'] = pd.to_numeric(df_clean['precipitacion_mm'], errors='coerce')
        
        if 'temperatura_promedio_c' in df_clean.columns:
            df_clean['temperatura_promedio_c'] = pd.to_numeric(df_clean['temperatura_promedio_c'], errors='coerce')
        
        # Filtrar filas con valores nulos en campos críticos
        if 'anio' in df_clean.columns:
            antes = len(df_clean)
            df_clean = df_clean.dropna(subset=['anio'])
            despues = len(df_clean)
            if antes > despues:
                print(f"    🧹 Filtradas {antes - despues} filas con anio nulo")
        
        # 🎯 MAPEO DE PROVINCIA A ID (usando tabla dimensional)
        if 'provincia' in df_clean.columns:
            df_clean = mapear_provincia_a_id(df_clean, 'provincia')
            
            # ELIMINAR registros con provincia_id NULL (agregaciones regionales/provincia 0)
            antes_filtro = len(df_clean)
            df_clean = df_clean.dropna(subset=['provincia_id'])
            despues_filtro = len(df_clean)
            eliminados = antes_filtro - despues_filtro
            if eliminados > 0:
                print(f"    🗑️  {eliminados} registros eliminados (agregaciones regionales/provincia 0)")
                print(f"    ✅ {despues_filtro} registros con provincia válida")
        
        # ✅ APLICAR KNN A COLUMNAS NUMÉRICAS EN SIPA
        print(f"    🔧 Aplicando KNN a columnas numéricas con NULL...")
        
        # Identificar columnas numéricas con NULL (excluir IDs)
        columnas_numericas_con_null = []
        for col in df_clean.columns:
            if col in ['provincia_id', 'anio', 'mes']:  # No aplicar KNN a IDs y fechas
                continue
            try:
                col_numeric = pd.to_numeric(df_clean[col], errors='coerce')
                nulls = col_numeric.isna().sum()
                if nulls > 0 and nulls < len(df_clean):  # Tiene NULL pero no todos
                    columnas_numericas_con_null.append(col)
                    print(f"       {col}: {nulls} NULL ({nulls/len(df_clean)*100:.1f}%)")
            except:
                pass
        
        if columnas_numericas_con_null and len(df_clean) >= 5:
            try:
                from sklearn.impute import KNNImputer
                
                # Preparar datos para KNN
                features_knn = columnas_numericas_con_null.copy()
                if 'anio' in df_clean.columns:
                    features_knn = ['anio'] + features_knn
                if 'mes' in df_clean.columns and df_clean['mes'].nunique() > 1:
                    features_knn.insert(1, 'mes')
                
                # Convertir a numérico
                df_knn = df_clean[features_knn].copy()
                for col in df_knn.columns:
                    df_knn[col] = pd.to_numeric(df_knn[col], errors='coerce')
                
                # Aplicar KNN
                imputer = KNNImputer(n_neighbors=min(5, len(df_clean)), weights='distance')
                df_imputed = imputer.fit_transform(df_knn)
                
                # Reemplazar valores en columnas originales (sin año/mes)
                start_idx = 0
                if 'anio' in features_knn:
                    start_idx += 1
                if 'mes' in features_knn:
                    start_idx += 1
                
                for i, col in enumerate(columnas_numericas_con_null):
                    df_clean[col] = df_imputed[:, start_idx + i]
                
                print(f"    ✅ KNN aplicado a {len(columnas_numericas_con_null)} columnas")
            except Exception as e:
                print(f"    ⚠️  No se pudo aplicar KNN: {e}")
        else:
            print(f"    ℹ️  KNN omitido (no hay suficientes datos o columnas)")
        
        # Eliminar filas completamente vacías
        df_clean = df_clean.dropna(how='all').reset_index(drop=True)
        # Eliminar columna 'mes': dato auxiliar interno, no se exporta
        df_clean = df_clean.drop(columns=['mes'], errors='ignore')
        
        print(f"    ✅ Transformación completada: {len(df_clean)} registros, {len(df_clean.columns)} columnas")
        print(f"    📊 Columnas finales: {list(df_clean.columns)}")
        
        return df_clean
        
    except Exception as e:
        print(f"    ⚠️  Error en transformación SIPA: {e}")
        import traceback
        traceback.print_exc()
        return df_pandas

print("✅ Función de transformación SIPA Temperatura cargada (con mapeo de provincia_id y KNN)")

In [ ]:
def transformar_uso_del_suelo(df_pandas: pd.DataFrame, nombre_archivo: str) -> pd.DataFrame:
    """
    Transformación especializada para archivos de Uso del Suelo.
    La columna 'region_y_provincia' puede contener nombres de provincias O regiones.
    Solo mapeamos las provincias a provincia_id. Las regiones quedan sin mapear (NULL).
    """
    print(f"  🌾 Transformación Uso del Suelo: {nombre_archivo}")
    
    try:
        df_clean = df_pandas.copy()
        
        # Normalizar nombres de columnas
        df_clean.columns = [normalizar_columna(str(c)) for c in df_clean.columns]
        
        print(f"    📊 Columnas detectadas: {list(df_clean.columns)[:10]}")
        
        # Mapear columnas comunes
        columnas_mapeo = {
            'ano': 'anio',
            'region_y_provincia': 'region_y_provincia',
            'categoria_de_uso_del_suelo': 'categoria_de_uso_del_suelo',
            'superficie_ha': 'superficie_ha'
        }
        
        # Renombrar columnas si existen
        for old_col, new_col in columnas_mapeo.items():
            if old_col in df_clean.columns:
                df_clean = df_clean.rename(columns={old_col: new_col})
        
        # Convertir columnas numéricas
        if 'anio' in df_clean.columns:
            df_clean['anio'] = pd.to_numeric(df_clean['anio'], errors='coerce').astype('Int64')
        
        if 'superficie_ha' in df_clean.columns:
            df_clean['superficie_ha'] = pd.to_numeric(df_clean['superficie_ha'], errors='coerce')
        
        # 🎯 SEPARAR region_y_provincia EN SOLO provincia
        # La columna puede tener nombres de provincia directos ("Azuay", "Guayas")
        # o nombres de región ("Centro-Suroriente", "Costa", etc.)
        # Solo mapeamos provincias, las regiones quedan como NULL y se ELIMINAN
        if 'region_y_provincia' in df_clean.columns:
            # Renombrar temporalmente para el mapeo
            df_clean = df_clean.rename(columns={'region_y_provincia': 'provincia'})
            
            print(f"    🗺️  Intentando mapear valores de región_y_provincia a provincia_id...")
            
            # Aplicar mapeo - los valores que NO son provincias quedarán NULL
            df_clean = mapear_provincia_a_id(df_clean, 'provincia')
            
            # ELIMINAR registros con provincia_id NULL (agregaciones regionales/provincia 0)
            antes_filtro = len(df_clean)
            df_clean = df_clean.dropna(subset=['provincia_id'])
            despues_filtro = len(df_clean)
            eliminados = antes_filtro - despues_filtro
            if eliminados > 0:
                print(f"    🗑️  {eliminados} registros eliminados (agregaciones regionales/provincia 0)")
                print(f"    ✅ {despues_filtro} registros con provincia válida")
        
        # Filtrar filas con valores nulos en campos críticos
        if 'anio' in df_clean.columns:
            antes = len(df_clean)
            df_clean = df_clean.dropna(subset=['anio'])
            despues = len(df_clean)
            if antes > despues:
                print(f"    🧹 Filtradas {antes - despues} filas con anio nulo")
        
        # ✅ APLICAR KNN A COLUMNAS NUMÉRICAS EN USO DEL SUELO
        print(f"    🔧 Aplicando KNN a columnas numéricas con NULL...")
        
        # Identificar columnas numéricas con NULL (excluir IDs)
        columnas_numericas_con_null = []
        for col in df_clean.columns:
            if col in ['provincia_id', 'anio']:  # No aplicar KNN a IDs y fechas
                continue
            try:
                col_numeric = pd.to_numeric(df_clean[col], errors='coerce')
                nulls = col_numeric.isna().sum()
                if nulls > 0 and nulls < len(df_clean):  # Tiene NULL pero no todos
                    columnas_numericas_con_null.append(col)
                    print(f"       {col}: {nulls} NULL ({nulls/len(df_clean)*100:.1f}%)")
            except:
                pass
        
        if columnas_numericas_con_null and len(df_clean) >= 5:
            try:
                from sklearn.impute import KNNImputer
                
                # Preparar datos para KNN
                features_knn = columnas_numericas_con_null.copy()
                if 'anio' in df_clean.columns:
                    features_knn = ['anio'] + features_knn
                if 'provincia_id' in df_clean.columns:
                    features_knn.insert(0 if 'anio' not in features_knn else 1, 'provincia_id')
                
                # Convertir a numérico
                df_knn = df_clean[features_knn].copy()
                for col in df_knn.columns:
                    df_knn[col] = pd.to_numeric(df_knn[col], errors='coerce')
                
                # Aplicar KNN
                imputer = KNNImputer(n_neighbors=min(5, len(df_clean)), weights='distance')
                df_imputed = imputer.fit_transform(df_knn)
                
                # Reemplazar valores en columnas originales (sin provincia_id/año)
                start_idx = 0
                if 'provincia_id' in features_knn:
                    start_idx += 1
                if 'anio' in features_knn:
                    start_idx += 1
                
                for i, col in enumerate(columnas_numericas_con_null):
                    df_clean[col] = df_imputed[:, start_idx + i]
                
                print(f"    ✅ KNN aplicado a {len(columnas_numericas_con_null)} columnas")
            except Exception as e:
                print(f"    ⚠️  No se pudo aplicar KNN: {e}")
        else:
            print(f"    ℹ️  KNN omitido (no hay suficientes datos o columnas)")
        
        # Eliminar filas completamente vacías
        df_clean = df_clean.dropna(how='all').reset_index(drop=True)
        
        print(f"    ✅ Transformación completada: {len(df_clean)} registros, {len(df_clean.columns)} columnas")
        print(f"    📊 Columnas finales: {list(df_clean.columns)}")
        
        return df_clean
        
    except Exception as e:
        print(f"    ⚠️  Error en transformación Uso del Suelo: {e}")
        import traceback
        traceback.print_exc()
        return df_pandas

print("✅ Función de transformación Uso del Suelo cargada (con mapeo de provincia_id y KNN)")

In [ ]:
def transformar_faostat(df_pandas: pd.DataFrame, nombre_archivo: str) -> pd.DataFrame:
    """
    Transformación especializada para archivos FAOSTAT (producción banan/plátano).
    Normaliza columnas y aplica mapeo de nombres.
    """
    print(f"  🌍 Transformación FAOSTAT: {nombre_archivo}")
    
    try:
        df_clean = df_pandas.copy()
        
        # Normalizar nombres de columnas
        df_clean.columns = [normalizar_columna(str(c)) for c in df_clean.columns]
        
        print(f"    📊 Columnas detectadas: {list(df_clean.columns)[:10]}")
        
        # Mapear columnas comunes de FAOSTAT
        columnas_mapeo = {
            'area_code': 'pais_codigo',
            'area': 'pais',
            'item_code': 'producto_codigo',
            'item': 'producto',
            'element_code': 'elemento_codigo',
            'element': 'elemento',
            'year': 'anio',
            'value': 'valor',
            'unit': 'unidad'
        }
        
        df_clean = df_clean.rename(columns=columnas_mapeo)
        
        # Convertir columnas numéricas
        for col in ['pais_codigo', 'producto_codigo', 'elemento_codigo', 'anio']:
            if col in df_clean.columns:
                df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        
        if 'valor' in df_clean.columns:
            df_clean['valor'] = pd.to_numeric(df_clean['valor'], errors='coerce')
        
        # Filtrar filas sin datos válidos
        registros_antes = len(df_clean)
        df_clean = df_clean.dropna(subset=['anio', 'valor'], how='any')
        registros_eliminados = registros_antes - len(df_clean)
        
        if registros_eliminados > 0:
            print(f"    🗑️  Filtrados {registros_eliminados} registros sin año o valor")
        
        print(f"    ✅ Transformación completada: {len(df_clean):,} registros, {len(df_clean.columns)} columnas")
        return df_clean
        
    except Exception as e:
        print(f"    ⚠️  Error en transformación FAOSTAT: {e}")
        print(f"    🔄 Usando estructura original...")
        return df_pandas

print("✅ Función transformar_faostat() creada")

In [ ]:
def transformar_aebe_bananotas(local_path: str, nombre_archivo: str) -> pd.DataFrame:
    """
    Transforma PDFs de AEBE Bananotas extrayendo rankings de:
    - Exportadores (Ubesa, Reybanpac, etc.)
    - Marcas (Dole, Chiquita, etc.)
    - Navieras (MSC, Maersk, etc.)
    - Puertos (San Petersburgo, Rotterdam, etc.)
    - Mercados/países (Unión Europea, Medio Oriente, etc.)

    MOTOR: pdfplumber (puro Python, sin JVM) en lugar de tabula-py.
    tabula-py lanza un subproceso JVM por cada llamada a read_pdf(), y el
    código anterior lo invocaba ~20 veces por PDF (5 tipos x ~4 páginas),
    lo que multiplicaba el overhead de arranque de JVM y explicaba los
    tiempos de horas. pdfplumber abre el PDF una sola vez y recorre todas
    las páginas en una sola pasada en memoria.

    DETECCIÓN DINÁMICA: no se asume un rango fijo de páginas (ej. 65-76,
    que puede variar de edición a edición). En su lugar, se inspecciona el
    texto de CADA página del PDF buscando tablas cuya fila de cabecera
    contenga dos columnas de año consecutivas (ej. "2025" y "2026") como
    valores aislados — patrón característico de las tablas de ranking de
    Bananotas. El tipo (exportador/marca/naviera/puerto/mercado) se infiere
    por palabras clave en el título inmediatamente anterior a la tabla, con
    fallback a la propia fila de cabecera.

    NORMALIZACIÓN A UN SOLO AÑO: cada tabla de ranking trae dos columnas de
    cantidad (año N y año N+1, ej. 2025 y 2026). Se conserva únicamente la
    columna del año MÁS RECIENTE de cada tabla — no se asume que todas las
    tablas del PDF compartan el mismo par de años, por lo que el año
    "más reciente" se calcula tabla por tabla, y el año de la revista
    completa es el máximo visto en todas las tablas.

    Estructura normalizada (UN registro por entidad del ranking):
    tipo | dato | cantidad | medida | anio | framework | fuente | archivo_origen

    Args:
        local_path: Ruta local al PDF descargado
        nombre_archivo: Nombre del archivo

    Returns:
        DataFrame consolidado con todos los rankings del año más reciente
        detectado en cada tabla.
    """
    print(f"  📊 Transformación AEBE Bananotas (pdfplumber): {nombre_archivo}")

    try:
        import pdfplumber
    except ImportError:
        print("  ⚠️  Instalando pdfplumber...")
        import subprocess
        subprocess.run(["pip", "install", "-q", "pdfplumber"], check=True)
        import pdfplumber

    COLUMNAS_VACIAS = ['tipo', 'dato', 'cantidad', 'medida', 'anio', 'framework', 'fuente', 'archivo_origen']

    RANKING_TIPOS = {
        'exportador': ['EXPORTADOR', 'EXPORTADORES'],
        'marca':      ['MARCA', 'MARCAS'],
        'naviera':    ['NAVIERA', 'NAVIERAS'],
        'puerto':     ['PUERTO', 'PUERTOS', 'DESTINO'],
        'mercado':    ['PAIS', 'PAÍS', 'MERCADO', 'MERCADOS'],
    }

    def _detectar_anios_header(tabla_filas):
        """Fila de cabecera real = la que contiene >=2 años de 4 dígitos.
        Busca años dentro de las celdas (no solo celdas que sean exactamente un año),
        para manejar headers con formato 'R MARCAS 2025 2026\n2025 2026...'."""
        for fila in tabla_filas:
            celdas = [str(c).strip() if c else "" for c in fila]
            # Buscar TODOS los años en TODAS las celdas (incluso si hay otros textos)
            anios_encontrados = []
            for celda in celdas:
                # Encuentra todos los años de 4 dígitos en esta celda
                matches = re.findall(r'\b20[12][0-9]\b', celda)
                anios_encontrados.extend(matches)
            
            # Si encontramos al menos 2 años distintos, es el header
            anios_unicos = sorted(set(int(a) for a in anios_encontrados))
            if len(anios_unicos) >= 2:
                return anios_unicos, fila
        return None, None

    def _clasificar_tipo(texto):
        texto_up = texto.upper()
        for tipo, keywords in RANKING_TIPOS.items():
            if any(kw in texto_up for kw in keywords):
                return tipo
        return None

    def _limpiar_numero(val):
        if val is None:
            return None
        s = str(val).strip()
        if s in ("", "-", "—", "–"):
            return None
        s = s.replace("%", "")
        s = re.sub(r"\s+", "", s)
        if "," in s and "." in s:
            s = s.replace(",", "")
        else:
            s = s.replace(",", ".")
        try:
            return float(s)
        except ValueError:
            return None

    registros = []
    anios_vistos = []

    try:
        with pdfplumber.open(local_path) as pdf:
            print(f"    🔍 Escaneando {len(pdf.pages)} páginas (detección dinámica)...")
            for page_num, page in enumerate(pdf.pages, start=1):
                # 1) intento con tablas de bordes reales (lattice); si no hay,
                #    fallback a deteccion por alineacion de texto (streams)
                tables = page.find_tables(table_settings={
                    "vertical_strategy": "lines", "horizontal_strategy": "lines"
                })
                if not tables:
                    tables = page.find_tables(table_settings={
                        "vertical_strategy": "text", "horizontal_strategy": "text"
                    })
                if not tables:
                    continue

                for tabla_obj in tables:
                    filas = tabla_obj.extract()
                    if not filas or len(filas) < 3:
                        continue

                    anios, header_fila = _detectar_anios_header(filas)
                    if not anios or header_fila is None:
                        continue  # no es tabla de ranking (sin 2 cols de año)

                    anio_reciente = max(anios)

                    # Título: texto de la página por encima del bbox de la tabla
                    top_tabla = tabla_obj.bbox[1]
                    texto_arriba = page.crop((0, 0, page.width, top_tabla)).extract_text() or ""
                    lineas_arriba = [l for l in texto_arriba.split("\n") if l.strip()]
                    titulo_candidato = " ".join(lineas_arriba[-3:]) if lineas_arriba else ""

                    tipo = _clasificar_tipo(titulo_candidato)
                    if tipo is None:
                        tipo = _clasificar_tipo(" ".join(str(c) for c in header_fila if c))
                    if tipo is None:
                        continue

                    anios_vistos.append(anio_reciente)
                    n_antes = len(registros)

                    # CASO ESPECIAL: Tabla extraída como UNA SOLA CELDA
                    # (común en PDFs sin bordes de tabla claros)
                    if len(header_fila) == 1 and len(header_fila[0]) > 50:
                        # Parseo manual de filas con patrón: ranking + nombre + valores
                        for fila in filas:
                            if not fila or not fila[0]:
                                continue
                            fila_texto = str(fila[0]).strip()
                            
                            # Patrón: dígitos (ranking) + texto (nombre) + 2+ números
                            match = re.match(r'^(\d{1,2})\s+(.+?)\s+([\d.]+)\s+([\d.]+)', fila_texto)
                            if not match:
                                continue
                            
                            ranking_txt = match.group(1)
                            nombre = match.group(2).strip()
                            
                            # Saltar filas de totales/otros
                            if nombre.upper() in ("TOTAL", "TOTAL GENERAL", "OTROS"):
                                continue
                            
                            # Los dos primeros valores numéricos son para los 2 años
                            valor_anio1 = _limpiar_numero(match.group(3))
                            valor_anio2 = _limpiar_numero(match.group(4))
                            
                            # Tomar el valor del año más reciente
                            # Si hay 2 años [2025, 2026], el más reciente es 2026 (segundo)
                            cantidad = valor_anio2 if len(anios) >= 2 and anio_reciente == anios[-1] else valor_anio1
                            
                            if cantidad is None or cantidad <= 0:
                                continue
                            
                            registros.append({
                                'tipo': tipo,
                                'dato': nombre,
                                'cantidad': cantidad,
                                'medida': 'cajas (millones) de 18.14kg',
                                'anio': anio_reciente,
                                'framework': 'llamaindex',
                                'fuente': 'AEBE_BANANOTAS',
                                'archivo_origen': nombre_archivo,
                            })
                    
                    else:
                        # CASO NORMAL: Tabla con columnas separadas
                        col_idx = None
                        for i, cell in enumerate(header_fila):
                            if cell and str(anio_reciente) in str(cell):
                                col_idx = i
                                break
                        if col_idx is None:
                            continue

                        for fila in filas:
                            if not fila or fila[0] is None:
                                continue
                            rank_val = str(fila[0]).strip()
                            if not re.match(r"^\d{1,2}$", rank_val):
                                continue

                            nombre = str(fila[1]).strip() if len(fila) > 1 and fila[1] else None
                            if not nombre or nombre.upper() in ("TOTAL", "TOTAL GENERAL", "OTROS"):
                                continue

                            if col_idx >= len(fila):
                                continue
                            cantidad = _limpiar_numero(fila[col_idx])
                            if cantidad is None or cantidad <= 0:
                                continue

                            registros.append({
                                'tipo': tipo,
                                'dato': nombre,
                                'cantidad': cantidad,
                                'medida': 'cajas (millones) de 18.14kg',
                                'anio': anio_reciente,
                                'framework': 'llamaindex',
                                'fuente': 'AEBE_BANANOTAS',
                                'archivo_origen': nombre_archivo,
                            })

                    n_nuevos = len(registros) - n_antes
                    if n_nuevos > 0:
                        print(f"       ✅ p.{page_num} [{tipo}] {n_nuevos} registros (año {anio_reciente})")

        if not registros:
            print(f"    ⚠️  No se encontraron rankings en el PDF - devolviendo DataFrame vacío")
            return pd.DataFrame(columns=COLUMNAS_VACIAS)

        anio_revista = max(anios_vistos)
        df_final = pd.DataFrame(registros)
        print(f"    ✅ Transformación completada: {len(df_final)} registros | año revista: {anio_revista} | tipos: {sorted(df_final['tipo'].unique())}")
        return df_final

    except Exception as e:
        print(f"    ❌ Error en transformación: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame(columns=COLUMNAS_VACIAS)


print("✅ Función transformar_aebe_bananotas (pdfplumber) cargada.")


In [ ]:
# Versión 2: Parseo basado en texto (sin detección de tablas)
def transformar_aebe_bananotas_v2(local_path: str, nombre_archivo: str) -> pd.DataFrame:
    """Versión simplificada que parsea texto directamente sin depender de find_tables."""
    print(f"  📊 Transformación AEBE Bananotas v2 (texto): {nombre_archivo}")
    
    try:
        import pdfplumber
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "pdfplumber"], check=True)
        import pdfplumber
    
    COLUMNAS_VACIAS = ['tipo', 'dato', 'cantidad', 'medida', 'anio', 'framework', 'fuente', 'archivo_origen']
    
    RANKING_KEYWORDS = {
        'exportador': ['EXPORTADOR'],
        'marca': ['MARCA'],
        'naviera': ['NAVIERA'],
        'puerto': ['PUERTO'],
        'mercado': ['PAIS', 'PAÍS', 'MERCADO'],
    }
    
    def _limpiar_numero(val):
        if not val:
            return None
        s = str(val).strip().replace("%", "").replace(",", ".")
        s = re.sub(r"\s+", "", s)
        try:
            return float(s)
        except:
            return None
    
    registros = []
    anio_revista = None
    
    try:
        with pdfplumber.open(local_path) as pdf:
            print(f"    🔍 Escaneando {len(pdf.pages)} páginas...")
            
            for page_num, page in enumerate(pdf.pages, start=1):
                texto = page.extract_text()
                if not texto:
                    continue
                
                lineas = texto.split('\n')
                
                # Buscar líneas con header (2 años consecutivos)
                for idx, linea in enumerate(lineas):
                    # Detectar header con años
                    anios = re.findall(r'\b20[12][0-9]\b', linea)
                    if len(set(anios)) < 2:
                        continue
                    
                    anios_unicos = sorted(set(int(a) for a in anios))
                    anio_reciente = max(anios_unicos)
                    
                    if anio_revista is None or anio_reciente > anio_revista:
                        anio_revista = anio_reciente
                    
                    # Determinar tipo de ranking
                    tipo = None
                    contexto = ' '.join(lineas[max(0, idx-3):idx+1]).upper()
                    for t, keywords in RANKING_KEYWORDS.items():
                        if any(kw in contexto for kw in keywords):
                            tipo = t
                            break
                    
                    if not tipo:
                        continue
                    
                    # Parsear filas de datos después del header
                    n_antes = len(registros)
                    for linea_datos in lineas[idx+1:idx+25]:  # Máximo 25 líneas después
                        # Patrón: ranking + nombre + números
                        match = re.match(r'^(\d{1,2})\s+(.+?)\s+([\d.,]+)\s+([\d.,]+)', linea_datos)
                        if not match:
                            continue
                        
                        ranking = match.group(1)
                        nombre = match.group(2).strip()
                        
                        if nombre.upper() in ['TOTAL', 'OTROS', 'TOTAL GENERAL']:
                            continue
                        
                        valor1 = _limpiar_numero(match.group(3))
                        valor2 = _limpiar_numero(match.group(4))
                        
                        # Segundo valor = año más reciente
                        cantidad = valor2 if valor2 else valor1
                        
                        if not cantidad or cantidad <= 0:
                            continue
                        
                        registros.append({
                            'tipo': tipo,
                            'dato': nombre,
                            'cantidad': cantidad,
                            'medida': 'cajas (millones) de 18.14kg',
                            'anio': anio_reciente,
                            'framework': 'llamaindex',
                            'fuente': 'AEBE_BANANOTAS',
                            'archivo_origen': nombre_archivo,
                        })
                    
                    if len(registros) > n_antes:
                        print(f"       ✅ p.{page_num} [{tipo}] {len(registros)-n_antes} registros (año {anio_reciente})")
                        break  # Solo procesar el primer ranking por página
        
        if not registros:
            print(f"    ⚠️  No se encontraron rankings")
            return pd.DataFrame(columns=COLUMNAS_VACIAS)
        
        df_final = pd.DataFrame(registros)
        print(f"    ✅ {len(df_final)} registros | año: {anio_revista} | tipos: {sorted(df_final['tipo'].unique())}")
        return df_final
    
    except Exception as e:
        print(f"    ❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame(columns=COLUMNAS_VACIAS)

print("✅ transformar_aebe_bananotas_v2 cargada")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# FUNCIONES AUXILIARES PARA AEBE (limpieza de números y porcentajes)
# ══════════════════════════════════════════════════════════════════════

def limpiar_numero(valor) -> float:
    """
    Extrae un número flotante de un string, manejando:
    - Separadores de miles (comas, puntos)
    - Decimales
    - Valores NULL/vacíos
    
    Ejemplos:
        "17,084.5" → 17084.5
        "17.084,5" → 17084.5
        "25.79%" → 25.79
        "" → None
    """
    if pd.isna(valor) or valor == '' or valor is None:
        return None
    
    try:
        # Convertir a string y limpiar
        s = str(valor).strip()
        
        # Eliminar símbolos no numéricos (excepto . , - +)
        s = re.sub(r'[^\d.,-]', '', s)
        
        if not s or s in ['.', ',', '-']:
            return None
        
        # Detectar formato: si tiene coma después de punto, es formato europeo
        if ',' in s and '.' in s:
            if s.rindex(',') > s.rindex('.'):
                # Formato europeo: 1.234,56
                s = s.replace('.', '').replace(',', '.')
            else:
                # Formato americano: 1,234.56
                s = s.replace(',', '')
        elif ',' in s:
            # Solo comas: puede ser separador de miles o decimal
            # Si hay más de una coma, es separador de miles
            if s.count(',') > 1:
                s = s.replace(',', '')
            else:
                # Una sola coma: puede ser decimal (europeo) o miles (americano)
                # Asumimos decimal si hay 1-2 dígitos después
                partes = s.split(',')
                if len(partes) == 2 and len(partes[1]) <= 2:
                    s = s.replace(',', '.')
                else:
                    s = s.replace(',', '')
        
        return float(s)
    
    except (ValueError, AttributeError):
        return None


def limpiar_porcentaje(valor) -> float:
    """
    Extrae un porcentaje y lo convierte a decimal.
    
    Ejemplos:
        "25.79%" → 25.79
        "4.39" → 4.39
        "25,79%" → 25.79
    """
    if pd.isna(valor) or valor == '' or valor is None:
        return None
    
    try:
        s = str(valor).strip()
        # Eliminar símbolo de porcentaje
        s = s.replace('%', '').strip()
        return limpiar_numero(s)
    
    except:
        return None

print("✅ Funciones auxiliares AEBE cargadas: limpiar_numero(), limpiar_porcentaje()")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DICCIONARIO DE CONOCIMIENTO CORREGIDO
# Mapeo archivo → tabla Delta (usado por el LLM para decidir destino)
# ═══════════════════════════════════════════════════════════════════════════

DICCIONARIO_CONOCIMIENTO = """
{
  "mapeo_archivos": [
    {"archivo": "ESPAC_T13",                    "tabla_destino": "espac_banano_platano_provincia", "fuente": "INEC_ESPAC"},
    {"archivo": "ESPAC_T26",                    "tabla_destino": "espac_banano_platano_provincia", "fuente": "INEC_ESPAC"},
    {"archivo": "ESPAC_Tabulados_excel",        "tabla_destino": "espac_banano_platano_provincia", "fuente": "INEC_ESPAC"},
    {"archivo": "ESPAC_Series_historicas",      "tabla_destino": "espac_banano_platano_provincia", "fuente": "INEC_ESPAC"},
    {"archivo": "ESPAC_USO_DEL_SUELO",          "tabla_destino": "espac_uso_del_suelo",            "fuente": "INEC_ESPAC"},
    {"archivo": "SIPA_USO_SUELO",               "tabla_destino": "espac_uso_del_suelo",            "fuente": "SIPA_MAG"},
    {"archivo": "TEMPERATURA_Y_PRECIPITACION",  "tabla_destino": "sipa_temperatura_precipitacion", "fuente": "SIPA_MAG"},
    {"archivo": "SIPA_TEMPERATURA",             "tabla_destino": "sipa_temperatura_precipitacion", "fuente": "SIPA_MAG"},
    {"archivo": "FAOSTAT",                      "tabla_destino": "faostat_produccion_banano_platano", "fuente": "FAOSTAT"},
    {"archivo": "AEBE_BANANOTAS",               "tabla_destino": "aebe_exportaciones_regiones",      "fuente": "AEBE_BANANOTAS",      "descripcion": "Rankings consolidados: exportadores, marcas, navieras, puertos, mercados. Año extraído del contenido (max year en texto). Estructura: tipo|ranking|dato|cantidad_2025|cantidad_2026|participacion_2025|participacion_2026|variacion_abs|variacion_pct|medida|anio|archivo"}
  ]
}
"""

# ═══════════════════════════════════════════════════════════════════════════
# CREAR/VERIFICAR TABLA AEBE CON SCHEMA CORRECTO
# ═══════════════════════════════════════════════════════════════════════════

# Borrar tabla vieja si existe con schema incorrecto
try:
    spark.sql(f"DROP TABLE IF EXISTS {DB_NAME}.aebe_exportaciones_regiones")
    print("♻️  Tabla aebe_exportaciones_regiones eliminada para recrear con schema correcto")
except:
    pass

# Crear tabla con schema CORRECTO (estructura normalizada)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.aebe_exportaciones_regiones (
    tipo              STRING COMMENT 'Tipo de ranking: exportador, marca, naviera, puerto, mercado',
    dato              STRING COMMENT 'Nombre de la entidad (empresa, marca, puerto, país)',
    cantidad          DOUBLE COMMENT 'Valor para el año detectado (cajas en millones)',
    medida            STRING COMMENT 'Unidad de medida (ej: cajas (millones))',
    anio              INT    COMMENT 'Año del dato (extraído del documento, siempre el mayor)',
    framework         STRING COMMENT 'Framework ETL usado: langraph o llamaindex',
    fuente            STRING COMMENT 'Fuente: AEBE_BANANOTAS',
    archivo_origen    STRING COMMENT 'Nombre del archivo PDF procesado',
    fecha_carga       TIMESTAMP COMMENT 'Timestamp de carga a Delta'
) USING DELTA
COMMENT 'Rankings de exportaciones de banano Ecuador desde AEBE Bananotas (estructura normalizada)'
""")

print("✅ Diccionario de conocimiento cargado.")
print("   • ESPAC_Tabulados_excel y ESPAC_Series_historicas → espac_banano_platano_provincia")
print("   • FAOSTAT → faostat_produccion_banano_platano")
print("   • SIPA_TEMPERATURA → sipa_temperatura_precipitacion")
print("✅ Tabla AEBE creada/verificada:")
print("   • AEBE_BANANOTAS → aebe_exportaciones_regiones (tipo|dato|cantidad|medida|año)")

## 🔄 Bloque 6 — ETL Workflow: Transformación y Carga (Agente 2)

Replica el Agente ETL de LangGraph con la misma lógica de transformación:

| Paso | LangGraph (nodo) | LlamaIndex (@step) |
|------|------------------|--------------------|
| 1 | `nodo_detectar_archivo` | `detectar_archivo` |
| 2 | `nodo_leer_archivo` | `leer_archivo_step` |
| 3 | `nodo_consultar_llm_mapeo` | `mapear_columnas` |
| 4 | `nodo_transformar_datos` | `transformar_datos` |
| 5 | `nodo_cargar_delta` | `cargar_delta` |

Las métricas M1–M6 se registran en las mismas tablas Delta con `framework = 'LlamaIndex'`.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# TABLAS DE MÉTRICAS ETL (compatibles con LangGraph)
# ══════════════════════════════════════════════════════════════════════════

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TRANSFORM} (
    execution_id        STRING,
    file_name           STRING,
    fuente              STRING,
    framework           STRING,
    tabla_destino       STRING,
    filas_entrada       INT,
    filas_salida        INT,
    duplicados_elim     INT,
    calidad_pct         DOUBLE,
    tasa_error_transformacion DOUBLE,
    tiempo_segundos     DOUBLE,
    timestamp_inicio    TIMESTAMP,
    timestamp_fin       TIMESTAMP,
    analisis_agente     STRING,
    llamadas_llm        INT,
    status              STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_CARGA} (
    execution_id         STRING,
    file_name            STRING,
    fuente               STRING,
    framework            STRING,
    tabla_destino        STRING,
    registros_insertados INT,
    tiempo_escritura_s   DOUBLE,
    timestamp_inicio     TIMESTAMP,
    timestamp_fin        TIMESTAMP,
    analisis_agente      STRING,
    llamadas_llm         INT,
    status               STRING
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    execution_id            STRING,
    file_name               STRING,
    framework               STRING,
    fuente                  STRING,
    tabla_destino           STRING,
    execution_timestamp     TIMESTAMP,
    status                  STRING,
    m1_tiempo_segundos      DOUBLE,
    m2_calidad_pct          DOUBLE,
    m3_llamadas_api         INT,
    m5_tasa_error                 DOUBLE,
    total_filas             INT,
    registros_validos       INT,
    registros_duplicados    INT,
    llm_response            STRING,
    error_message           STRING
) USING DELTA
""")

print("OK Tablas de metricas ETL listas (compatibles con LangGraph).")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EVENTOS AGENTE 2: ETL (Transformación + Carga)
# ══════════════════════════════════════════════════════════════════════════

class ETLStartEvent(Event):
    nombre_archivo:   str
    ruta_archivo:     str
    fuente:           str
    timestamp_inicio: str

class ArchivoDetectadoEvent(Event):
    nombre_archivo:   str
    ruta_archivo:     str
    fuente:           str
    extension:        str
    timestamp_inicio: str
    llamadas_llm:     int

class ArchivoLeidoEvent(Event):
    nombre_archivo:   str
    ruta_archivo:     str
    fuente:           str
    extension:        str
    columnas_raw:     List[str]
    n_registros_raw:  int
    timestamp_inicio: str
    llamadas_llm:     int

class ColumnasMapeadasEvent(Event):
    nombre_archivo:   str
    ruta_archivo:     str
    fuente:           str
    tabla_destino:    str
    cols_double:      List[str]
    cols_integer:     List[str]
    razonamiento_llm: str
    columnas_raw:     List[str]
    n_registros_raw:  int
    timestamp_inicio: str
    llamadas_llm:     int

class DatosTransformadosEvent(Event):
    nombre_archivo:   str
    fuente:           str
    tabla_destino:    str
    ruta_archivo:     str
    n_registros:      int
    registros_dup:    int
    calidad_pct:      float
    tasa_error:             float
    columnas_raw:     List[str]
    temp_view_name:   str
    razonamiento_llm: str
    timestamp_inicio: str
    llamadas_llm:     int

class ETLErrorEvent(Event):
    nombre_archivo:   str
    fuente:           str
    causa:            str
    timestamp_inicio: str
    llamadas_llm:     int


def _mapeo_por_reglas(file_name: str, columnas: List[str]) -> dict:
    """Fallback identico al de LangGraph cuando el LLM falla."""
    fn   = file_name.upper()
    cols = " ".join(columnas).lower()
    if any(k in fn for k in ["TABULADOS_EXCEL","SERIES_HISTORICAS","SERIES_HIST","TABULADOS_ESPAC"]):
        return {"tabla":"espac_banano_platano_provincia",
                "double":["superficie_plantada_ha","superficie_cosechada_ha","produccion_tm","ventas_tm","rendimiento"],
                "int":["anio"]}
    if "T13" in fn or "T26" in fn:
        return {"tabla":"espac_banano_platano_provincia",
                "double":["superficie_plantada_ha","superficie_cosechada_ha","produccion_tm","ventas_tm","rendimiento"],
                "int":["anio"]}
    if "SIPA_USO" in fn or ("uso" in cols and "suelo" in cols):
        return {"tabla":"espac_uso_del_suelo","double":["superficie_ha"],"int":["anio"]}
    if "TEMPERATURA" in fn or ("temperatura" in cols and "precipitacion" in cols):
        return {"tabla":"sipa_temperatura_precipitacion",
                "double":["precipitacion_mm","temperatura_promedio_c"],"int":["anio"]}
    if "FAOSTAT" in fn:
        return {"tabla":"faostat_produccion_banano_platano","double":["value"],"int":["year","area_code","item_code"]}
    if "BANANO_PREC" in fn or "precios" in fn.lower():
        return {"tabla":"banano_precios_semanales","double":[],"int":["anio"]}
    return {"tabla":"tabla_temporal","double":[],"int":[]}


print("OK Eventos AGENTE 2 (ETL) y helper de mapeo definidos.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ESTADO ETL Y FUNCIONES DE TRANSFORMACIÓN
# Funciones compartidas con LangGraph (patrón bridge)
# ══════════════════════════════════════════════════════════════════════════
# ── DEFINICIÓN DEL ESTADO ──────────────────────────────────────────────────
class ETLState(TypedDict):
    # ── Input ─────────────────────────────────────────────────────────────
    file_name:              str
    local_path:             str
    # ── Nodo 1: Detección ─────────────────────────────────────────────────
    fuente:                 str
    inicio_ts:              str          # timestamp ISO inicio general
    # ── Nodo 2: Lectura ───────────────────────────────────────────────────
    df_columnas:            List[str]
    total_filas:            int
    ts_fin_lectura:         str          # timestamp ISO fin lectura
    error_lectura:          Optional[str]
    # ── Nodo 3: Mapeo Híbrido (LLM + Reglas) ──────────────────────────────────────
    tabla_destino:          str
    cols_double:            List[str]
    cols_integer:           List[str]
    llm_response:           str
    llamadas_api:           int
    error_mapeo:            Optional[str]
    # ── Nodo 4: Transformación ────────────────────────────────────────────
    registros_validos:      int
    registros_duplicados:   int
    ts_fin_transformacion:  str          # timestamp ISO fin transformación
    error_transform:        Optional[str]
    temp_view_name:         str          # nombre único temp view Spark
    # ── Nodo 5: Carga ─────────────────────────────────────────────────────
    tabla_completa:         str
    ts_fin_carga:           str          # timestamp ISO fin carga
    error_carga:            Optional[str]
    # ── Nodo 6: Métricas generales ────────────────────────────────────────
    tiempo_segundos:        float
    tasa_error:                   float
    calidad_pct:            float
    status:                 str
    error_final:            Optional[str]

# ── NODO 1: DETECCIÓN ─────────────────────────────────────────────────────
def nodo_deteccion(state: ETLState) -> dict:
    """
    Primer nodo del grafo. Identifica el organismo de origen del archivo
    y registra el timestamp de inicio para medir el tiempo total (M1).
    """
    print(f"\n{'='*55}")
    print(f"[NODO 1 — DETECCIÓN] {state['file_name']}")
    fuente = identificar_fuente(state["file_name"])
    print(f"  Fuente: {fuente}")
    return {"fuente": fuente, "inicio_ts": datetime.now().isoformat(), "llamadas_api": 0}

# ── NODO 2: LECTURA ───────────────────────────────────────────────────────
def nodo_lectura(state: ETLState) -> dict:
    """
    Lee el archivo físico del volumen y detecta automáticamente el header.
    Registra ts_fin_lectura para calcular la métrica de tiempo de lectura.
    """
    print(f"[NODO 2 — LECTURA]")
    try:
        df = leer_archivo(state["local_path"], state["file_name"])
        n_filas, n_cols = df.shape
        print(f"  {n_filas} filas × {n_cols} columnas")
        print(f"  Primeras columnas: {list(df.columns)[:5]}")
        if n_filas == 0:
            return {"error_lectura":"Archivo vacío","total_filas":0,"df_columnas":[],
                    "ts_fin_lectura": datetime.now().isoformat()}
        return {"df_columnas":list(df.columns), "total_filas":n_filas, "error_lectura":None,
                "ts_fin_lectura": datetime.now().isoformat()}
    except Exception as e:
        print(f"  ❌ {e}")
        return {"error_lectura":str(e),"total_filas":0,"df_columnas":[],
                "ts_fin_lectura": datetime.now().isoformat()}

# ── NODO 3: MAPEO SEMÁNTICO CON LLM + FALLBACK ──────────────────────────
def _mapeo_basado_en_reglas(file_name: str, columnas: List[str]) -> dict:
    """
    Fallback basado en reglas cuando el LLM falla.
    Usa el nombre del archivo y columnas para determinar la tabla destino.
    """
    fn = file_name.upper()
    cols_text = " ".join(columnas).lower()
    
    # Reglas por nombre de archivo
    # PRIORIDAD MÁXIMA: ESPAC Tabulados y Series históricas → TABLA UNIFICADA banano/plátano
    # (debe estar ANTES de otras reglas para evitar falsos positivos)
    if "TABULADOS_EXCEL" in fn or "TABULADOS" in fn or "SERIES_HISTORICAS" in fn or "SERIES_HIST" in fn:
        return {"tabla":"espac_banano_platano_provincia", "conf":0.95, "double":["superficie_plantada_ha","superficie_cosechada_ha","produccion_tm","ventas_tm","rendimiento"], "int":["anio"]}
    
    # ESPAC T13 (banano) y T26 (plátano) → TABLA UNIFICADA
    if "T13" in fn or "T26" in fn:
        return {"tabla":"espac_banano_platano_provincia", "conf":0.95, "double":["superficie_plantada_ha","superficie_cosechada_ha","produccion_tm","ventas_tm","rendimiento"], "int":["anio"]}
    
    if "USO_DEL_SUELO" in fn or "uso" in cols_text and "suelo" in cols_text:
        return {"tabla":"espac_uso_del_suelo", "conf":0.85, "double":["superficie_ha"], "int":["ano"]}
    if "TEMPERATURA" in fn or ("temperatura" in cols_text and "precipitacion" in cols_text):
        return {"tabla":"sipa_temperatura_precipitacion", "conf":0.95, "double":["precipitacion_mm","temperatura_promedio_c"], "int":["ano"]}
    
    # FAOSTAT bananas y plantains → TABLA UNIFICADA
    if "FAOSTAT" in fn and ("BANANAS" in fn or "PLANTAINS" in fn):
        return {"tabla":"faostat_produccion_banano_platano", "conf":0.95, "double":["value"], "int":["year","area_code","item_code"]}
    
    if "BANANO_PREC" in fn or "precios" in fn.lower():
        return {"tabla":"banano_precios_semanales", "conf":0.85, "double":["22xu","2080"], "int":["cont"]}
    
    # Fallback genérico
    return {"tabla":"tabla_temporal", "conf":0.3, "double":[], "int":[]}

def nodo_mapeo(state: ETLState) -> dict:
    """
    Mapeo híbrido: intenta LLM primero, fallback a reglas si falla.
    Cada llamada LLM incrementa el contador M5.
    """
    print(f"[NODO 3 — MAPEO HÍBRIDO] columnas: {state['df_columnas'][:4]}...")
    # Estrategia 1: Intentar LLM (solo 1 intento para ser más rápido)
    llamadas = state.get("llamadas_api", 0)
    try:
        print(f"  LLM — intento 1/1...")
        
        # Prompt simplificado para mejor respuesta
        prompt_simple = f"""Tabla destino para: {state['file_name']}
Columnas: {', '.join(state['df_columnas'][:8])}

Opciones:
- espac_banano_platano_provincia (T13 o T26, banano/plátano por provincia UNIFICADO)
- espac_uso_del_suelo (uso del suelo)
- espac_cultivos_permanentes (tabulados generales)
- sipa_temperatura_precipitacion (clima)
- faostat_produccion_banano_platano (FAO bananas/plantains UNIFICADO)
- banano_precios_semanales (precios)

Respuesta JSON:
{{"tabla_destino":"","confianza":0.0,"columnas_double":[],"columnas_integer":[]}}"""
        
        resp = llm.invoke([HumanMessage(content=prompt_simple)])
        llamadas += 1
        
        if resp.content and resp.content.strip():
            texto = re.sub(r"^```json\s*|^```\s*|```$","",resp.content.strip(),flags=re.MULTILINE).strip()
            start = texto.find("{")
            end = texto.rfind("}")
            if start != -1 and end != -1:
                texto = texto[start:end+1]
                mapeo = json.loads(texto)
                tabla = mapeo.get("tabla_destino","").lower().replace(" ","_")
                if tabla and tabla != "tabla_temporal":
                    print(f"  ✅ LLM: '{tabla}' (confianza: {mapeo.get('confianza',0)})")
                    return {"tabla_destino":tabla, "cols_double":mapeo.get("columnas_double",[]),
                            "cols_integer":mapeo.get("columnas_integer",[]),
                            "llm_response":json.dumps(mapeo,ensure_ascii=False),
                            "llamadas_api":llamadas, "error_mapeo":None}
    except Exception as e:
        print(f"  ⚠️ LLM falló: {str(e)[:50]}...")
    
    # Estrategia 2: Fallback basado en reglas
    print(f"  🔧 Usando mapeo basado en reglas...")
    fallback = _mapeo_basado_en_reglas(state["file_name"], state["df_columnas"])
    print(f"  ✅ Regla: '{fallback['tabla']}' (confianza: {fallback['conf']})")
    
    return {"tabla_destino":fallback["tabla"], 
            "cols_double":fallback["double"],
            "cols_integer":fallback["int"],
            "llm_response":json.dumps({"metodo":"fallback_reglas","confianza":fallback["conf"]},ensure_ascii=False),
            "llamadas_api":llamadas, 
            "error_mapeo":None}

# ── FUNCIÓN AUXILIAR: TRANSFORMACIÓN ESPECIALIZADA T13/T26 ───────────────
def transformar_espac_t13_t26(df_pandas: pd.DataFrame, nombre_archivo: str) -> pd.DataFrame:
    """
    Transforma archivos T13 (banano) y T26 (plátano) de ESPAC al formato estandarizado.
    Pasos: Elimina sub-headers, renombra columnas, filtra provincias y notas, convierte numéricos, extrae año.
    """
    import numpy as np
    print(f"  📋 Transformación específica T13/T26 para: {nombre_archivo}")
    
    # Extraer año del nombre del archivo con regex flexible
    # Captura: _2021_, 2021.csv, ESPAC2021, T13_2022, etc.
    anio_match = re.search(r'(\d{4})', nombre_archivo)
    anio = int(anio_match.group(1)) if anio_match else None
    print(f"    📅 Año extraído del archivo: {anio if anio else 'No detectado'}")
    
    df = df_pandas.iloc[1:].copy()
    df.columns = ['PROVINCIA','CATEGORIA','SUPERFICIE_PLANTADA_HA','SUPERFICIE_COSECHADA_HA','PRODUCCION_TM','VENTAS_TM']
    
    # Filtrar filas que NO son datos reales (provincias)
    palabras_excluir = ['TOTAL', 'REGIÓN', 'ZONA', 'NOTA', 'FUENTE', 'OBSERVACIÓN', 'OBSERVACION', 
                        'La tabla', 'INSTITUT', 'INEC', 'HTTP', 'WWW', 'ENCUESTA']
    
    df = df[
        df['PROVINCIA'].notna() &
        (~df['PROVINCIA'].astype(str).str.upper().str.contains('|'.join(palabras_excluir), na=False, case=False, regex=True))
    ].copy()
    
    print(f"    🗑️  Filtradas filas con notas/totales/fuentes")
    print(f"    → {len(df)} provincias válidas después del filtrado")
    
    # Convertir columnas numéricas
    for col in ['SUPERFICIE_PLANTADA_HA', 'SUPERFICIE_COSECHADA_HA', 'PRODUCCION_TM', 'VENTAS_TM']:
        df[col] = df[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    
    # Agregar columna de año
    df['ANIO'] = anio if anio else 0
    
    # Agregar columnas calculadas
    df['PRODUCTO'] = 'BANANO' if 'T13' in nombre_archivo.upper() else 'PLATANO' if 'T26' in nombre_archivo.upper() else 'DESCONOCIDO'
    df['RENDIMIENTO'] = np.where(df['SUPERFICIE_COSECHADA_HA'] > 0, (df['PRODUCCION_TM'] / df['SUPERFICIE_COSECHADA_HA']).round(2), 0)
    
    # Limpiar strings
    df['PROVINCIA'] = df['PROVINCIA'].astype(str).str.strip()
    df['CATEGORIA'] = df['CATEGORIA'].fillna('Solo').astype(str).str.strip()
    
    df.reset_index(drop=True, inplace=True)
    print(f"    ✅ Transformación completada: {len(df)} registros, {len(df.columns)} columnas (incluye ANIO)")
    return df

# ── NODO 4: TRANSFORMACIÓN ────────────────────────────────────────────────
def nodo_transformacion(state: ETLState) -> dict:
    """
    Transformaciones de calidad de dato en Spark:
    1. Casteo numérico con guía del LLM
    2. Limpieza de strings nulos/vacíos/sin sentido
    3. Filtro banano/plátano si hay columna de producto
    4. Deduplicación — base para calcular tasa_error y M4
    Registra ts_fin_transformacion y escribe en LOG_TRANSFORM.
    """
    print(f"[NODO 4 — TRANSFORMACIÓN]")
    ts_ini_t = datetime.now()
    try:
        nombre_upper = state["file_name"].upper()
        
        # ── CORRECCIÓN #6: TRANSFORMACIÓN PRECIOS ────────────────────────
        if "PRECIOS" in nombre_upper or "BANANO_PREC" in nombre_upper:
            print(f"  🎯 Detectado archivo de Precios — Aplicando transformación especializada")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])  # Leer primero
            df_pandas = transformar_precios(df_pandas, state["file_name"])     # Transformar estructura alternada
        
        # ── CORRECCIÓN #7: TRANSFORMACIÓN ESPAC TABULADOS/SERIES HISTÓRICAS ──
        elif "TABULADOS_EXCEL" in nombre_upper or "SERIES_HISTORICAS" in nombre_upper or "SERIES_HIST" in nombre_upper:
            print(f"  🎯 Detectado ESPAC Tabulados/Series Históricas — Aplicando transformación de múltiples hojas")
            df_pandas = transformar_espac_t13_t26_mejorado(state["local_path"], state["file_name"])
            # CORRECCIÓN #15: Filtrar registros con provincia_id NULL
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id")
        
        # ── TRANSFORMACIÓN ESPECÍFICA PARA T13/T26 (archivos individuales) ──
        elif "T13" in nombre_upper or "T26" in nombre_upper:
            print(f"  🎯 Detectado archivo T13/T26 — Aplicando transformación especializada")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_espac_t13_t26(df_pandas, state["file_name"])
            # CORRECCIÓN #15: Filtrar registros con provincia_id NULL
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id")
        
        # ── TRANSFORMACIÓN FAOSTAT ──────────────────────────────────────
        elif "FAOSTAT" in nombre_upper:
            print(f"  🎯 Detectado archivo FAOSTAT — Aplicando transformación especializada")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_faostat(df_pandas, state["file_name"])
        
        # ── TRANSFORMACIÓN SIPA TEMPERATURA ──────────────────────────────
        elif "TEMPERATURA" in nombre_upper or ("SIPA" in nombre_upper and "USO" not in nombre_upper):
            print(f"  🎯 Detectado SIPA Temperatura — Aplicando transformación especializada con mapeo provincia_id")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_sipa_temperatura(df_pandas, state["file_name"])
            # CORRECCIÓN #15: Filtrar registros con provincia_id NULL
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id")
        
        # ── TRANSFORMACIÓN USO DEL SUELO ──────────────────────────────────
        elif "USO_DEL_SUELO" in nombre_upper or "USO" in nombre_upper:
            print(f"  🎯 Detectado Uso del Suelo — Aplicando transformación especializada con mapeo provincia_id")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_uso_del_suelo(df_pandas, state["file_name"])
            # CORRECCIÓN #15: Filtrar registros con provincia_id NULL (totales regionales)
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id (totales regionales)")
        
        # ── TRANSFORMACIÓN ESTÁNDAR ──────────────────────────────────────
        else:
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
        
        df_spark  = spark.createDataFrame(df_pandas)
        df_spark  = castear_columnas(df_spark, state["cols_double"], state["cols_integer"])
        df_spark  = (df_spark
            .withColumn("pipeline_source_file", F.lit(state["file_name"]))
            .withColumn("pipeline_framework",   F.lit("LangGraph"))
            .withColumn("pipeline_load_ts",     F.current_timestamp()))

        # CORRECCIÓN SCPAP001 + SCPAP004: Cachear columnas antes del loop y usar transformación batch
        cols_to_clean = df_spark.columns  # Cachear lista de columnas UNA VEZ
        clean_exprs = {}
        for c in cols_to_clean:
            clean_exprs[c] = when(
                trim(col(c).cast("string")).isin("","null","NULL","None","nan","NaN",".","-","0"),
                None).otherwise(trim(col(c).cast("string")))
        df_spark = df_spark.withColumns(clean_exprs)  # Aplicar TODAS las transformaciones de una vez
        df_spark = df_spark.na.drop(how="all")

        # ── EXTRACCIÓN DE AÑO (para todas las tablas) ────────────────────────
        # Extraer año del nombre del archivo si no existe columna de año
        cols_anio = [c for c in df_spark.columns if c.lower() in ['anio', 'ano', 'year', 'fecha']]
        if not cols_anio:
            anio_match = re.search(r'(\d{4})', state["file_name"])
            if anio_match:
                anio_extraido = int(anio_match.group(1))
                df_spark = df_spark.withColumn("anio", F.lit(anio_extraido))
                print(f"  📅 Año extraído del nombre: {anio_extraido}")
        
        # ── LIMPIEZA DE COLUMNAS VACÍAS (>95% nulos) ─────────────────────────
        # Aplicar solo a tablas que lo necesiten (precios semanales tiene muchas columnas vacías)
        if "precios" in state.get("tabla_destino", "").lower():
            total_rows = df_spark.count()
            if total_rows > 0:
                # Columnas de metadatos a preservar
                meta_cols_preserve = {'_input_file_name', '_rescued_data', '_metadata', 'anio', 'pipeline_source_file', 'pipeline_load_ts', 'pipeline_framework'}
                cols_to_drop = []
                for col_name in df_spark.columns:
                    if col_name not in meta_cols_preserve:
                        null_count = df_spark.filter(col(col_name).isNull() | (trim(col(col_name)) == "")).count()
                        null_pct = (null_count / total_rows) * 100
                        if null_pct > 95:
                            cols_to_drop.append(col_name)
                
                if cols_to_drop:
                    df_spark = df_spark.drop(*cols_to_drop)
                    print(f"  🗑️  Eliminadas {len(cols_to_drop)} columnas vacías (>95% nulos)")
        
        # ── NO APLICAR FILTRO DE BANANO/PLÁTANO ──────────────────────────────
        # Los archivos ya vienen filtrados por fuente (ESPAC solo descarga archivos de banano)
        # El filtro anterior estaba eliminando TODOS los registros porque buscaba
        # columnas que no existían. DESHABILITADO.
        print(f"  ✓ Filtro de producto deshabilitado (archivos pre-filtrados por fuente)")

        meta_cols = {"pipeline_source_file","pipeline_load_ts","pipeline_framework"}
        # CORRECCIÓN SCPAP001: Cachear columnas antes del list comprehension
        all_columns = df_spark.columns
        real_cols = [c for c in all_columns if c not in meta_cols]
        
        # CORRECCIÓN SCPAP005: Materializar con count() (NO usar .cache() en Serverless)
        antes = df_spark.count()
        df_spark = df_spark.dropDuplicates(subset=real_cols)
        despues = df_spark.count()
        duplicados = antes - despues
        print(f"  Registros: {antes} → {despues} (−{duplicados} duplicados)")

        # CORRECCIÓN SCPAP003: Usar nombre único para temp view (evitar sobrescritura silenciosa)
        temp_view_name = f"langgraph_etl_temp_{state['file_name'].replace('.','_').replace('-','_')}"
        df_spark.createOrReplaceTempView(temp_view_name)

        ts_fin_t = datetime.now()
        tiempo_t = (ts_fin_t - ts_ini_t).total_seconds()

        # ── Métricas de Transformación ──────────────────────────────────
        n_cols   = max(len(state["df_columnas"]),1)
        total    = max(state["total_filas"],1)
        oport_t  = total * n_cols
        defect_t = (total - despues) + duplicados
        tasa_error_t   = round(defect_t / oport_t * 1_000_000, 2)
        calidad_t= round(despues / total * 100, 2)

        schema_t = StructType([
            StructField("file_name",         StringType(),  True),
            StructField("fuente",            StringType(),  True),
            StructField("tabla_destino",     StringType(),  True),
            StructField("filas_entrada",     IntegerType(), True),
            StructField("filas_salida",      IntegerType(), True),
            StructField("duplicados_elim",   IntegerType(), True),
            StructField("calidad_pct",       DoubleType(),  True),
            StructField("tasa_error_transformacion",DoubleType(), True),
            StructField("tiempo_segundos",   DoubleType(),  True),
            StructField("timestamp_inicio",  TimestampType(),True),
            StructField("timestamp_fin",     TimestampType(),True),
        ])
        spark.createDataFrame([(
            state["file_name"], state["fuente"], state.get("tabla_destino","?"),
            total, despues, duplicados, calidad_t, tasa_error_t, tiempo_t, ts_ini_t, ts_fin_t
        )], schema_t).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_TRANSFORM)

        print(f"  M2={calidad_t:.1f}%  Tasa_Error_T={tasa_error_t:.1f}  Tiempo={tiempo_t:.1f}s  → log guardado")
        return {"registros_validos":despues, "registros_duplicados":duplicados,
                "ts_fin_transformacion": ts_fin_t.isoformat(), "error_transform":None,
                "temp_view_name": temp_view_name}  # Pasar nombre de vista temporal al siguiente nodo

    except Exception as e:
        print(f"  ❌ {e}")
        return {"registros_validos":0,"registros_duplicados":0,
                "ts_fin_transformacion": datetime.now().isoformat(), "error_transform":str(e)}

# ── NODO 5: CARGA EN DELTA LAKE ───────────────────────────────────────────
def nodo_carga(state: ETLState) -> dict:
    """
    Persiste el DataFrame en Delta Lake con mergeSchema=true.
    
    CORRECCIÓN #20: Estrategia diferenciada por tipo de tabla:
    - MÉTRICAS/LOGS: Siempre APPEND (acumular histórico)
    - DATOS: Si framework diferente → OVERWRITE (reemplazar todo)
             Si mismo framework → APPEND (agregar solo nuevos)
    
    Registra las métricas de carga (tiempo de escritura, registros insertados)
    en la tabla LOG_CARGA para análisis independiente de esta fase.
    """
    tabla_completa = f"{DB_NAME}.{state['tabla_destino']}"
    print(f"[NODO 5 — CARGA] → {tabla_completa}")
    ts_ini_c = datetime.now()
    try:
        # Usar el nombre único de temp view del nodo anterior
        temp_view_name = state.get("temp_view_name", "langgraph_etl_temp")
        df_spark = spark.table(temp_view_name)
        
        # CORRECCIÓN SCPAP005: Ejecutar count() dentro del try para materializar
        registros_a_insertar = df_spark.count()

        # ⭐ CORRECCIÓN #20: Determinar modo de escritura según tipo de tabla
        es_tabla_metricas = any(keyword in tabla_completa.lower() for keyword in 
                                ['control_logs', 'metricas_', 'log_'])
        
        modo_escritura = "append"  # Por defecto
        
        if not es_tabla_metricas:
            # Es tabla de DATOS → verificar si hay framework diferente
            if spark.catalog.tableExists(tabla_completa):
                try:
                    # Verificar si existe la columna pipeline_framework
                    df_existente = spark.table(tabla_completa)
                    
                    if 'pipeline_framework' in df_existente.columns:
                        frameworks_existentes = df_existente.select('pipeline_framework').distinct().collect()
                        frameworks_set = {r['pipeline_framework'] for r in frameworks_existentes if r['pipeline_framework']}
                        
                        if frameworks_set and FRAMEWORK_NAME not in frameworks_set:
                            # Hay datos de otro framework → OVERWRITE
                            modo_escritura = "overwrite"
                            print(f"  🔄 Framework diferente detectado ({frameworks_set}) → OVERWRITE")
                        else:
                            # Mismo framework → APPEND
                            print(f"  ➕ Mismo framework ({FRAMEWORK_NAME}) → APPEND")
                    else:
                        # No tiene columna framework (tabla antigua) → APPEND por seguridad
                        print(f"  ℹ️  Tabla sin columna framework → APPEND")
                except Exception as e:
                    print(f"  ⚠️  No se pudo verificar framework: {e} → usando APPEND")
            else:
                print(f"  🆕 Tabla nueva → CREATE")
        else:
            print(f"  📊 Tabla de métricas → APPEND (acumular histórico)")

        # Si la tabla destino existe y es APPEND, castear columnas para coincidir con el schema
        if spark.catalog.tableExists(tabla_completa) and modo_escritura == "append":
            schema_dest = spark.table(tabla_completa).schema
            # CORRECCIÓN SCPAP001 + SCPAP004: Cachear columnas y schema, usar batch cast
            df_cols = df_spark.columns
            cast_exprs = {}
            for campo in schema_dest.fields:
                if campo.name in df_cols:
                    cast_exprs[campo.name] = col(campo.name).cast(campo.dataType)
            if cast_exprs:  # Solo aplicar si hay columnas que castear
                df_spark = df_spark.withColumns(cast_exprs)

        # CORRECCIÓN SCPAP005: write.saveAsTable() es una action, materializa inmediatamente
        df_spark.write.format("delta").mode(modo_escritura).option("mergeSchema","true").saveAsTable(tabla_completa)
        ts_fin_c  = datetime.now()
        tiempo_c  = (ts_fin_c - ts_ini_c).total_seconds()

        print(f"  ✅ {registros_a_insertar} registros en {tabla_completa} ({tiempo_c:.1f}s)")

        # ── Métricas de Carga ────────────────────────────────────────────
        schema_c = StructType([
            StructField("file_name",         StringType(),  True),
            StructField("fuente",            StringType(),  True),
            StructField("tabla_destino",     StringType(),  True),
            StructField("registros_insertados",IntegerType(),True),
            StructField("tiempo_escritura_s",DoubleType(),  True),
            StructField("status",            StringType(),  True),
            StructField("timestamp_inicio",  TimestampType(),True),
            StructField("timestamp_fin",     TimestampType(),True),
        ])
        spark.createDataFrame([(
            state["file_name"], state["fuente"], tabla_completa,
            registros_a_insertar, tiempo_c, "OK", ts_ini_c, ts_fin_c
        )], schema_c).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_CARGA)

        return {"tabla_completa":tabla_completa, "ts_fin_carga":ts_fin_c.isoformat(), "error_carga":None}

    except Exception as e:
        ts_fin_c = datetime.now()
        print(f"  ❌ {e}")
        schema_c = StructType([
            StructField("file_name",         StringType(),  True),
            StructField("fuente",            StringType(),  True),
            StructField("tabla_destino",     StringType(),  True),
            StructField("registros_insertados",IntegerType(),True),
            StructField("tiempo_escritura_s",DoubleType(),  True),
            StructField("status",            StringType(),  True),
            StructField("timestamp_inicio",  TimestampType(),True),
            StructField("timestamp_fin",     TimestampType(),True),
        ])
        spark.createDataFrame([(
            state["file_name"], state["fuente"], tabla_completa,
            0, 0.0, f"ERROR: {str(e)[:100]}", ts_ini_c, ts_fin_c
        )], schema_c).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_CARGA)
        return {"tabla_completa":tabla_completa, "ts_fin_carga":ts_fin_c.isoformat(), "error_carga":str(e)}

# ── NODO 6: MÉTRICAS GENERALES M1–M6 ────────────────────────────────────
def nodo_metricas(state: ETLState) -> dict:
    """
    Calcula las 6 métricas del experimento para el proceso ETL completo
    y las graba en la tabla de logs general (LOG_TABLE).

    M1 Tiempo de ejecución    → segundos inicio→fin completo
    M2 Intervención manual    → siempre 0 (LangGraph es 100% autónomo)
    M3 Recuperación errores   → 2=sin reintentos, 1=hubo reintentos, 0=falló
    M2 Calidad del dato       → % registros válidos / total leído
    M3 Llamadas API LLM       → número de peticiones al LLM
    M5 Tasa de Error                   → proporción de registros con defecto por millón
    """
    print(f"[NODO 6 — MÉTRICAS GENERALES]")
    fin    = datetime.now()
    inicio = datetime.fromisoformat(state["inicio_ts"])
    tiempo = (fin - inicio).total_seconds()

    total   = max(state["total_filas"],1)
    validos = state["registros_validos"]
    calidad = round(validos / total * 100, 2)

    n_cols   = max(len(state["df_columnas"]),1)
    oport    = total * n_cols
    defectos = (total - validos) + state["registros_duplicados"]
    tasa_error     = round(defectos / oport * 1_000_000, 2)

    hay_error = any([state.get("error_lectura"),state.get("error_mapeo"),
                     state.get("error_transform"),state.get("error_carga")])
    status = "ERROR" if hay_error else "PROCESADO"

    print(f"  ┌──────────────────────────────────────────────")
    print(f"  │ M1 Tiempo total       : {tiempo:.1f}s")
    print(f"  │ M3 Llamadas API LLM   : {state.get('llamadas_api',0)}")
    print(f"  │ M5 Tasa de Error               : {tasa_error:.1f}")
    print(f"  │ Status                : {status}")
    print(f"  └──────────────────────────────────────────────")

    error_msg = " | ".join(filter(None,[
        state.get("error_lectura"),state.get("error_mapeo"),
        state.get("error_transform"),state.get("error_carga"),
    ])) or None

    log_schema = StructType([
        StructField("execution_id",            StringType(),  True),  # 🆔 CORRECCIÓN #21
        StructField("file_name",               StringType(),  True),
        StructField("framework",               StringType(),  True),
        StructField("fuente",                  StringType(),  True),
        StructField("tabla_destino",           StringType(),  True),
        StructField("execution_timestamp",     TimestampType(),True),
        StructField("status",                  StringType(),  True),
        StructField("m1_tiempo_segundos",      DoubleType(),  True),
        StructField("m2_calidad_pct",          DoubleType(),  True),
        StructField("m3_llamadas_api",         IntegerType(), True),
        StructField("m5_tasa_error",                 DoubleType(),  True),
        StructField("total_filas",             IntegerType(), True),
        StructField("registros_validos",       IntegerType(), True),
        StructField("registros_duplicados",    IntegerType(), True),
        StructField("llm_response",            StringType(),  True),
        StructField("error_message",           StringType(),  True),
    ])
    spark.createDataFrame([(
        EXECUTION_ID,  # 🆔 CORRECCIÓN #21
        state["file_name"],FRAMEWORK_NAME,state["fuente"],  # ⭐ CORRECCIÓN #20
        state.get("tabla_completa","N/A"),fin,status,
        tiempo, calidad, state.get("llamadas_api", 0), tasa_error,
        state["total_filas"],validos,state["registros_duplicados"],
        state.get("llm_response","{}"),error_msg,
    )], log_schema).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_TABLE)

    print(f"  ✅ Métricas generales registradas en {LOG_TABLE}")
    return {"tiempo_segundos":tiempo,"calidad_pct":calidad,"tasa_error":tasa_error,
            "status":status,"error_final":error_msg}

# ── NODO ERROR ────────────────────────────────────────────────────────────
def nodo_error(state: ETLState) -> dict:
    """
    Nodo de manejo centralizado de errores.
    LangGraph enruta aquí automáticamente cuando cualquier nodo falla.
    Esto es lo que mide M3 (Recuperación ante errores).
    """
    print(f"[NODO ERROR] — {state.get('file_name','?')}")
    errores = " | ".join(filter(None,[
        state.get("error_lectura"),state.get("error_mapeo"),
        state.get("error_transform"),state.get("error_carga"),
    ]))
    print(f"  Causa: {errores or 'desconocida'}")
    return {"status":"ERROR","error_final":errores or "Error desconocido"}

print("✅ Estado ETLState y 7 nodos del grafo definidos.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# NODO DE TRANSFORMACIÓN CON MÉTRICAS HOMOGENEIZADAS
# Todas las tablas tendrán las mismas columnas comunes:
# - fuente, timestamp_inicio, timestamp_fin, tiempo_segundos
# - llamadas_llm, analisis_agente (antes razonamiento_llm), status
# ══════════════════════════════════════════════════════════════════════════

# Esta celda redefine los nodos de transformación y carga para incluir las columnas faltantes

def nodo_transformacion_con_metricas(state: ETLState) -> dict:
    """
    Versión actualizada del nodo de transformación con métricas homogeneizadas.
    Incluye: llamadas_llm, analisis_agente, status.
    """
    print(f"[NODO 4 — TRANSFORMACIÓN CON MÉTRICAS HOMOGENEIZADAS]")
    ts_ini_t = datetime.now()
    try:
        nombre_upper = state["file_name"].upper()
        
        # ── TRANSFORMACIONES ESPECIALIZADAS (igual que antes) ──
        if "PRECIOS" in nombre_upper or "BANANO_PREC" in nombre_upper:
            print(f"  🎯 Detectado archivo de Precios — Aplicando transformación especializada")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_precios(df_pandas, state["file_name"])
        
        elif "TABULADOS_EXCEL" in nombre_upper or "SERIES_HISTORICAS" in nombre_upper or "SERIES_HIST" in nombre_upper:
            print(f"  🎯 Detectado ESPAC Tabulados/Series Históricas — Aplicando transformación de múltiples hojas")
            df_pandas = transformar_espac_t13_t26_mejorado(state["local_path"], state["file_name"])
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id")
        
        elif "T13" in nombre_upper or "T26" in nombre_upper:
            print(f"  🎯 Detectado archivo T13/T26 — Aplicando transformación especializada")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_espac_t13_t26(df_pandas, state["file_name"])
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id")
        
        elif "FAOSTAT" in nombre_upper:
            print(f"  🎯 Detectado archivo FAOSTAT — Aplicando transformación especializada")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_faostat(df_pandas, state["file_name"])
        
        elif "AEBE_BANANOTAS" in nombre_upper or ("AEBE" in nombre_upper and "BANANOTAS" in nombre_upper):
            print(f"  🎯 Detectado PDF AEBE Bananotas — Aplicando transformación especializada con pdfplumber v2")
            df_pandas = transformar_aebe_bananotas_v2(state["local_path"], state["file_name"])
        
        elif "TEMPERATURA" in nombre_upper or ("SIPA" in nombre_upper and "USO" not in nombre_upper):
            print(f"  🎯 Detectado SIPA Temperatura — Aplicando transformación especializada con mapeo provincia_id")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_sipa_temperatura(df_pandas, state["file_name"])
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id")
        
        elif "USO_DEL_SUELO" in nombre_upper or "USO" in nombre_upper:
            print(f"  🎯 Detectado Uso del Suelo — Aplicando transformación especializada con mapeo provincia_id")
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
            df_pandas = transformar_uso_del_suelo(df_pandas, state["file_name"])
            if 'provincia_id' in df_pandas.columns:
                antes_filtro = len(df_pandas)
                df_pandas = df_pandas[df_pandas['provincia_id'].notna()]
                eliminados = antes_filtro - len(df_pandas)
                if eliminados > 0:
                    print(f"  🗑️  Filtrados {eliminados} registros sin provincia_id (totales regionales)")
        
        else:
            df_pandas = leer_archivo(state["local_path"], state["file_name"])
        
        # ❌ CORRECCIÓN CRÍTICA: Validar que el DataFrame no esté vacío ANTES de createDataFrame
        if df_pandas is None or len(df_pandas) == 0:
            print(f"  ⚠️  DataFrame vacío después de transformación - retornando error")
            ts_fin_t = datetime.now()
            tiempo_t = (ts_fin_t - ts_ini_t).total_seconds()
            
            # Registrar métrica de transformación fallida
            columnas = state.get('df_columnas') or state.get('columnas_raw', [])
            analisis_agente = f"Transformación falló: DataFrame vacío | Archivo: {state['file_name']} | Posible PDF sin datos extraíbles"
            
            schema_t = StructType([
                StructField("execution_id",      StringType(),  True),
                StructField("file_name",         StringType(),  True),
                StructField("fuente",            StringType(),  True),
                StructField("framework",         StringType(),  True),
                StructField("tabla_destino",     StringType(),  True),
                StructField("filas_entrada",     IntegerType(), True),
                StructField("filas_salida",      IntegerType(), True),
                StructField("duplicados_elim",   IntegerType(), True),
                StructField("calidad_pct",       DoubleType(),  True),
                StructField("tasa_error_transformacion",DoubleType(), True),
                StructField("tiempo_segundos",   DoubleType(),  True),
                StructField("timestamp_inicio",  TimestampType(),True),
                StructField("timestamp_fin",     TimestampType(),True),
                StructField("analisis_agente",   StringType(),  True),
                StructField("llamadas_llm",      IntegerType(), True),
                StructField("status",            StringType(),  True),
            ])
            spark.createDataFrame([(
                EXECUTION_ID,
                state["file_name"], state["fuente"], FRAMEWORK_NAME,
                state.get("tabla_destino","?"),
                state.get("total_filas", 0), 0, 0, 0.0, 0.0, tiempo_t, ts_ini_t, ts_fin_t,
                analisis_agente[:1000], state.get("llamadas_api", 0), "ERROR"
            )], schema_t).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_TRANSFORM)
            
            return {"registros_validos":0, "registros_duplicados":0,
                    "ts_fin_transformacion": ts_fin_t.isoformat(), 
                    "error_transform":"DataFrame vacío después de transformación",
                    "temp_view_name": ""}
        
        df_spark  = spark.createDataFrame(df_pandas)
        df_spark  = castear_columnas(df_spark, state["cols_double"], state["cols_integer"])
        df_spark  = (df_spark
            .withColumn("pipeline_source_file", F.lit(state["file_name"]))
            .withColumn("pipeline_framework",   F.lit(FRAMEWORK_NAME))
            .withColumn("pipeline_load_ts",     F.current_timestamp()))

        # Limpiar columnas PERO preservar tipos INT/BIGINT
        cols_to_clean = df_spark.columns
        clean_exprs = {}
        
        # Identificar columnas que deben mantenerse como enteros
        columnas_enteras = ['anio', 'ano', 'year', 'mes', 'provincia_id', 'canton_id', 'pais_codigo', 'producto_codigo']
        
        for c in cols_to_clean:
            # Obtener el tipo actual de la columna
            col_tipo = dict(df_spark.dtypes)[c]
            
            # Si es una columna que debe ser entera Y ya es IntegerType/LongType, preservarla
            if c.lower() in columnas_enteras and col_tipo in ['int', 'bigint', 'integer', 'long']:
                # NO convertir a string, solo limpiar nulls/ceros invalidos
                clean_exprs[c] = when(
                    col(c).isNull() | (col(c) == 0),
                    None).otherwise(col(c))
            else:
                # Para otras columnas, convertir a string y limpiar
                clean_exprs[c] = when(
                    trim(col(c).cast("string")).isin("","null","NULL","None","nan","NaN",".","-","0"),
                    None).otherwise(trim(col(c).cast("string")))
        
        df_spark = df_spark.withColumns(clean_exprs)
        df_spark = df_spark.na.drop(how="all")

        cols_anio = [c for c in df_spark.columns if c.lower() in ['anio', 'ano', 'year', 'fecha']]
        if not cols_anio:
            anio_match = re.search(r'(\d{4})', state["file_name"])
            if anio_match:
                anio_extraido = int(anio_match.group(1))
                df_spark = df_spark.withColumn("anio", F.lit(anio_extraido))
                print(f"  📅 Año extraído del nombre: {anio_extraido}")
        
        if "precios" in state.get("tabla_destino", "").lower():
            total_rows = df_spark.count()
            if total_rows > 0:
                meta_cols_preserve = {'_input_file_name', '_rescued_data', '_metadata', 'anio', 'pipeline_source_file', 'pipeline_load_ts', 'pipeline_framework'}
                cols_to_drop = []
                for col_name in df_spark.columns:
                    if col_name not in meta_cols_preserve:
                        null_count = df_spark.filter(col(col_name).isNull() | (trim(col(col_name)) == "")).count()
                        null_pct = (null_count / total_rows) * 100
                        if null_pct > 95:
                            cols_to_drop.append(col_name)
                
                if cols_to_drop:
                    df_spark = df_spark.drop(*cols_to_drop)
                    print(f"  🗑️  Eliminadas {len(cols_to_drop)} columnas vacías (>95% nulos)")
        
        print(f"  ✓ Filtro de producto deshabilitado (archivos pre-filtrados por fuente)")

        meta_cols = {"pipeline_source_file","pipeline_load_ts","pipeline_framework"}
        all_columns = df_spark.columns
        real_cols = [c for c in all_columns if c not in meta_cols]
        
        antes = df_spark.count()
        df_spark = df_spark.dropDuplicates(subset=real_cols)
        despues = df_spark.count()
        duplicados = antes - despues
        print(f"  Registros: {antes} → {despues} (−{duplicados} duplicados)")

        temp_view_name = f"langgraph_etl_temp_{state['file_name'].replace('.','_').replace('-','_')}"
        df_spark.createOrReplaceTempView(temp_view_name)

        ts_fin_t = datetime.now()
        tiempo_t = (ts_fin_t - ts_ini_t).total_seconds()

        # ── Métricas de Transformación HOMOGENEIZADAS ──────────────────────────────────
        # Compatibilidad: LangGraph usa 'df_columnas', LlamaIndex usa 'columnas_raw'
        columnas = state.get('df_columnas') or state.get('columnas_raw', [])
        n_cols   = max(len(columnas),1)
        total    = max(state["total_filas"],1)
        oport_t  = total * n_cols
        defect_t = (total - despues) + duplicados
        tasa_error_t   = round(defect_t / oport_t * 1_000_000, 2)
        calidad_t= round(despues / total * 100, 2)
        
        # Construir análisis del agente
        analisis_partes = []
        
        # Extraer decisión del LLM del mapeo
        try:
            llm_data = json.loads(state.get("llm_response", "{}"))
            if "tabla_destino" in llm_data:
                analisis_partes.append(f"LLM mapeó a '{llm_data.get('tabla_destino')}' (conf={llm_data.get('confianza', 0)})")
            elif "metodo" in llm_data:
                analisis_partes.append(f"Fallback: {llm_data.get('metodo', 'N/A')}")
        except:
            analisis_partes.append("Mapeo: sin análisis LLM")
        
        # Agregar información de transformación especializada
        if "TABULADOS" in nombre_upper or "SERIES_HIST" in nombre_upper:
            analisis_partes.append("Trans: ESPAC Tabulados (múltiples hojas T13/T26, mapeo por índice)")
        elif "PRECIOS" in nombre_upper:
            analisis_partes.append("Trans: Precios (estructura simplificada, eliminadas columnas erróneas)")
        elif "SIPA" in nombre_upper and "TEMPERATURA" in nombre_upper:
            analisis_partes.append("Trans: SIPA (año entero, mes extraído, sin KNN)")
        elif "USO_DEL_SUELO" in nombre_upper:
            analisis_partes.append("Trans: Uso del Suelo (provincia_id mapeada, eliminadas agregaciones regionales)")
        elif "FAOSTAT" in nombre_upper:
            analisis_partes.append("Trans: FAOSTAT (normalización estándar)")
        else:
            analisis_partes.append("Trans: estándar")
        
        # Agregar resultados de limpieza
        analisis_partes.append(f"Limpieza: {antes}→{despues} registros, −{duplicados} dups")
        
        analisis_agente = " | ".join(analisis_partes)
        
        # Determinar status
        status_final = "OK"
        if calidad_t < 90:
            status_final = "WARNING"  # Calidad baja
        if despues == 0:
            status_final = "ERROR"  # Sin registros válidos
        
        # Llamadas LLM (heredadas del estado de mapeo)
        llamadas_llm = state.get("llamadas_api", 0)
        
        schema_t = StructType([
            StructField("execution_id",      StringType(),  True),  # 🆔 CORRECCIÓN #21
            StructField("file_name",         StringType(),  True),
            StructField("fuente",            StringType(),  True),
            StructField("framework",         StringType(),  True),  # ⭐ CORRECCIÓN #20
            StructField("tabla_destino",     StringType(),  True),
            StructField("filas_entrada",     IntegerType(), True),
            StructField("filas_salida",      IntegerType(), True),
            StructField("duplicados_elim",   IntegerType(), True),
            StructField("calidad_pct",       DoubleType(),  True),
            StructField("tasa_error_transformacion",DoubleType(), True),
            StructField("tiempo_segundos",   DoubleType(),  True),
            StructField("timestamp_inicio",  TimestampType(),True),
            StructField("timestamp_fin",     TimestampType(),True),
            StructField("analisis_agente",   StringType(),  True),  # NUEVO
            StructField("llamadas_llm",      IntegerType(), True),  # NUEVO
            StructField("status",            StringType(),  True),  # NUEVO
        ])
        spark.createDataFrame([(
            EXECUTION_ID,  # 🆔 CORRECCIÓN #21
            state["file_name"], state["fuente"], FRAMEWORK_NAME,  # ⭐ CORRECCIÓN #20
            state.get("tabla_destino","?"),
            total, despues, duplicados, calidad_t, tasa_error_t, tiempo_t, ts_ini_t, ts_fin_t,
            analisis_agente[:1000], llamadas_llm, status_final
        )], schema_t).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_TRANSFORM)

        print(f"  M2={calidad_t:.1f}%  Tasa_Error_T={tasa_error_t:.1f}  Tiempo={tiempo_t:.1f}s  → log guardado")
        return {"registros_validos":despues, "registros_duplicados":duplicados,
                "ts_fin_transformacion": ts_fin_t.isoformat(), "error_transform":None,
                "temp_view_name": temp_view_name,
                "llm_response": state.get("llm_response", ""),  # 🔧 CORRECCIÓN: preservar llm_response
                "calidad_pct": calidad_t,  # 🔧 CORRECCIÓN: devolver calidad_pct
                "tasa_error": tasa_error_t}  # 🔧 CORRECCIÓN: devolver tasa_error

    except Exception as e:
        print(f"  ❌ {e}")
        return {"registros_validos":0,"registros_duplicados":0,
                "ts_fin_transformacion": datetime.now().isoformat(), "error_transform":str(e)}


def nodo_carga_con_metricas(state: ETLState) -> dict:
    """
    Versión actualizada del nodo de carga con métricas homogeneizadas.
    Incluye: llamadas_llm (siempre 0), analisis_agente, status.
    CORRECCIÓN: Maneja DataFrames vacíos sin intentar cargarlos (evita PARSE_EMPTY_STATEMENT).
    """
    tabla_completa = f"{DB_NAME}.{state['tabla_destino']}"
    print(f"[NODO 5 — CARGA CON MÉTRICAS HOMOGENEIZADAS] → {tabla_completa}")
    ts_ini_c = datetime.now()
    try:
        temp_view_name = state.get("temp_view_name", "langgraph_etl_temp")
        
        # ❌ CORRECCIÓN CRÍTICA: Si temp_view_name está vacío, significa que la transformación falló
        if not temp_view_name or temp_view_name == "":
            print(f"  ⚠️  No hay temp view (transformación previa falló) - Saltando carga")
            ts_fin_c = datetime.now()
            tiempo_c = (ts_fin_c - ts_ini_c).total_seconds()
            
            analisis_carga = f"Carga omitida: transformación previa falló (no hay temp view) | Archivo: {state.get('file_name', 'N/A')}"
            
            schema_c = StructType([
                StructField("execution_id",      StringType(),  True),
                StructField("file_name",         StringType(),  True),
                StructField("fuente",            StringType(),  True),
                StructField("framework",         StringType(),  True),
                StructField("tabla_destino",     StringType(),  True),
                StructField("registros_insertados",IntegerType(),True),
                StructField("tiempo_escritura_s",DoubleType(),  True),
                StructField("timestamp_inicio",  TimestampType(),True),
                StructField("timestamp_fin",     TimestampType(),True),
                StructField("analisis_agente",   StringType(),  True),
                StructField("llamadas_llm",      IntegerType(), True),
                StructField("status",            StringType(),  True),
            ])
            spark.createDataFrame([(
                EXECUTION_ID,
                state["file_name"], state["fuente"], FRAMEWORK_NAME,
                state.get("tabla_destino","?"),
                0, tiempo_c, ts_ini_c, ts_fin_c,
                analisis_carga[:1000], 0, "ERROR"
            )], schema_c).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_CARGA)
            
            return {"registros_cargados": 0, "ts_fin_carga": ts_fin_c.isoformat(), "error_carga": "No hay temp view - transformación falló"}
        
        df_spark = spark.table(temp_view_name)
        
        registros_a_insertar = df_spark.count()
        
        # ❌ CORRECCIÓN CRITICA: Si DataFrame está vacío, NO intentar cargarlo
        if registros_a_insertar == 0:
            print(f"  ⚠️  DataFrame vacío - Saltando carga (no hay datos para escribir)")
            ts_fin_c = datetime.now()
            tiempo_c = (ts_fin_c - ts_ini_c).total_seconds()
            
            # Registrar métrica de carga vacía con status WARNING
            analisis_carga = f"Carga omitida: DataFrame vacío (0 registros) | Tabla destino: {state.get('tabla_destino', 'N/A')} | Posible error en transformación"
            
            schema_c = StructType([
                StructField("execution_id",      StringType(),  True),
                StructField("file_name",         StringType(),  True),
                StructField("fuente",            StringType(),  True),
                StructField("framework",         StringType(),  True),
                StructField("tabla_destino",     StringType(),  True),
                StructField("registros_insertados",IntegerType(),True),
                StructField("tiempo_escritura_s",DoubleType(),  True),
                StructField("timestamp_inicio",  TimestampType(),True),
                StructField("timestamp_fin",     TimestampType(),True),
                StructField("analisis_agente",   StringType(),  True),
                StructField("llamadas_llm",      IntegerType(), True),
                StructField("status",            StringType(),  True),
            ])
            spark.createDataFrame([(
                EXECUTION_ID,
                state["file_name"], state["fuente"], FRAMEWORK_NAME,
                state.get("tabla_destino","?"),
                0,  # registros_insertados = 0
                tiempo_c, ts_ini_c, ts_fin_c,
                analisis_carga[:1000], 0, "WARNING"  # status = WARNING
            )], schema_c).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_CARGA)
            
            return {"registros_cargados": 0, "ts_fin_carga": ts_fin_c.isoformat(), "error_carga": None}
        
        # Si hay registros, proceder con carga normal

        if spark.catalog.tableExists(tabla_completa):
            schema_dest = spark.table(tabla_completa).schema
            df_cols = df_spark.columns
            cast_exprs = {}
            for campo in schema_dest.fields:
                if campo.name in df_cols:
                    cast_exprs[campo.name] = col(campo.name).cast(campo.dataType)
            if cast_exprs:
                df_spark = df_spark.withColumns(cast_exprs)

        df_spark.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(tabla_completa)
        ts_fin_c  = datetime.now()
        tiempo_c  = (ts_fin_c - ts_ini_c).total_seconds()

        print(f"  ✅ {registros_a_insertar} registros en {tabla_completa} ({tiempo_c:.1f}s)")

        # ── Métricas de Carga HOMOGENEIZADAS ────────────────────────────────────────────
        analisis_carga = f"Carga exitosa de {registros_a_insertar:,} registros en {tabla_completa} usando Delta Lake con mergeSchema=true | Tabla destino: {state.get('tabla_destino', 'N/A')}"
        
        schema_c = StructType([
            StructField("execution_id",      StringType(),  True),  # 🆔 CORRECCIÓN #21
            StructField("file_name",         StringType(),  True),
            StructField("fuente",            StringType(),  True),
            StructField("framework",         StringType(),  True),  # ⭐ CORRECCIÓN #20
            StructField("tabla_destino",     StringType(),  True),
            StructField("registros_insertados",IntegerType(),True),
            StructField("tiempo_escritura_s",DoubleType(),  True),
            StructField("timestamp_inicio",  TimestampType(),True),
            StructField("timestamp_fin",     TimestampType(),True),
            StructField("analisis_agente",   StringType(),  True),  # NUEVO
            StructField("llamadas_llm",      IntegerType(), True),  # NUEVO (siempre 0 en carga)
            StructField("status",            StringType(),  True),
        ])
        spark.createDataFrame([(
            EXECUTION_ID,  # 🆔 CORRECCIÓN #21
            state["file_name"], state["fuente"], FRAMEWORK_NAME,  # ⭐ CORRECCIÓN #20
            tabla_completa,
            registros_a_insertar, tiempo_c, ts_ini_c, ts_fin_c,
            analisis_carga[:1000], 0, "OK"  # 0 llamadas LLM en carga
        )], schema_c).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_CARGA)

        return {"tabla_completa":tabla_completa, "ts_fin_carga":ts_fin_c.isoformat(), "error_carga":None}

    except Exception as e:
        ts_fin_c = datetime.now()
        print(f"  ❌ {e}")
        
        analisis_error = f"Error en carga: {str(e)[:200]} | Tabla destino: {state.get('tabla_destino', 'N/A')}"
        
        schema_c = StructType([
            StructField("execution_id",      StringType(),  True),  # 🆔 CORRECCIÓN #21
            StructField("file_name",         StringType(),  True),
            StructField("fuente",            StringType(),  True),
            StructField("framework",         StringType(),  True),  # ⭐ CORRECCIÓN #20
            StructField("tabla_destino",     StringType(),  True),
            StructField("registros_insertados",IntegerType(),True),
            StructField("tiempo_escritura_s",DoubleType(),  True),
            StructField("timestamp_inicio",  TimestampType(),True),
            StructField("timestamp_fin",     TimestampType(),True),
            StructField("analisis_agente",   StringType(),  True),
            StructField("llamadas_llm",      IntegerType(), True),
            StructField("status",            StringType(),  True),
        ])
        spark.createDataFrame([(
            EXECUTION_ID,  # 🆔 CORRECCIÓN #21
            state["file_name"], state["fuente"], FRAMEWORK_NAME,  # ⭐ CORRECCIÓN #20
            tabla_completa,
            0, 0.0, ts_ini_c, ts_fin_c,
            analisis_error[:1000], 0, "ERROR"
        )], schema_c).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_CARGA)
        return {"tabla_completa":tabla_completa, "ts_fin_carga":ts_fin_c.isoformat(), "error_carga":str(e)}


print("✅ CORRECCIÓN #19 APLICADA: Métricas homogeneizadas")
print("   Todas las tablas de métricas ahora incluyen:")
print("   - analisis_agente: razonamiento del agente")
print("   - llamadas_llm: número de consultas al LLM")
print("   - status: OK/WARNING/ERROR/PARCIAL")
print("")
print("   NOTA: Para aplicar estos cambios, reemplaza manualmente:")
print("         - nodo_transformacion() por nodo_transformacion_con_metricas()")
print("         - nodo_carga() por nodo_carga_con_metricas()")
print("   O ejecuta el bloque 9 (orquestador) que usará los nodos actualizados.")

In [ ]:
# ── CORRECCIÓN: MAPEO CON VALIDACIÓN DE EXTENSIONES ──────────────────────
def nodo_mapeo_corregido(state: ETLState) -> dict:
    """
    Mapeo híbrido corregido: valida que el LLM NO incluya extensiones de archivo.
    CORRECCIÓN AEBE: Para archivos AEBE, usar SIEMPRE las reglas (no el LLM).
    """
    # Compatibilidad: LangGraph usa 'df_columnas', LlamaIndex usa 'columnas_raw'
    columnas = state.get('df_columnas') or state.get('columnas_raw', [])
    print(f"[NODO 3 — MAPEO HÍBRIDO CORREGIDO] columnas: {columnas[:4]}...")
    # CORRECCIÓN: homogenizar nombres de claves
    llamadas = state.get("llamadas_llm", 0) or state.get("llamadas_api", 0)
    
    # 🔧 HÍBRIDO SEGURO: Intentar LLM para TODOS los archivos (incluyendo AEBE)
    # Si el LLM falla o mapea mal AEBE, usar reglas como fallback
    
    try:
        print(f"  LLM — intento 1/1...")
        
        # Prompt mejorado con AEBE incluido
        prompt_mejorado = f"""Identifica la tabla destino para el archivo: {state['file_name']}
Columnas encontradas: {', '.join(columnas[:8])}

Tablas válidas (elige EXACTAMENTE una):
- espac_banano_platano_provincia (T13/T26, Tabulados, Series Históricas - banano/plátano por provincia UNIFICADO)
- espac_uso_del_suelo (uso del suelo)
- sipa_temperatura_precipitacion (clima)
- faostat_produccion_banano_platano (FAO bananas/plantains UNIFICADO)
- aebe_exportaciones_regiones (AEBE Bananotas - rankings de exportaciones por región/país)
- banano_precios_semanales (precios)

⚠️ SI el archivo tiene "Tabulados" o "Series" en el nombre → SIEMPRE usar espac_banano_platano_provincia
⚠️ SI el archivo tiene "AEBE" o "BANANOTAS" en el nombre → SIEMPRE usar aebe_exportaciones_regiones

⚠️ IMPORTANTE: La tabla_destino debe ser EXACTAMENTE uno de los nombres arriba.
⚠️ NO incluyas extensiones (.xlsx, .csv, .xls) en el nombre.

Responde SOLO con JSON válido (sin markdown):
{{"tabla_destino":"nombre_sin_extension","confianza":0.9,"columnas_double":[],"columnas_integer":[]}}"""
        
        resp = llm.invoke([HumanMessage(content=prompt_mejorado)])
        llamadas += 1
        
        if resp.content and resp.content.strip():
            texto = re.sub(r"^```json\s*|^```\s*|```$","",resp.content.strip(),flags=re.MULTILINE).strip()
            start = texto.find("{")
            end = texto.rfind("}")
            if start != -1 and end != -1:
                texto = texto[start:end+1]
                mapeo = json.loads(texto)
                tabla = mapeo.get("tabla_destino","").lower().replace(" ","_")
                
                # ✅ VALIDACIÓN: Eliminar extensiones si el LLM las incluyó
                for ext in [".xlsx", ".xls", ".csv", ".json"]:
                    if tabla.endswith(ext):
                        print(f"  ⚠️ LLM incluyó extensión '{ext}' — removiendo...")
                        tabla = tabla[:-len(ext)]
                
                if tabla and tabla != "tabla_temporal":
                    # 🛡️ VALIDACIÓN SEGURIDAD: Si es archivo AEBE, verificar que el LLM mapeó correctamente
                    if "AEBE" in state["file_name"].upper():
                        if tabla == "aebe_exportaciones_regiones":
                            print(f"  ✅ LLM correcto para AEBE: '{tabla}' (confianza: {mapeo.get('confianza',0)})")
                            return {"tabla_destino":tabla, "cols_double":mapeo.get("columnas_double",[]),
                                    "cols_integer":mapeo.get("columnas_integer",[]),
                                    "llm_response":json.dumps(mapeo,ensure_ascii=False),
                                    "llamadas_llm":llamadas, "llamadas_api":llamadas, "error_mapeo":None}
                        else:
                            print(f"  ⚠️ LLM INCORRECTO para AEBE: mapeó a '{tabla}' (debería ser 'aebe_exportaciones_regiones')")
                            print(f"  🔧 Usando reglas como fallback de seguridad...")
                            # Continuar al fallback de reglas (no return aquí)
                    else:
                        # Para archivos NO-AEBE, confiar en el LLM
                        print(f"  ✅ LLM: '{tabla}' (confianza: {mapeo.get('confianza',0)})")
                        return {"tabla_destino":tabla, "cols_double":mapeo.get("columnas_double",[]),
                                "cols_integer":mapeo.get("columnas_integer",[]),
                                "llm_response":json.dumps(mapeo,ensure_ascii=False),
                                "llamadas_llm":llamadas, "llamadas_api":llamadas, "error_mapeo":None}
    except Exception as e:
        print(f"  ⚠️ LLM falló: {str(e)[:50]}...")
    
    # Fallback a reglas
    print(f"  🔧 Usando mapeo basado en reglas...")
    fallback = _mapeo_basado_en_reglas(state["file_name"], columnas)
    print(f"  ✅ Regla: '{fallback['tabla']}' (confianza: {fallback['conf']})")
    
    # Distinguir entre fallback normal y fallback por validación AEBE
    es_fallback_aebe = "AEBE" in state["file_name"].upper() and llamadas > 0
    metodo_fallback = "validacion_aebe_fallback" if es_fallback_aebe else "fallback_reglas"
    
    return {"tabla_destino":fallback["tabla"], 
            "cols_double":fallback["double"],
            "cols_integer":fallback["int"],
            "llm_response":json.dumps({"metodo":metodo_fallback,"confianza":fallback["conf"],"llm_intento":llamadas>0},ensure_ascii=False),
            "llamadas_llm":llamadas, "llamadas_api":llamadas, 
            "error_mapeo":None}

# Reemplazar el nodo en la definición del grafo
nodo_mapeo = nodo_mapeo_corregido

print("✅ Nodo de mapeo corregido: ahora elimina extensiones automáticamente")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# HELPER: MAPEO BASADO EN REGLAS (fallback del LLM)
# ══════════════════════════════════════════════════════════════════════════
def _mapeo_basado_en_reglas(file_name: str, columnas: List[str]) -> dict:
    """
    Fallback basado en reglas cuando el LLM falla.
    Usa el nombre del archivo y columnas para determinar la tabla destino.
    """
    fn = file_name.upper()
    cols_text = " ".join(columnas).lower()
    
    # Reglas por nombre de archivo
    # PRIORIDAD MÁXIMA: ESPAC Tabulados y Series históricas → TABLA UNIFICADA banano/plátano
    # (debe estar ANTES de otras reglas para evitar falsos positivos)
    if "TABULADOS_EXCEL" in fn or "TABULADOS" in fn or "SERIES_HISTORICAS" in fn or "SERIES_HIST" in fn:
        return {"tabla":"espac_banano_platano_provincia", "conf":0.95, "double":["superficie_plantada_ha","superficie_cosechada_ha","produccion_tm","ventas_tm","rendimiento"], "int":["anio"]}
    
    # ESPAC T13 (banano) y T26 (plátano) → TABLA UNIFICADA
    if "T13" in fn or "T26" in fn:
        return {"tabla":"espac_banano_platano_provincia", "conf":0.95, "double":["superficie_plantada_ha","superficie_cosechada_ha","produccion_tm","ventas_tm","rendimiento"], "int":["anio"]}
    
    if "USO_DEL_SUELO" in fn or "uso" in cols_text and "suelo" in cols_text:
        return {"tabla":"espac_uso_del_suelo", "conf":0.85, "double":["superficie_ha"], "int":["ano"]}
    if "TEMPERATURA" in fn or ("temperatura" in cols_text and "precipitacion" in cols_text):
        return {"tabla":"sipa_temperatura_precipitacion", "conf":0.95, "double":["precipitacion_mm","temperatura_promedio_c"], "int":["ano"]}
    
    # FAOSTAT bananas y plantains → TABLA UNIFICADA
    if "FAOSTAT" in fn and ("BANANAS" in fn or "PLANTAINS" in fn):
        return {"tabla":"faostat_produccion_banano_platano", "conf":0.95, "double":["value"], "int":["year","area_code","item_code"]}
    
    # AEBE Bananotas → rankings normalizados (UN registro por entidad para el año detectado)
    if "AEBE_BANANOTAS" in fn or ("AEBE" in fn and "BANANOTAS" in fn):
        return {"tabla":"aebe_exportaciones_regiones", "conf":0.95, "double":["cantidad"], "int":["anio"]}
    
    # Fallback genérico
    return {"tabla":"tabla_temporal", "conf":0.3, "double":[], "int":[]}

# ==============================================================================
# AGENTE 2: ETL WORKFLOW (LlamaIndex)
# Patron "bridge": cada @step construye un state dict temporal, llama al nodo
# LangGraph original, y convierte el resultado en el Event de salida.
# La logica de transformacion es IDENTICA a LangGraph - cero duplicacion.
# ==============================================================================

class ETLWorkflow(Workflow):

    @step
    async def iniciar(self, ctx: Context, ev: StartEvent) -> ETLStartEvent:
        return ETLStartEvent(
            nombre_archivo   = ev.get("nombre_archivo"),
            ruta_archivo     = ev.get("ruta_archivo"),
            fuente           = ev.get("fuente"),
            timestamp_inicio = datetime.now().isoformat(),
        )

    # PASO 1: Deteccion (= nodo_deteccion en LangGraph)
    @step
    async def detectar_archivo(self, ctx: Context, ev: ETLStartEvent) -> ArchivoDetectadoEvent | ETLErrorEvent:
        print(f"\n[AGENTE 2 PASO 1 - DETECTAR] {ev.nombre_archivo}")
        state = {"file_name": ev.nombre_archivo, "local_path": ev.ruta_archivo,
                 "fuente": ev.fuente, "llamadas_llm": 0}
        state = nodo_deteccion(state)
        if state.get("error_deteccion"):
            return ETLErrorEvent(nombre_archivo=ev.nombre_archivo, fuente=ev.fuente,
                causa=state["error_deteccion"], timestamp_inicio=ev.timestamp_inicio, llamadas_llm=0)
        return ArchivoDetectadoEvent(
            nombre_archivo=ev.nombre_archivo, ruta_archivo=ev.ruta_archivo,
            fuente=state.get("fuente", ev.fuente), extension=state.get("extension",""),
            timestamp_inicio=ev.timestamp_inicio, llamadas_llm=0)

    # PASO 2: Lectura (= nodo_lectura en LangGraph)
    @step
    async def leer_archivo_step(self, ctx: Context, ev: ArchivoDetectadoEvent) -> ArchivoLeidoEvent | ETLErrorEvent:
        print(f"\n[AGENTE 2 PASO 2 - LEER] {ev.nombre_archivo}")
        state = {"file_name": ev.nombre_archivo, "local_path": ev.ruta_archivo,
                 "fuente": ev.fuente, "extension": ev.extension, "llamadas_llm": 0}
        state = nodo_lectura(state)
        if state.get("error_lectura"):
            return ETLErrorEvent(nombre_archivo=ev.nombre_archivo, fuente=ev.fuente,
                causa=state["error_lectura"], timestamp_inicio=ev.timestamp_inicio, llamadas_llm=0)
        return ArchivoLeidoEvent(
            nombre_archivo=ev.nombre_archivo, ruta_archivo=ev.ruta_archivo,
            fuente=ev.fuente, extension=ev.extension,
            columnas_raw=state.get("df_columnas",[]),  # nodo_lectura devuelve df_columnas
            n_registros_raw=state.get("total_filas",0),  # nodo_lectura devuelve total_filas
            timestamp_inicio=ev.timestamp_inicio, llamadas_llm=0)

    # PASO 3: Mapeo con LLM (= nodo_mapeo_corregido en LangGraph)
    @step
    async def mapear_columnas(self, ctx: Context, ev: ArchivoLeidoEvent) -> ColumnasMapeadasEvent | ETLErrorEvent:
        print(f"\n[AGENTE 2 PASO 3 - MAPEO LLM] {ev.nombre_archivo}")
        state = {"file_name": ev.nombre_archivo, "local_path": ev.ruta_archivo,
                 "fuente": ev.fuente, "extension": ev.extension,
                 "df_columnas": ev.columnas_raw, "total_filas": ev.n_registros_raw,  # Usar nombres LangGraph
                 "llamadas_llm": ev.llamadas_llm, "llamadas_api": ev.llamadas_llm, "llm_response": ""}
        state = nodo_mapeo_corregido(state)
        if state.get("error_mapeo"):
            return ETLErrorEvent(nombre_archivo=ev.nombre_archivo, fuente=ev.fuente,
                causa=state["error_mapeo"], timestamp_inicio=ev.timestamp_inicio,
                llamadas_llm=state.get("llamadas_llm",0))
        return ColumnasMapeadasEvent(
            nombre_archivo=ev.nombre_archivo, ruta_archivo=ev.ruta_archivo,
            fuente=ev.fuente,
            tabla_destino=state.get("tabla_destino","tabla_temporal"),
            cols_double=state.get("cols_double",[]),
            cols_integer=state.get("cols_integer",[]),
            razonamiento_llm=state.get("llm_response","")[:500],
            columnas_raw=ev.columnas_raw, n_registros_raw=ev.n_registros_raw,
            timestamp_inicio=ev.timestamp_inicio,
            llamadas_llm=state.get("llamadas_llm",0))

    # PASO 4: Transformacion (= nodo_transformacion_con_metricas en LangGraph)
    @step
    async def transformar_datos(self, ctx: Context, ev: ColumnasMapeadasEvent) -> DatosTransformadosEvent | ETLErrorEvent:
        print(f"\n[AGENTE 2 PASO 4 - TRANSFORMAR] {ev.nombre_archivo}")
        state = {
            "file_name":      ev.nombre_archivo,
            "local_path":     ev.ruta_archivo,
            "fuente":         ev.fuente,
            "tabla_destino":  ev.tabla_destino,
            "cols_double":    ev.cols_double,
            "cols_integer":   ev.cols_integer,
            "df_columnas":    ev.columnas_raw,  # Usar nombre LangGraph
            "total_filas":    ev.n_registros_raw,  # Usar nombre LangGraph
            "llm_response":   ev.razonamiento_llm,
            "llamadas_llm":   ev.llamadas_llm,
            "llamadas_api":   ev.llamadas_llm,  # Agregar alias
            "timestamp_inicio": ev.timestamp_inicio,
        }
        state = nodo_transformacion_con_metricas(state)
        if state.get("error_transformacion"):
            return ETLErrorEvent(nombre_archivo=ev.nombre_archivo, fuente=ev.fuente,
                causa=state["error_transformacion"], timestamp_inicio=ev.timestamp_inicio,
                llamadas_llm=state.get("llamadas_llm",0))
        return DatosTransformadosEvent(
            nombre_archivo=ev.nombre_archivo, fuente=ev.fuente,
            tabla_destino=state.get("tabla_destino", ev.tabla_destino),
            ruta_archivo=ev.ruta_archivo,
            n_registros=state.get("registros_validos",0),  # CORRECCIÓN: usar clave LangGraph
            registros_dup=state.get("registros_duplicados",0),  # CORRECCIÓN: usar clave LangGraph
            calidad_pct=state.get("calidad_pct",0.0),
            tasa_error=state.get("tasa_error",0.0),
            columnas_raw=ev.columnas_raw,
            temp_view_name=state.get("temp_view_name",""),
            razonamiento_llm=state.get("llm_response","")[:500],
            timestamp_inicio=ev.timestamp_inicio,
            llamadas_llm=state.get("llamadas_llm",0))

    # PASO 5: Carga Delta (= nodo_carga_con_metricas en LangGraph)
    @step
    async def cargar_delta(self, ctx: Context, ev: DatosTransformadosEvent) -> StopEvent | ETLErrorEvent:
        print(f"\n[AGENTE 2 PASO 5 - CARGAR] {ev.nombre_archivo}")
        state = {
            "file_name":       ev.nombre_archivo,
            "fuente":          ev.fuente,
            "tabla_destino":   ev.tabla_destino,
            "registros_validos": ev.n_registros,  # Usar nombre LangGraph
            "registros_duplicados": ev.registros_dup,  # Usar nombre LangGraph
            "total_filas":     ev.n_registros,  # Agregar para compatibilidad
            "df_columnas":     ev.columnas_raw,  # Agregar para compatibilidad
            "calidad_pct":     ev.calidad_pct,
            "tasa_error":            ev.tasa_error,
            "temp_view_name":  ev.temp_view_name,
            "llm_response":    ev.razonamiento_llm,
            "llamadas_llm":    ev.llamadas_llm,
            "llamadas_api":    ev.llamadas_llm,  # Agregar alias
            "inicio_ts":       ev.timestamp_inicio,
            "timestamp_inicio":ev.timestamp_inicio,
        }
        state = nodo_carga_con_metricas(state)
        if state.get("error_carga"):
            return ETLErrorEvent(nombre_archivo=ev.nombre_archivo, fuente=ev.fuente,
                causa=state["error_carga"], timestamp_inicio=ev.timestamp_inicio,
                llamadas_llm=state.get("llamadas_llm",0))
        
        # ══════════════════════════════════════════════════════════════════════
        # REGISTRAR MÉTRICAS CONSOLIDADAS M1-M6 EN control_logs_etl
        # ══════════════════════════════════════════════════════════════════════
        try:
            ts_ini = datetime.fromisoformat(ev.timestamp_inicio)
            ts_fin = datetime.now()
            tiempo_total = (ts_fin - ts_ini).total_seconds()
            
            # Calcular métricas M1-M6
            # Las métricas M4-M6 vienen del nodo de transformación y fueron guardadas en metricas_transformacion
            # Necesitamos leer la última entrada de esa tabla para este archivo
            m1_tiempo = tiempo_total
            m2_intervencion = 0  # Sin intervención manual
            m3_recuperacion = 1 if ev.n_registros > 0 else 0  # 1 si procesó datos
            
            # Leer M4-M6 desde metricas_transformacion
            try:
                metricas_t = spark.sql(f"""
                    SELECT calidad_pct, tasa_error_transformacion, llamadas_llm
                    FROM {LOG_TRANSFORM}
                    WHERE file_name = '{ev.nombre_archivo}' 
                      AND framework = '{FRAMEWORK_NAME}'
                    ORDER BY timestamp_fin DESC
                    LIMIT 1
                """).first()
                m4_calidad = metricas_t['calidad_pct'] if metricas_t else ev.calidad_pct
                m5_llamadas = metricas_t['llamadas_llm'] if metricas_t else ev.llamadas_llm
                m5_tasa_error = metricas_t['tasa_error_transformacion'] if metricas_t else ev.tasa_error
            except:
                # Fallback a valores del evento si la consulta falla
                m4_calidad = ev.calidad_pct
                m5_llamadas = ev.llamadas_llm
                m5_tasa_error = ev.tasa_error
            
            schema_log = StructType([
                StructField("execution_id",            StringType(),   True),
                StructField("file_name",               StringType(),   True),
                StructField("framework",               StringType(),   True),
                StructField("fuente",                  StringType(),   True),
                StructField("tabla_destino",           StringType(),   True),
                StructField("execution_timestamp",     TimestampType(), True),
                StructField("status",                  StringType(),   True),
                StructField("m1_tiempo_segundos",      DoubleType(),   True),
                StructField("m2_calidad_pct",          DoubleType(),   True),
                StructField("m3_llamadas_api",         IntegerType(),  True),
                StructField("m5_tasa_error",                 DoubleType(),   True),
                StructField("total_filas",             IntegerType(),  True),
                StructField("registros_validos",       IntegerType(),  True),
                StructField("registros_duplicados",    IntegerType(),  True),
                StructField("llm_response",            StringType(),   True),
                StructField("error_message",           StringType(),   True),
            ])
            spark.createDataFrame([(
                EXECUTION_ID, ev.nombre_archivo, FRAMEWORK_NAME, ev.fuente,
                ev.tabla_destino, ts_fin, "PROCESADO",
                m1_tiempo, m4_calidad, m5_llamadas, m5_tasa_error,
                ev.columnas_raw.__len__() * ev.n_registros,  # total_filas estimado
                ev.n_registros, ev.registros_dup,
                ev.razonamiento_llm[:500], None
            )], schema_log).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_TABLE)
            print(f"  📊 Métricas M1-M6 guardadas en control_logs_etl")
        except Exception as e:
            print(f"  ⚠️  Error guardando métricas consolidadas: {e}")
        
        return StopEvent(result={
            "status":          "PROCESADO",
            "archivo":         ev.nombre_archivo,
            "tabla":           ev.tabla_destino,
            "registros":       ev.n_registros,
            "calidad_pct":     ev.calidad_pct,
            "tasa_error":            ev.tasa_error,
            "tiempo_segundos": tiempo_total,
            "llamadas_api":    ev.llamadas_llm,
        })

    @step
    async def manejar_error(self, ctx: Context, ev: ETLErrorEvent) -> StopEvent:
        print(f"\nERROR ETL {ev.nombre_archivo}: {ev.causa}")
        # Registrar error en LOG_TABLE
        try:
            schema_err = StructType([
                StructField("execution_id",            StringType(),   True),
                StructField("file_name",               StringType(),   True),
                StructField("framework",               StringType(),   True),
                StructField("fuente",                  StringType(),   True),
                StructField("tabla_destino",           StringType(),   True),
                StructField("execution_timestamp",     TimestampType(), True),
                StructField("status",                  StringType(),   True),
                StructField("m1_tiempo_segundos",      DoubleType(),   True),
                StructField("m2_calidad_pct",          DoubleType(),   True),
                StructField("m3_llamadas_api",         IntegerType(),  True),
                StructField("m5_tasa_error",                 DoubleType(),   True),
                StructField("total_filas",             IntegerType(),  True),
                StructField("registros_validos",       IntegerType(),  True),
                StructField("registros_duplicados",    IntegerType(),  True),
                StructField("llm_response",            StringType(),   True),
                StructField("error_message",           StringType(),   True),
            ])
            spark.createDataFrame([(
                EXECUTION_ID, ev.nombre_archivo, FRAMEWORK_NAME, ev.fuente,
                "N/A", datetime.now(), "ERROR",
                0.0, 0.0, ev.llamadas_llm, 0.0,
                0, 0, 0, "{}", ev.causa[:500]
            )], schema_err).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(LOG_TABLE)
        except Exception as e:
            print(f"  No se pudo registrar error: {e}")
        return StopEvent(result={"status":"ERROR","archivo":ev.nombre_archivo,"causa":ev.causa})

print("OK ETLWorkflow (AGENTE 2) definido - patron bridge sobre nodos LangGraph.")
print("   Pasos: iniciar -> detectar -> leer -> mapear -> transformar -> cargar -> END")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# HELPER: DECISIÓN KNN CON LLM
# ══════════════════════════════════════════════════════════════════════════

def decidir_aplicar_knn_con_llm(df_pandas: pd.DataFrame, nombre_archivo: str, fuente: str) -> dict:
    """
    Consulta al LLM para decidir si es apropiado aplicar imputación KNN.
    
    Returns:
        dict con:
        - aplicar_knn: bool (True/False)
        - razon: str (explicación del LLM)
        - columnas_excluir: list (columnas que NO deben imputarse)
    """
    # Analizar valores faltantes
    numeric_cols = df_pandas.select_dtypes(include=[np.number]).columns.tolist()
    missing_info = {}
    
    for col in numeric_cols[:10]:  # Máximo 10 columnas para no saturar el prompt
        missing_count = df_pandas[col].isnull().sum()
        if missing_count > 0:
            missing_pct = (missing_count / len(df_pandas)) * 100
            missing_info[col] = f"{missing_pct:.1f}%"
    
    if not missing_info:
        return {"aplicar_knn": False, "razon": "No hay valores faltantes", "columnas_excluir": []}
    
    # Prompt para el LLM
    prompt = f"""Analiza si es apropiado usar imputación KNN (vecinos cercanos) para este dataset:

Archivo: {nombre_archivo}
Fuente: {fuente}
Total registros: {len(df_pandas)}

Valores faltantes por columna:
{json.dumps(missing_info, indent=2, ensure_ascii=False)}

Contexto:
- KNN rellena valores numéricos faltantes usando los 5 vecinos más cercanos
- Es apropiado cuando los valores faltantes son ERRORES de captura o medición
- NO es apropiado cuando los valores NULL tienen significado estructural (ej: provincia NULL = agregación regional)

Decide:
1. ¿Aplicar KNN? (true/false)
2. ¿Por qué?
3. ¿Qué columnas EXCLUIR del KNN? (claves primarias, IDs, campos estructuralmente NULL)

Responde en JSON:
{{
  "aplicar_knn": true/false,
  "razon": "explicación breve",
  "columnas_excluir": ["provincia_id", "anio", ...]
}}"""
    
    try:
        print(f"    🤔 Consultando al LLM sobre uso de KNN...")
        resp = llm.invoke([HumanMessage(content=prompt)])
        
        # Extraer JSON de la respuesta
        texto = re.sub(r"^```json\s*|^```\s*|```$", "", resp.content.strip(), flags=re.MULTILINE).strip()
        start = texto.find("{")
        end = texto.rfind("}")
        
        if start != -1 and end != -1:
            texto = texto[start:end+1]
            decision = json.loads(texto)
            
            aplicar = decision.get("aplicar_knn", False)
            razon = decision.get("razon", "Sin razón")
            excluir = decision.get("columnas_excluir", [])
            
            # 🚫 FORZAR EXCLUSIÓN de columnas que NUNCA deben imputarse
            columnas_protegidas = ['provincia_id', 'region_id', 'anio', 'ano', 'year', 'mes', 'month', 'canton_id', 'pais_codigo', 'producto_codigo']
            excluir = list(set(excluir + columnas_protegidas))
            
            print(f"    🧠 LLM decisión: {'✅ Aplicar KNN' if aplicar else '❌ No aplicar KNN'}")
            print(f"    📝 Razón: {razon}")
            if excluir:
                print(f"    🚫 Excluir: {excluir}")
            
            return {"aplicar_knn": aplicar, "razon": razon, "columnas_excluir": excluir}
    
    except Exception as e:
        print(f"    ⚠️  Error consultando LLM: {str(e)[:50]}")
        print(f"    🔧 Fallback: No aplicar KNN por seguridad")
    
    # Fallback conservador: NO aplicar KNN si el LLM falla
    return {"aplicar_knn": False, "razon": "Error en LLM - decisión conservadora", "columnas_excluir": []}

print("✅ Función decidir_aplicar_knn_con_llm() cargada")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 🧪 TEST: Procesar UN archivo AEBE para verificar transformación
# ══════════════════════════════════════════════════════════════════════

import asyncio
import nest_asyncio
nest_asyncio.apply()

async def test_aebe_un_archivo():
    """
    Procesa SOLO un archivo AEBE para verificar la transformación.
    """
    RAW_PATH_AEBE = "/Volumes/workspace/default/raw_aebe"
    
    # Listar archivos AEBE
    archivos = [f for f in os.listdir(RAW_PATH_AEBE) if f.endswith('.pdf')]
    
    if not archivos:
        print("❌ No se encontraron archivos PDF en", RAW_PATH_AEBE)
        return None
    
    # Tomar el primero
    archivo_test = archivos[0]
    ruta_test = f"{RAW_PATH_AEBE}/{archivo_test}"
    
    print(f"\n📝 TEST: Procesando {archivo_test}")
    print(f"="*70)
    
    try:
        wf = ETLWorkflow(timeout=180, verbose=False)  # 3 minutos timeout
        resultado = await wf.run(
            nombre_archivo = archivo_test,
            ruta_archivo   = ruta_test,
            fuente         = "AEBE_BANANOTAS",
        )
        
        print(f"\n✅ Test completado")
        print(f"   Status: {resultado.get('status', '?')}")
        print(f"   Registros: {resultado.get('registros', 0)}")
        
        # Consultar tabla para verificar
        print(f"\n🔍 Verificando tabla aebe_exportaciones_regiones:")
        df_result = spark.sql(f"""
            SELECT tipo, dato, cantidad, medida, anio, framework, fuente, archivo_origen
            FROM {DB_NAME}.aebe_exportaciones_regiones
            WHERE archivo_origen = '{archivo_test}'
            ORDER BY tipo, cantidad DESC
            LIMIT 10
        """)
        
        display(df_result)
        
        return resultado
        
    except Exception as e:
        print(f"\n❌ Error en test: {e}")
        import traceback
        traceback.print_exc()
        return None

# Ejecutar test
resultado_test = asyncio.run(test_aebe_un_archivo())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ORQUESTADOR AGENTE 2: PROCESAR 4 VOLÚMENES (ESPAC, SIPA, FAOSTAT, AEBE)
# ══════════════════════════════════════════════════════════════════════════

import asyncio

async def ejecutar_agente2_etl_solo_aebe():
    """Procesa SOLO archivos AEBE (versión rápida para testing)."""
    volumenes = [
        (RAW_PATH_AEBE,    "AEBE_BANANOTAS"),
    ]

    resultados_etl = []
    ts_global      = datetime.now()
    total_archivos = 0

    for vol_path, fuente in volumenes:
        print(f"\n{'='*70}")
        print(f"AGENTE 2 ETL - Volumen: {vol_path}  |  Fuente: {fuente}")
        print(f"{'='*70}")
        try:
            archivos = [f for f in os.listdir(vol_path)
                        if not f.startswith('.') and
                        any(f.lower().endswith(ext) for ext in ['.xlsx','.xls','.csv','.json','.pdf'])]
            print(f"  Archivos encontrados: {len(archivos)}")
            for nombre in sorted(archivos):
                ruta = f"{vol_path}/{nombre}"
                total_archivos += 1
                print(f"\n  Procesando ({total_archivos}): {nombre}")
                try:
                    wf = ETLWorkflow(timeout=300, verbose=False)  # 5 min por archivo
                    resultado = await wf.run(
                        nombre_archivo = nombre,
                        ruta_archivo   = ruta,
                        fuente         = fuente,
                    )
                    resultados_etl.append(resultado)
                    st = resultado.get('status','?') if resultado else 'ERROR'
                    nr = resultado.get('registros',0) if resultado else 0
                    print(f"  => {st} | {nr:,} registros")
                except Exception as e:
                    print(f"  ERROR en {nombre}: {e}")
                    resultados_etl.append({"status":"ERROR","archivo":nombre,"causa":str(e)})
        except Exception as e:
            print(f"  ERROR accediendo a {vol_path}: {e}")

    tiempo_total = (datetime.now() - ts_global).total_seconds()
    ok    = sum(1 for r in resultados_etl if r and r.get('status') == 'PROCESADO')
    err   = sum(1 for r in resultados_etl if r and r.get('status') == 'ERROR')
    recs  = sum(r.get('registros',0) for r in resultados_etl if r and r.get('status')=='PROCESADO')
    print(f"\n{'='*70}")
    print(f"ETL AEBE COMPLETADO en {tiempo_total:.1f}s")
    print(f"  Archivos OK: {ok} | Errores: {err} | Total registros: {recs:,}")
    print(f"{'='*70}")
    return resultados_etl

async def ejecutar_agente2_etl():
    """Recorre los 4 volumenes y procesa cada archivo con ETLWorkflow."""
    volumenes = [
        (RAW_PATH_ESPAC,   "INEC_ESPAC"),
        (RAW_PATH_SIPA,    "SIPA_MAG"),
        (RAW_PATH_FAOSTAT, "FAOSTAT"),
        (RAW_PATH_AEBE,    "AEBE_BANANOTAS"),
    ]

    resultados_etl = []
    ts_global      = datetime.now()
    total_archivos = 0

    for vol_path, fuente in volumenes:
        print(f"\n{'='*70}")
        print(f"AGENTE 2 ETL - Volumen: {vol_path}  |  Fuente: {fuente}")
        print(f"{'='*70}")
        try:
            archivos = [f for f in os.listdir(vol_path)
                        if not f.startswith('.') and
                        any(f.lower().endswith(ext) for ext in ['.xlsx','.xls','.csv','.json','.pdf'])]
            print(f"  Archivos encontrados: {len(archivos)}")
            for nombre in sorted(archivos):
                ruta = f"{vol_path}/{nombre}"
                total_archivos += 1
                print(f"\n  Procesando ({total_archivos}): {nombre}")
                try:
                    wf = ETLWorkflow(timeout=900, verbose=False)
                    resultado = await wf.run(
                        nombre_archivo = nombre,
                        ruta_archivo   = ruta,
                        fuente         = fuente,
                    )
                    resultados_etl.append(resultado)
                    st = resultado.get('status','?') if resultado else 'ERROR'
                    nr = resultado.get('registros',0) if resultado else 0
                    print(f"  => {st} | {nr:,} registros")
                except Exception as e:
                    print(f"  ERROR en {nombre}: {e}")
                    resultados_etl.append({"status":"ERROR","archivo":nombre,"causa":str(e)})
        except Exception as e:
            print(f"  ERROR accediendo a {vol_path}: {e}")

    tiempo_total = (datetime.now() - ts_global).total_seconds()
    ok    = sum(1 for r in resultados_etl if r and r.get('status') == 'PROCESADO')
    err   = sum(1 for r in resultados_etl if r and r.get('status') == 'ERROR')
    recs  = sum(r.get('registros',0) for r in resultados_etl if r and r.get('status')=='PROCESADO')
    print(f"\n{'='*70}")
    print(f"ETL COMPLETADO en {tiempo_total:.1f}s")
    print(f"  Archivos OK: {ok} | Errores: {err} | Total registros: {recs:,}")
    print(f"{'='*70}")
    return resultados_etl

# Ejecutar
import nest_asyncio
nest_asyncio.apply()
resultados_etl = asyncio.run(ejecutar_agente2_etl())


## 📊 Bloque 7 — Métricas y Consultas

Consultas de los resultados del pipeline. Identico al LangGraph — mismas tablas Delta.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
print("📥 MÉTRICAS DE EXTRACCIÓN — por fuente:")
spark.table(LOG_EXTRACCION).orderBy("timestamp_fin", ascending=False).display()

print("\n📊 RESUMEN EXTRACCIÓN:")
spark.sql(f"""
SELECT
    fuente,
    archivos_nuevos,
    archivos_omitidos,
    archivos_error,
    ROUND(kb_descargados/1024,2)  AS mb_descargados,
    ROUND(tiempo_segundos,1)      AS tiempo_s
FROM {LOG_EXTRACCION}
ORDER BY fuente
""").display()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
print("🔧 MÉTRICAS DE TRANSFORMACIÓN — por archivo:")
spark.table(LOG_TRANSFORM).orderBy("timestamp_fin", ascending=False).display()

print("\n📊 RESUMEN TRANSFORMACIÓN por fuente:")
spark.sql(f"""
SELECT
    fuente,
    COUNT(*)                            AS archivos,
    SUM(filas_entrada)                  AS total_filas_entrada,
    SUM(filas_salida)                   AS total_filas_validas,
    SUM(duplicados_elim)                AS total_duplicados,
    ROUND(AVG(calidad_pct),2)           AS calidad_promedio_pct,
    ROUND(AVG(tasa_error_transformacion),1)   AS tasa_error_promedio,
    ROUND(SUM(tiempo_segundos),1)       AS tiempo_total_s
FROM {LOG_TRANSFORM}
GROUP BY fuente
ORDER BY fuente
""").display()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
print("💾 MÉTRICAS DE CARGA — por archivo:")
spark.table(LOG_CARGA).orderBy("timestamp_fin", ascending=False).display()

print("\n📊 RESUMEN CARGA por fuente:")
spark.sql(f"""
SELECT
    fuente,
    COUNT(*)                          AS archivos,
    SUM(registros_insertados)         AS total_insertados,
    ROUND(AVG(tiempo_escritura_s),2)  AS tiempo_escritura_promedio_s,
    SUM(CASE WHEN status='OK' THEN 1 ELSE 0 END) AS exitosos,
    SUM(CASE WHEN status!='OK' THEN 1 ELSE 0 END) AS con_error
FROM {LOG_CARGA}
GROUP BY fuente
ORDER BY fuente
""").display()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
print("📊 MÉTRICAS M1–M6 — RESULTADOS COMPLETOS:")
spark.table(LOG_TABLE).orderBy("execution_timestamp", ascending=False).display()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
print("\n📈 RESUMEN POR FUENTE:")
spark.sql(f"""
SELECT
    fuente,
    COUNT(*)                              AS archivos,
    ROUND(AVG(m1_tiempo_segundos),1)      AS M1_tiempo_prom_s,
    ROUND(AVG(m2_calidad_pct),2)          AS M2_calidad_pct,
    ROUND(AVG(m3_llamadas_api),2)         AS M3_llamadas_api,
    ROUND(AVG(m5_tasa_error),1)                 AS M5_tasa_error,
    SUM(CASE WHEN status='PROCESADO' THEN 1 ELSE 0 END) AS exitosos,
    SUM(CASE WHEN status='ERROR'     THEN 1 ELSE 0 END) AS errores
FROM {LOG_TABLE}
GROUP BY fuente
ORDER BY fuente
""").display()

In [ ]:
print(f"\n📋 RESUMEN EJECUTIVO {FRAMEWORK_NAME.upper()}:")
spark.sql(f"""
SELECT
    '{FRAMEWORK_NAME}'                       AS framework,
    COUNT(*)                                 AS archivos_procesados,
    ROUND(AVG(m1_tiempo_segundos),2)         AS avg_M1_tiempo_s,
    0                                        AS M2_intervencion_manual,
    ROUND(AVG(m2_calidad_pct),2)             AS avg_M2_calidad_pct,
    ROUND(AVG(m3_llamadas_api),2)            AS avg_M5_llamadas,
    ROUND(AVG(m5_tasa_error),1)                    AS avg_M5_tasa_error,
    SUM(CASE WHEN status='PROCESADO' THEN 1 ELSE 0 END) AS exitosos,
    ROUND(SUM(CASE WHEN status='PROCESADO' THEN 1 ELSE 0 END)*100.0/COUNT(*),1) AS tasa_exito_pct
FROM {LOG_TABLE}
""").display()

## 📤 Bloque 8 — Exportación a Google Drive (Agente 3)

Replica el Agente de Exportación de LangGraph:

| Paso | LangGraph (nodo) | LlamaIndex (@step) |
|------|------------------|--------------------|
| 1 | `nodo_deteccion_export` | `detectar_tabla` |
| 2 | `nodo_seleccion_export` (LLM) | `seleccionar_con_llm` |
| 3 | `nodo_conversion_export` | `convertir_a_excel` |
| 4 | `nodo_carga_export` | `subir_a_drive` |
| 5 | `nodo_validacion_export` | `validar_subida` |
| 6 | `nodo_metricas_export` | `registrar_metricas` |

**Diferencia clave:** exporta **.xlsx** (igual que LangGraph), NO csv.


In [ ]:
# ==============================================================================
# EXPORTACION GOOGLE DRIVE — 11 archivos fijos
#
# REGLAS (basadas en la carpeta real de Drive):
#   - Archivos de DATOS       -> sobreescribir siempre (borron y cuenta nueva)
#   - Archivos de METRICAS    -> append por framework (diferenciar con columna framework)
#   - dim_tiempo              -> NO exportar (no tiene fuente de datos)
#
# Archivos fijos en Drive:
#   control_logs_langgraph.xlsx            -> control_logs_etl.xlsx (metricas - append)
#   espac_banano_platano_provincia.xlsx    (datos - overwrite)
#   espac_uso_del_suelo.xlsx               (datos - overwrite)
#   faostat_produccion_banano_platano.xlsx (datos - overwrite)
#   metricas_carga.xlsx                    (metricas - append)
#   metricas_extraccion.xlsx               (metricas - append)
#   metricas_transformacion.xlsx           (metricas - append)
#   PROVINCIAS.xlsx                        (dimension - overwrite)
#   REGIONES.xlsx                          (dimension - overwrite)
#   sipa_temperatura_precipitacion.xlsx    (datos - overwrite)
# ==============================================================================

from googleapiclient.http import MediaFileUpload, MediaInMemoryUpload
import io as _io

# Mapeo tabla -> nombre de archivo fijo en Drive
DRIVE_FILE_MAP = {
    # DATOS (sobreescribir)
    f"{DB_NAME}.espac_banano_platano_provincia":   "espac_banano_platano_provincia.xlsx",
    f"{DB_NAME}.espac_uso_del_suelo":              "espac_uso_del_suelo.xlsx",
    f"{DB_NAME}.faostat_produccion_banano_platano":"faostat_produccion_banano_platano.xlsx",
    f"{DB_NAME}.sipa_temperatura_precipitacion":   "sipa_temperatura_precipitacion.xlsx",
    f"{DB_NAME}.aebe_exportaciones_regiones":      "aebe_exportaciones_regiones.xlsx",
    # DIMENSIONES (sobreescribir)
    f"{DB_NAME}.dim_provincias":                   "PROVINCIAS.xlsx",
    # METRICAS (append por framework — NO sobreescribir)
    f"{DB_NAME}.metricas_extraccion":              "metricas_extraccion.xlsx",
    f"{DB_NAME}.metricas_transformacion":          "metricas_transformacion.xlsx",
    f"{DB_NAME}.metricas_carga":                   "metricas_carga.xlsx",
    f"{DB_NAME}.control_logs_etl":                 "control_logs_langgraph.xlsx",  # nombre fijo en Drive
}

# Tablas de metricas: se hace APPEND (no borrar datos previos de LangGraph)
METRICAS_TABLES = {
    f"{DB_NAME}.metricas_extraccion",
    f"{DB_NAME}.metricas_transformacion",
    f"{DB_NAME}.metricas_carga",
    f"{DB_NAME}.control_logs_etl",
}

# Tablas que NO se exportan
SKIP_TABLES = {
    f"{DB_NAME}.dim_tiempo",      # no tiene fuente
    f"{DB_NAME}.dim_productos",   # no esta en la lista de archivos fijos
}


def _df_to_excel_bytes(df_pandas) -> bytes:
    """Convierte un DataFrame pandas a bytes Excel en memoria."""
    buf = _io.BytesIO()
    df_pandas.to_excel(buf, index=False, engine='openpyxl')
    return buf.getvalue()


def _buscar_archivo_drive(nombre_archivo: str, folder_id: str) -> str | None:
    """Retorna el fileId si el archivo existe en la carpeta, None si no."""
    query = f"name='{nombre_archivo}' and '{folder_id}' in parents and trashed=false"
    res = drive_service.files().list(q=query, spaces='drive', fields='files(id,name)').execute()
    archivos = res.get('files', [])
    return archivos[0]['id'] if archivos else None


def actualizar_archivo_drive(nombre_archivo: str,
                              contenido_bytes: bytes = None,
                              df_pandas=None,
                              folder_id: str = None) -> dict:
    """
    Sube o actualiza un archivo .xlsx en Google Drive.
    Siempre sobreescribe el contenido del archivo existente (nombre fijo).
    Acepta bytes directos o un DataFrame pandas.
    """
    if folder_id is None:
        folder_id = FOLDER_OUTPUT_ID

    if contenido_bytes is None and df_pandas is not None:
        contenido_bytes = _df_to_excel_bytes(df_pandas)

    if contenido_bytes is None:
        return {"status": "ERROR", "mensaje": "Sin contenido", "fileId": ""}

    mime = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
    media = MediaInMemoryUpload(contenido_bytes, mimetype=mime, resumable=True)

    try:
        file_id = _buscar_archivo_drive(nombre_archivo, folder_id)
        if file_id:
            drive_service.files().update(
                fileId=file_id, media_body=media
            ).execute()
            print(f"    OK Actualizado: {nombre_archivo}")
            return {"status": "ACTUALIZADO", "fileId": file_id, "mensaje": f"Actualizado: {nombre_archivo}"}
        else:
            f = drive_service.files().create(
                body={"name": nombre_archivo, "parents": [folder_id]},
                media_body=media, fields='id'
            ).execute()
            print(f"    OK Creado: {nombre_archivo}")
            return {"status": "CREADO", "fileId": f.get('id'), "mensaje": f"Creado: {nombre_archivo}"}
    except Exception as e:
        err = str(e)
        if 'storageQuotaExceeded' in err:
            return {"status": "ERROR_QUOTA", "fileId": "",
                    "mensaje": f"Sin cuota Drive. Libera espacio o comparte la carpeta con {SERVICE_ACCOUNT_INFO.get('client_email','')}"}
        return {"status": "ERROR", "fileId": "", "mensaje": f"Error: {err}"}


def exportar_tabla_a_drive(tabla_completa: str, es_metrica: bool = False) -> dict:
    """
    Exporta una tabla Delta a Drive.
    - es_metrica=False -> sobreescribe (datos frescos)
    - es_metrica=True  -> append: descarga xlsx existente, agrega filas de la ÚLTIMA ejecución LlamaIndex
    """
    if tabla_completa in SKIP_TABLES:
        return {"status": "SKIP", "tabla": tabla_completa, "motivo": "Tabla excluida"}

    nombre_xlsx = DRIVE_FILE_MAP.get(tabla_completa)
    if not nombre_xlsx:
        return {"status": "SKIP", "tabla": tabla_completa, "motivo": "No mapeada"}

    try:
        if not spark.catalog.tableExists(tabla_completa):
            return {"status": "SKIP", "tabla": tabla_completa, "motivo": "Tabla no existe en Delta"}

        # Para métricas: filtrar solo la última ejecución de LlamaIndex
        if es_metrica:
            df_spark = spark.table(tabla_completa)
            # Obtener el execution_id más reciente de LlamaIndex
            ultima_ejecucion = df_spark.filter(
                F.col("framework") == FRAMEWORK_NAME
            ).agg(F.max("execution_id").alias("max_exec")).collect()[0]["max_exec"]
            
            if ultima_ejecucion:
                df_spark = df_spark.filter(
                    (F.col("framework") == FRAMEWORK_NAME) & 
                    (F.col("execution_id") == ultima_ejecucion)
                )
                df_nuevo = df_spark.toPandas()
                print(f"    Filtrando última ejecución: {ultima_ejecucion} ({len(df_nuevo)} registros)")
            else:
                df_nuevo = spark.table(tabla_completa).toPandas()
        else:
            df_nuevo = spark.table(tabla_completa).toPandas()
        
        n_nuevo  = len(df_nuevo)

        if es_metrica:
            # Intentar descargar el xlsx existente y combinar
            try:
                file_id = _buscar_archivo_drive(nombre_xlsx, FOLDER_OUTPUT_ID)
                if file_id:
                    content = drive_service.files().get_media(fileId=file_id).execute()
                    df_prev = pd.read_excel(_io.BytesIO(content))
                    # Eliminar filas previas de LlamaIndex para no duplicar
                    if 'framework' in df_prev.columns:
                        df_prev = df_prev[df_prev['framework'] != FRAMEWORK_NAME]
                    df_combined = pd.concat([df_prev, df_nuevo], ignore_index=True)
                    print(f"    Append: {len(df_prev)} prev + {n_nuevo} nuevos = {len(df_combined)} total")
                else:
                    df_combined = df_nuevo
                    print(f"    Nuevo archivo de metricas: {n_nuevo} filas")
            except Exception as e:
                print(f"    No se pudo leer xlsx previo ({e}), usando solo datos nuevos")
                df_combined = df_nuevo

            contenido = _df_to_excel_bytes(df_combined)
        else:
            # Datos: sobreescribir siempre
            contenido = _df_to_excel_bytes(df_nuevo)
            print(f"    Overwrite: {n_nuevo} registros")

        return actualizar_archivo_drive(nombre_xlsx, contenido_bytes=contenido)

    except Exception as e:
        import traceback; traceback.print_exc()
        return {"status": "ERROR", "tabla": tabla_completa, "mensaje": str(e)}


print("OK actualizar_archivo_drive y exportar_tabla_a_drive definidos.")
print(f"   Archivos fijos en Drive: {len(DRIVE_FILE_MAP)}")
print(f"   Tablas de metricas (append): {len(METRICAS_TABLES)}")
print(f"   Tablas excluidas: {SKIP_TABLES}")


In [ ]:
%sql
-- Verificar que llm_response ahora SÍ se guardó correctamente
SELECT 
    file_name,
    fuente,
    m3_llamadas_api,
    LEFT(llm_response, 100) as llm_response_preview,
    LENGTH(llm_response) as llm_response_length,
    status
FROM bd_banano_ec.control_logs_etl
WHERE framework = 'LlamaIndex'
  AND execution_id = (SELECT MAX(execution_id) FROM bd_banano_ec.control_logs_etl WHERE framework = 'LlamaIndex')
ORDER BY execution_timestamp DESC

In [ ]:
from typing import TypedDict, List, Optional
from datetime import datetime
import json

class ExportState(TypedDict):
    """Estado del agente de exportación a Google Drive."""
    
    # Identificación
    tabla_nombre: str           # Nombre completo de la tabla (ej: BD_BANANO_EC.espac_banano_provincia)
    archivo_csv: str            # Nombre del archivo CSV destino
    tipo: str                   # 'DATOS' o 'METRICAS'
    
    # Detección
    existe_tabla: bool          # Si la tabla existe en el catálogo
    num_registros: int          # Número de registros en la tabla
    ultima_modificacion: str    # Timestamp de última modificación
    ultima_exportacion: str     # Timestamp de última exportación
    requiere_exportacion: bool  # Si el LLM decide exportar
    
    # Selección (LLM)
    razonamiento_llm: str       # Por qué el LLM decidió exportar o no
    prioridad: int              # 1=ALTA, 2=MEDIA, 3=BAJA
    llamadas_llm: int           # Número de llamadas al LLM
    
    # Conversión
    csv_content: str            # Contenido CSV generado
    csv_size_bytes: int         # Tamaño del CSV en bytes
    hash_md5: str               # Hash MD5 del contenido
    ts_fin_conversion: str      # Timestamp fin conversión
    error_conversion: Optional[str]
    
    # Carga a Drive
    file_id_drive: str          # ID del archivo en Drive
    drive_status: str           # ACTUALIZADO, CREADO, ERROR
    intentos_carga: int         # Número de reintentos
    ts_fin_carga: str           # Timestamp fin carga
    error_carga: Optional[str]
    
    # Validación
    validacion_ok: bool         # Si la validación pasó
    registros_drive: int        # Registros leídos desde Drive (verificación)
    precision_pct: float        # % de precisión (registros_drive/num_registros)
    ts_fin_validacion: str      # Timestamp fin validación
    error_validacion: Optional[str]
    
    # Métricas
    inicio_ts: str              # Timestamp inicio proceso
    tiempo_total_s: float       # E1: Tiempo total
    bytes_transferidos: int     # E5: Bytes transferidos
    
    # Estado final
    status: str                 # EXPORTADO, ERROR, OMITIDO
    error_final: Optional[str]  # Error final si hubo

print("✅ ExportState definido.")

In [ ]:
def nodo_deteccion_export(estado: ExportState) -> ExportState:
    """
    Detecta si la tabla existe, cuántos registros tiene y cuándo fue modificada.
    """
    print(f"\n🔍 DETECCIÓN: {estado['tabla_nombre']}")
    
    try:
        # Verificar si la tabla existe
        if not spark.catalog.tableExists(estado["tabla_nombre"]):
            estado["existe_tabla"] = False
            estado["num_registros"] = 0
            estado["requiere_exportacion"] = False
            estado["status"] = "OMITIDO"
            estado["error_final"] = "Tabla no existe"
            print(f"  ⚠️  Tabla no existe: {estado['tabla_nombre']}")
            return estado
        
        estado["existe_tabla"] = True
        
        # Contar registros
        df = spark.table(estado["tabla_nombre"])
        count = df.count()
        estado["num_registros"] = count
        
        if count == 0:
            estado["requiere_exportacion"] = False
            estado["status"] = "OMITIDO"
            estado["error_final"] = "Tabla vacía (0 registros)"
            print(f"  ⚠️  Tabla vacía: 0 registros")
            return estado
        
        print(f"  📈 Tabla encontrada: {count:,} registros")
        
        # Inicializar campos requeridos si no existen
        if "archivo_csv" not in estado:
            tabla_sin_prefijo = estado["tabla_nombre"].split(".")[-1]
            estado["archivo_csv"] = f"{tabla_sin_prefijo}.xlsx"
        
        if "tipo" not in estado:
            # Determinar tipo basado en el nombre
            if "metricas_" in estado["tabla_nombre"] or "control_logs" in estado["tabla_nombre"]:
                estado["tipo"] = "METRICAS"
            else:
                estado["tipo"] = "DATOS"
        
        # Obtener timestamp de última modificación (si está disponible)
        try:
            table_details = spark.sql(f"DESCRIBE DETAIL {estado['tabla_nombre']}").collect()[0]
            estado["ultima_modificacion"] = str(table_details["lastModified"])
            print(f"  📅 Última modificación: {estado['ultima_modificacion']}")
        except:
            estado["ultima_modificacion"] = datetime.now().isoformat()
        
        # Verificar si ya fue exportada antes (si existe tabla de control)
        # TODO: implementar tabla de control de exportaciones
        estado["ultima_exportacion"] = ""  # Por ahora siempre exportar
        
        # Por defecto, marcar como que requiere exportación (el LLM decidirá)
        estado["requiere_exportacion"] = True
        
        print(f"  ✅ Detección completada")
        
    except Exception as e:
        estado["existe_tabla"] = False
        estado["num_registros"] = 0
        estado["requiere_exportacion"] = False
        estado["status"] = "ERROR"
        estado["error_final"] = f"Error en detección: {str(e)}"
        print(f"  ❌ Error: {e}")
    
    return estado

print("✅ nodo_deteccion_export() definido.")

In [ ]:
def nodo_seleccion_export(estado: ExportState) -> ExportState:
    """
    Usa el LLM para decidir si exportar la tabla y con qué prioridad.
    """
    print(f"\n🤖 SELECCIÓN LLM: {estado['tabla_nombre']}")
    
    try:
        # Preparar contexto para el LLM
        contexto = f"""
Tabla: {estado['tabla_nombre']}
Archivo CSV destino: {estado['archivo_csv']}
Tipo: {estado['tipo']}
Número de registros: {estado['num_registros']:,}
Última modificación: {estado['ultima_modificacion']}
Última exportación: {estado.get('ultima_exportacion', 'Nunca')}

Decide si esta tabla debe ser exportada a Google Drive en este momento.

Criterios de decisión:
1. Tablas de DATOS son de alta prioridad si tienen >100 registros
2. Tablas de MÉTRICAS son de alta prioridad si tienen datos nuevos
3. Tablas vacías o con muy pocos registros son de baja prioridad
4. Si fue exportada recientemente (<1 hora) y no cambió, es baja prioridad

Responde en formato JSON:
{{
  "debe_exportar": true/false,
  "prioridad": 1-3 (1=ALTA, 2=MEDIA, 3=BAJA),
  "razonamiento": "Explicación breve de la decisión"
}}
"""
        
        # Llamar al LLM (usando ChatDatabricks)
        response = llm.invoke([
            SystemMessage(content="Eres un asistente experto en gestión de datos. Respondes siempre en formato JSON válido."),
            HumanMessage(content=contexto)
        ])
        
        estado["llamadas_llm"] = estado.get("llamadas_llm", 0) + 1
        
        # Parsear respuesta
        respuesta = response.content.strip()
        
        # Limpiar markdown si existe
        if respuesta.startswith("```json"):
            respuesta = respuesta.replace("```json", "").replace("```", "").strip()
        
        decision = json.loads(respuesta)
        
        estado["requiere_exportacion"] = decision.get("debe_exportar", True)
        estado["procesar"] = decision.get("debe_exportar", True)  # Agregar para compatibilidad con workflow
        estado["prioridad"] = decision.get("prioridad", 2)
        estado["razonamiento_llm"] = decision.get("razonamiento", "Sin razonamiento")
        estado["llm_response"] = estado["razonamiento_llm"]  # Agregar alias
        
        print(f"  🎯 Decisión: {'EXPORTAR' if estado['requiere_exportacion'] else 'OMITIR'}")
        print(f"  📈 Prioridad: {estado['prioridad']} ({'ALTA' if estado['prioridad']==1 else 'MEDIA' if estado['prioridad']==2 else 'BAJA'})")
        print(f"  💬 Razonamiento: {estado['razonamiento_llm']}")
        
        if not estado["requiere_exportacion"]:
            estado["status"] = "OMITIDO"
            print(f"  ⚠️  Exportación omitida por decisión del LLM")
        
    except Exception as e:
        # Si el LLM falla, usar regla por defecto: exportar si tiene >0 registros
        print(f"  ⚠️  Error en LLM: {e}")
        print(f"  🔄 Aplicando regla por defecto...")
        
        estado["requiere_exportacion"] = estado["num_registros"] > 0
        estado["procesar"] = estado["num_registros"] > 0  # Agregar para compatibilidad con workflow
        estado["prioridad"] = 2  # MEDIA por defecto
        estado["razonamiento_llm"] = f"Fallback: Exportar porque tiene {estado['num_registros']} registros"
        estado["llm_response"] = estado["razonamiento_llm"]  # Agregar alias
        estado["llamadas_llm"] = estado.get("llamadas_llm", 0) + 1
        
        if not estado["requiere_exportacion"]:
            estado["status"] = "OMITIDO"
    
    return estado

print("✅ nodo_seleccion_export() definido.")

In [ ]:
import hashlib

def nodo_conversion_export(estado: ExportState) -> ExportState:
    """
    Convierte la tabla Spark a Excel y calcula hash MD5.
    """
    print(f"\n🔄 CONVERSIÓN: {estado['tabla_nombre']} → Excel")
    
    try:
        # Leer tabla
        df = spark.table(estado["tabla_nombre"])
        
        # Convertir a Pandas
        print(f"  🐌 Convirtiendo a Pandas...")
        pdf = df.toPandas()
        
        # Generar Excel en memoria
        print(f"  📝 Generando Excel...")
        excel_buffer = io.BytesIO()
        pdf.to_excel(excel_buffer, index=False, engine='openpyxl')
        excel_content = excel_buffer.getvalue()
        
        estado["csv_content"] = excel_content  # Mantenemos el nombre por compatibilidad
        estado["csv_size_bytes"] = len(excel_content)
        estado["excel_path"] = estado.get("archivo_csv", "temp.xlsx")  # Agregar excel_path
        
        # Calcular hash MD5
        hash_md5 = hashlib.md5(excel_content).hexdigest()
        estado["hash_md5"] = hash_md5
        
        estado["ts_fin_conversion"] = datetime.now().isoformat()
        estado["error_conversion"] = None
        
        print(f"  ✅ Excel generado: {estado['csv_size_bytes']:,} bytes")
        print(f"  🔑 Hash MD5: {hash_md5[:16]}...")
        
    except Exception as e:
        estado["error_conversion"] = str(e)
        estado["status"] = "ERROR"
        estado["error_final"] = f"Error en conversión: {str(e)}"
        print(f"  ❌ Error: {e}")
    
    return estado

print("✅ nodo_conversion_export() definido.")

In [ ]:
def nodo_carga_export(estado: ExportState) -> ExportState:
    """
    Sube el CSV a Google Drive con reintentos automáticos.
    """
    print(f"\n📤 CARGA A DRIVE: {estado['archivo_csv']}")
    
    max_intentos = 3
    estado["intentos_carga"] = 0
    
    for intento in range(1, max_intentos + 1):
        try:
            estado["intentos_carga"] = intento
            print(f"  🔄 Intento {intento}/{max_intentos}...")
            
            # Usar la función actualizar_archivo_drive que ya existe
            resultado = actualizar_archivo_drive(
                nombre_archivo=estado["archivo_csv"],
                contenido_bytes=estado["csv_content"],  # Usar contenido_bytes, no contenido_csv
                folder_id=FOLDER_OUTPUT_ID
            )
            
            estado["drive_status"] = resultado["status"]
            estado["file_id_drive"] = resultado.get("fileId", "")
            
            if resultado["status"] in ["ACTUALIZADO", "CREADO"]:
                print(f"  {resultado['mensaje']}")
                print(f"  🆔 File ID: {estado['file_id_drive']}")
                
                estado["error_carga"] = None
                estado["ts_fin_carga"] = datetime.now().isoformat()
                estado["bytes_transferidos"] = estado["csv_size_bytes"]
                break  # Éxito, salir del loop
            else:
                # Error, pero podemos reintentar
                print(f"  ⚠️  {resultado['mensaje']}")
                if intento < max_intentos:
                    print(f"  🔄 Reintentando en 2 segundos...")
                    import time
                    time.sleep(2)
                else:
                    # Último intento fallido
                    estado["error_carga"] = resultado.get("mensaje", "Error desconocido")
                    estado["status"] = "ERROR"
                    estado["error_final"] = f"Error en carga después de {max_intentos} intentos"
                    print(f"  ❌ Carga fallida después de {max_intentos} intentos")
        
        except Exception as e:
            print(f"  ❌ Error en intento {intento}: {e}")
            if intento < max_intentos:
                print(f"  🔄 Reintentando en 2 segundos...")
                import time
                time.sleep(2)
            else:
                estado["error_carga"] = str(e)
                estado["status"] = "ERROR"
                estado["error_final"] = f"Error en carga: {str(e)}"
    
    return estado

print("✅ nodo_carga_export() definido.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════

def nodo_validacion_export(estado: ExportState) -> ExportState:
    """
    Valida que el archivo en Drive tiene el número correcto de registros.
    """
    print(f"\n✅ VALIDACIÓN: {estado['archivo_csv']}")
    
    try:
        # Por simplicidad, asumimos que si se subió sin error, está OK
        # En producción, podrías descargar el archivo y verificar registros
        
        if estado["drive_status"] in ["ACTUALIZADO", "CREADO"]:
            # Para archivos Excel, simplemente asumir que si se subió OK, la cantidad es correcta
            # (no podemos contar líneas en un Excel binario fácilmente)
            estado["registros_drive"] = estado.get("num_registros", 0)  # ✅ Obtener de estado
            
            # Calcular precisión
            esperados = estado["num_registros"]
            obtenidos = estado["registros_drive"]
            
            if esperados > 0:
                estado["precision_pct"] = (obtenidos / esperados) * 100
            else:
                estado["precision_pct"] = 0.0
            
            # Validación pasa si precision >= 99%
            estado["validacion_ok"] = estado["precision_pct"] >= 99.0
            
            if estado["validacion_ok"]:
                print(f"  ✅ Validación exitosa: {obtenidos:,}/{esperados:,} registros ({estado['precision_pct']:.2f}%)")
                estado["status"] = "EXPORTADO"
            else:
                print(f"  ⚠️  Validación parcial: {obtenidos:,}/{esperados:,} registros ({estado['precision_pct']:.2f}%)")
                estado["status"] = "EXPORTADO"
                estado["error_validacion"] = f"Precisión baja: {estado['precision_pct']:.2f}%"
        else:
            estado["validacion_ok"] = False
            estado["registros_drive"] = 0
            estado["precision_pct"] = 0.0
            estado["status"] = "ERROR"
            print(f"  ❌ Validación fallida: carga no exitosa")
        
        estado["ts_fin_validacion"] = datetime.now().isoformat()
        
    except Exception as e:
        estado["error_validacion"] = str(e)
        estado["validacion_ok"] = False
        estado["status"] = "ERROR"
        print(f"  ❌ Error en validación: {e}")
    
    return estado

print("✅ nodo_validacion_export() definido.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════

def nodo_metricas_export(estado: ExportState) -> ExportState:
    """
    Calcula métricas finales de exportación (E1-E5).
    """
    print(f"\n📊 MÉTRICAS: {estado['archivo_csv']}")
    
    try:
        # E1: Tiempo total
        inicio = datetime.fromisoformat(estado["inicio_ts"])
        fin = datetime.now()
        estado["tiempo_total_s"] = (fin - inicio).total_seconds()
        
        # E2: Tablas exportadas (siempre 1 en este contexto)
        # E3: Precisión ya calculada en validación
        # E4: Reintentos ya registrado en estado["intentos_carga"]
        # E5: Bytes transferidos ya en estado["bytes_transferidos"]
        
        print(f"  E1 Tiempo total:     {estado['tiempo_total_s']:.2f}s")
        print(f"  E2 Tablas exportadas: 1")
        print(f"  E3 Precisión:        {estado.get('precision_pct', 0):.2f}%")
        print(f"  E4 Reintentos:        {estado.get('intentos_carga', 0)}")
        print(f"  E5 Bytes transferidos: {estado.get('bytes_transferidos', 0):,} bytes")
        
        # Guardar métricas en tabla Delta (opcional)
        # TODO: crear tabla de métricas de exportación
        
        print(f"  ✅ Métricas calculadas")
        
    except Exception as e:
        print(f"  ⚠️  Error calculando métricas: {e}")
    
    return estado


def nodo_error_export(estado: ExportState) -> ExportState:
    """
    Maneja errores del proceso de exportación.
    """
    print(f"\n❌ ERROR: {estado['archivo_csv']}")
    print(f"  Razón: {estado.get('error_final', 'Error desconocido')}")
    
    estado["status"] = "ERROR"
    
    # Calcular tiempo hasta el error
    if estado.get("inicio_ts"):
        inicio = datetime.fromisoformat(estado["inicio_ts"])
        fin = datetime.now()
        estado["tiempo_total_s"] = (fin - inicio).total_seconds()
    
    return estado

print("✅ nodo_metricas_export() y nodo_error_export() definidos.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# AGENTE 3: EXPORT WORKFLOW (LlamaIndex)
# Mismo patron bridge: cada @step construye ExportState, llama al nodo LangGraph,
# extrae campos y emite el Event de salida.
# ==============================================================================

class InicioExportEvent(Event):
    tabla_nombre:   str
    timestamp_inicio: str

class TablaDetectadaEvent(Event):
    tabla_nombre:   str
    existe:         bool
    n_registros:    int
    timestamp_inicio: str

class TablaSeleccionadaEvent(Event):
    tabla_nombre:   str
    procesar:       bool
    razon_llm:      str
    llamadas_llm:   int
    timestamp_inicio: str

class ExcelConvertidoEvent(Event):
    tabla_nombre:   str
    excel_path:     str
    n_registros:    int
    llamadas_llm:   int
    timestamp_inicio: str
    excel_content_b64: str  # Contenido Excel codificado en base64

class SubidaCompletaEvent(Event):
    tabla_nombre:   str
    file_id:        str
    web_view_link:  str
    n_registros:    int
    llamadas_llm:   int
    timestamp_inicio: str

class ExportErrorEvent(Event):
    tabla_nombre:   str
    causa:          str
    llamadas_llm:   int
    timestamp_inicio: str


class ExportWorkflow(Workflow):

    @step
    async def iniciar(self, ctx: Context, ev: StartEvent) -> InicioExportEvent:
        return InicioExportEvent(
            tabla_nombre     = ev.get("tabla_nombre"),
            timestamp_inicio = datetime.now().isoformat(),
        )

    @step
    async def detectar_tabla(self, ctx: Context, ev: InicioExportEvent) -> TablaDetectadaEvent | ExportErrorEvent:
        print(f"\n[EXPORT PASO 1 - DETECTAR] {ev.tabla_nombre}")
        estado = {"tabla_nombre": ev.tabla_nombre, "error_export": None, "llamadas_llm": 0}
        estado = nodo_deteccion_export(estado)
        if estado.get("error_export"):
            return ExportErrorEvent(tabla_nombre=ev.tabla_nombre, causa=estado["error_export"],
                llamadas_llm=0, timestamp_inicio=ev.timestamp_inicio)
        return TablaDetectadaEvent(
            tabla_nombre=ev.tabla_nombre, existe=estado.get("existe_tabla", False),  # Usar existe_tabla
            n_registros=estado.get("num_registros", 0), timestamp_inicio=ev.timestamp_inicio)  # Usar num_registros

    @step
    async def seleccionar_con_llm(self, ctx: Context, ev: TablaDetectadaEvent) -> TablaSeleccionadaEvent | ExportErrorEvent:
        print(f"\n[EXPORT PASO 2 - LLM SELECCION] {ev.tabla_nombre}")
        if not ev.existe:
            return TablaSeleccionadaEvent(tabla_nombre=ev.tabla_nombre, procesar=False,
                razon_llm="Tabla no existe", llamadas_llm=0, timestamp_inicio=ev.timestamp_inicio)
        # Preparar estado con TODOS los campos necesarios
        # ✅ Usar DRIVE_FILE_MAP para obtener el nombre correcto del archivo
        nombre_archivo = DRIVE_FILE_MAP.get(ev.tabla_nombre)
        if not nombre_archivo:
            # Fallback: generar desde el nombre de la tabla
            tabla_sin_prefijo = ev.tabla_nombre.split(".")[-1]
            nombre_archivo = f"{tabla_sin_prefijo}.xlsx"
        
        estado = {
            "tabla_nombre": ev.tabla_nombre,
            "num_registros": ev.n_registros,
            "existe_tabla": ev.existe,
            "archivo_csv": nombre_archivo,  # ✅ Usar nombre del mapeo
            "tipo": "METRICAS" if "metricas_" in ev.tabla_nombre or "control_logs" in ev.tabla_nombre else "DATOS",
            "ultima_modificacion": datetime.now().isoformat(),
            "error_export": None,
            "llamadas_llm": 0
        }
        estado = nodo_seleccion_export(estado)
        if estado.get("error_export"):
            return ExportErrorEvent(tabla_nombre=ev.tabla_nombre, causa=estado["error_export"],
                llamadas_llm=estado.get("llamadas_llm",0), timestamp_inicio=ev.timestamp_inicio)
        return TablaSeleccionadaEvent(
            tabla_nombre=ev.tabla_nombre, procesar=estado.get("procesar", True),
            razon_llm=estado.get("llm_response","")[:200],
            llamadas_llm=estado.get("llamadas_llm", 1), timestamp_inicio=ev.timestamp_inicio)

    @step
    async def convertir_a_excel(self, ctx: Context, ev: TablaSeleccionadaEvent) -> ExcelConvertidoEvent | ExportErrorEvent:
        print(f"\n[EXPORT PASO 3 - CONVERTIR] {ev.tabla_nombre}")
        if not ev.procesar:
            return ExportErrorEvent(tabla_nombre=ev.tabla_nombre,
                causa=f"Omitido por LLM: {ev.razon_llm}",
                llamadas_llm=ev.llamadas_llm, timestamp_inicio=ev.timestamp_inicio)
        
        # Obtener num_registros directamente de la tabla
        df = spark.table(ev.tabla_nombre)
        num_registros = df.count()
        
        # ✅ Usar DRIVE_FILE_MAP para obtener el nombre correcto del archivo
        nombre_archivo = DRIVE_FILE_MAP.get(ev.tabla_nombre)
        if not nombre_archivo:
            tabla_sin_prefijo = ev.tabla_nombre.split(".")[-1]
            nombre_archivo = f"{tabla_sin_prefijo}.xlsx"
        
        estado = {
            "tabla_nombre": ev.tabla_nombre,
            "procesar": True,
            "archivo_csv": nombre_archivo,  # ✅ Usar nombre del mapeo
            "num_registros": num_registros,  # ✅ Agregar num_registros
            "llamadas_llm": ev.llamadas_llm,
            "error_export": None
        }
        estado = nodo_conversion_export(estado)
        if estado.get("error_export"):
            return ExportErrorEvent(tabla_nombre=ev.tabla_nombre, causa=estado["error_export"],
                llamadas_llm=estado.get("llamadas_llm",0), timestamp_inicio=ev.timestamp_inicio)
        # Codificar contenido en base64 para pasar en el evento
        import base64
        excel_b64 = base64.b64encode(estado.get("csv_content", b"")).decode('utf-8')
        return ExcelConvertidoEvent(
            tabla_nombre=ev.tabla_nombre,
            excel_path=estado.get("excel_path",""),
            n_registros=estado.get("num_registros", 0),  # ✅ Propagado desde estado
            llamadas_llm=estado.get("llamadas_llm",0),
            timestamp_inicio=ev.timestamp_inicio,
            excel_content_b64=excel_b64)

    @step
    async def subir_a_drive(self, ctx: Context, ev: ExcelConvertidoEvent) -> SubidaCompletaEvent | ExportErrorEvent:
        print(f"\n[EXPORT PASO 4 - SUBIR DRIVE] {ev.tabla_nombre}")
        # Decodificar contenido desde base64
        import base64
        excel_content = base64.b64decode(ev.excel_content_b64)
        estado = {
            "tabla_nombre": ev.tabla_nombre,
            "archivo_csv": ev.excel_path,
            "csv_content": excel_content,
            "csv_size_bytes": len(excel_content),
            "num_registros": ev.n_registros,
            "llamadas_llm": ev.llamadas_llm,
            "error_export": None
        }
        estado = nodo_carga_export(estado)
        if estado.get("error_carga"):  # ✅ Corregir: error_carga, no error_export
            return ExportErrorEvent(tabla_nombre=ev.tabla_nombre, causa=estado["error_carga"],
                llamadas_llm=estado.get("llamadas_llm",0), timestamp_inicio=ev.timestamp_inicio)
        estado = nodo_validacion_export(estado)
        return SubidaCompletaEvent(
            tabla_nombre=ev.tabla_nombre,
            file_id=estado.get("file_id_drive",""),  # ✅ Corregir nombre campo
            web_view_link=estado.get("drive_link",""),
            n_registros=ev.n_registros,  # ✅ Ya viene del evento
            llamadas_llm=estado.get("llamadas_llm",0), timestamp_inicio=ev.timestamp_inicio)

    @step
    async def registrar_metricas(self, ctx: Context, ev: SubidaCompletaEvent) -> StopEvent:
        print(f"\n[EXPORT PASO 5 - METRICAS] {ev.tabla_nombre}")
        # ✅ Usar DRIVE_FILE_MAP para obtener el nombre correcto del archivo
        nombre_archivo = DRIVE_FILE_MAP.get(ev.tabla_nombre)
        if not nombre_archivo:
            tabla_sin_prefijo = ev.tabla_nombre.split(".")[-1]
            nombre_archivo = f"{tabla_sin_prefijo}.xlsx"
        
        # ✅ Calcular bytes transferidos desde el archivo Excel
        df = spark.table(ev.tabla_nombre)
        pdf = df.toPandas()
        excel_buffer = io.BytesIO()
        pdf.to_excel(excel_buffer, index=False, engine='openpyxl')
        bytes_transferidos = len(excel_buffer.getvalue())
        
        estado = {
            "tabla_nombre": ev.tabla_nombre,
            "archivo_csv": nombre_archivo,  # ✅ Usar nombre del mapeo
            "drive_file_id": ev.file_id,
            "drive_link": ev.web_view_link,
            "num_registros": ev.n_registros,  # ✅ Cambiar a num_registros
            "bytes_transferidos": bytes_transferidos,  # ✅ Agregar bytes transferidos
            "llamadas_llm": ev.llamadas_llm,
            "inicio_ts": ev.timestamp_inicio
        }
        estado = nodo_metricas_export(estado)
        return StopEvent(result={
            "status": "EXPORTADO", "tabla": ev.tabla_nombre,
            "file_id": ev.file_id, "link": ev.web_view_link, "registros": ev.n_registros,
        })

    @step
    async def manejar_error(self, ctx: Context, ev: ExportErrorEvent) -> StopEvent:
        print(f"\nERROR EXPORT {ev.tabla_nombre}: {ev.causa}")
        return StopEvent(result={"status":"ERROR","tabla":ev.tabla_nombre,"causa":ev.causa})

print("OK ExportWorkflow (AGENTE 3) definido.")
print("   Pasos: detectar -> seleccionar_llm -> convertir_excel -> subir_drive -> metricas -> END")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ORQUESTADOR AGENTE 3: EXPORTAR TABLAS A GOOGLE DRIVE
# ══════════════════════════════════════════════════════════════════════════

import asyncio

async def ejecutar_agente3_exportacion():
    """Exporta cada tabla Delta al Drive usando ExportWorkflow."""
    # Tablas de datos (Gold)
    tablas_gold = [
        f"{DB_NAME}.espac_banano_platano_provincia",
        f"{DB_NAME}.espac_uso_del_suelo",
        f"{DB_NAME}.sipa_temperatura_precipitacion",
        f"{DB_NAME}.faostat_produccion_banano_platano",
        f"{DB_NAME}.aebe_exportaciones_regiones",
    ]
    
    # Tablas de métricas
    tablas_metricas = [
        f"{DB_NAME}.metricas_extraccion",
        f"{DB_NAME}.metricas_transformacion",
        f"{DB_NAME}.metricas_carga",
        f"{DB_NAME}.control_logs_etl",
    ]
    
    # Combinar todas las tablas a exportar
    todas_las_tablas = tablas_gold + tablas_metricas

    resultados_export = []
    ts_global         = datetime.now()

    for tabla in todas_las_tablas:
        print(f"\n{'='*70}")
        print(f"AGENTE 3 EXPORT: {tabla}")
        print(f"{'='*70}")
        try:
            wf = ExportWorkflow(timeout=300, verbose=False)
            resultado = await wf.run(tabla_nombre=tabla)
            resultados_export.append(resultado)
            st = resultado.get('status','?') if resultado else 'ERROR'
            print(f"  => {st}")
        except Exception as e:
            print(f"  ERROR en {tabla}: {e}")
            resultados_export.append({"status":"ERROR","tabla":tabla,"causa":str(e)})

    tiempo_total = (datetime.now() - ts_global).total_seconds()
    ok  = sum(1 for r in resultados_export if r and r.get('status') == 'EXPORTADO')
    err = sum(1 for r in resultados_export if r and r.get('status') == 'ERROR')
    print(f"\n{'='*70}")
    print(f"EXPORTACION COMPLETADA en {tiempo_total:.1f}s")
    print(f"  Tablas exportadas: {ok}/{len(todas_las_tablas)} | Errores: {err}")
    print(f"{'='*70}")
    return resultados_export

resultados_export = asyncio.run(ejecutar_agente3_exportacion())
